# Análisis de datos de materiales MPEA
**Proyecto 1 · Ingeniería de Materiales**  
**Manuel Ernesto Santos Ramos**

## Cómo usar este reporte
Este archivo contiene explicaciones, datos, resultados descriptivos ya calculados y código Python para reproducir el análisis. No necesitas cargar el Excel: los 1,545 registros están incluidos.

1. Abre el archivo en Google Colab: **Archivo → Subir notebook**.
2. Lee las secciones de texto, tablas y la figura incluida.
3. Selecciona **Entorno de ejecución → Ejecutar todas** para generar las salidas de Python.
4. Revisa que no haya errores y descarga el notebook con las salidas guardadas para entregarlo.

**Estado de verificación:** las tablas y la figura de referencia se calcularon directamente con los datos integrados mediante JavaScript. El código Python incluido no se ejecutó en esta actualización porque no estaba disponible el entorno de ejecución. Las celdas no se presentan como ejecutadas. Si modificas datos o filtros, actualiza también los resultados y conclusiones escritos.

## 1. Objetivo y flujo de trabajo
Analizar propiedades mecánicas de aleaciones multicomponente y responder una comparación técnicamente sustentada, haciendo explícitas las decisiones.

La exploración de datos permite reconocer distribuciones, valores faltantes, heterogeneidad y posibles relaciones antes de interpretar una comparación. Se adopta como referencia la [guía de análisis exploratorio del NIST](https://itl.nist.gov/div898/handbook/eda/eda.htm), sin imponer un procedimiento único.

| Etapa | Qué se hace y para qué |
| --- | --- |
| Comprender | Identificar qué representa cada fila, variable y unidad. |
| Auditar | Contar faltantes, duplicados y categorías para detectar riesgos. |
| Preparar | Aplicar reglas explícitas sin alterar la base original. |
| Explorar | Examinar frecuencias, distribuciones y relaciones. |
| Preguntar | Elegir una comparación con datos disponibles y relevancia mecánica. |
| Seleccionar y comparar | Separar ensayos y limitar condiciones; cuantificar las diferencias. |
| Interpretar | Responder la pregunta, reconocer explicaciones alternativas y límites. |

## 2. Descripción del dataset
La copia integrada procede del archivo **MPEA_dataset(1).xlsx** utilizado en la página; la exportación CSV conserva los mismos registros. Contiene **1,545 filas y 23 variables**. Una fila es un registro de propiedades reportadas, no necesariamente una aleación única ni una observación independiente. La misma composición puede aparecer con distintos procesos o temperaturas.

Las columnas originales y sus valores se conservan. Los nombres en español son ayudas de lectura; no implican convertir unidades. **wppm** significa partes por millón en masa.

| Columna original | Lectura en español |
| --- | --- |
| IDENTIFIER: Reference ID | ID de referencia |
| FORMULA | Composición química |
| PROPERTY: Microstructure | Microestructura |
| PROPERTY: Processing method | Procesamiento |
| PROPERTY: BCC/FCC/other | Clasificación BCC/FCC/other |
| PROPERTY: grain size ($\mu$m) | Tamaño de grano (µm) |
| PROPERTY: Exp. Density (g/cm$^3$) | Densidad experimental (g/cm³) |
| PROPERTY: Calculated Density (g/cm$^3$) | Densidad calculada (g/cm³) |
| PROPERTY: HV | Dureza Vickers (HV) |
| PROPERTY: Type of test | Tipo de ensayo |
| PROPERTY: Test temperature ($^\circ$C) | Temperatura de ensayo (°C) |
| PROPERTY: YS (MPa) | Límite elástico (MPa) |
| PROPERTY: UTS (MPa) | Resistencia máxima / UTS (MPa) |
| PROPERTY: Elongation (%) | Elongación (%) |
| PROPERTY: Elongation plastic (%) | Elongación plástica (%) |
| PROPERTY: Exp. Young modulus (GPa) | Módulo de Young experimental (GPa) |
| PROPERTY: Calculated Young modulus (GPa) | Módulo de Young calculado (GPa) |
| PROPERTY: O content (wppm) | Contenido de O (wppm) |
| PROPERTY: N content (wppm) | Contenido de N (wppm) |
| PROPERTY: C content (wppm) | Contenido de C (wppm) |
| REFERENCE: doi | DOI |
| REFERENCE: year | Año |
| REFERENCE: title | Título del artículo |

BCC y FCC son categorías de estructura cristalina; **other** incluye otras clasificaciones. La columna de microestructura puede tener etiquetas más detalladas y no siempre coincide con esa clasificación. **T** indica tracción y **C** compresión. Un DOI identifica el artículo de origen.

Se mantienen separados valores experimentales y calculados. “Elongación” y “Elongación plástica” tampoco se consideran intercambiables: su definición exacta depende de la fuente. El campo UTS en compresión necesita interpretación según el artículo original.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Datos originales integrados: no se modifican en el análisis.
data = json.loads(r'''{"headers":["IDENTIFIER: Reference ID","FORMULA","PROPERTY: Microstructure","PROPERTY: Processing method","PROPERTY: BCC/FCC/other","PROPERTY: grain size ($\\mu$m)","PROPERTY: Exp. Density (g/cm$^3$)","PROPERTY: Calculated Density (g/cm$^3$)","PROPERTY: HV","PROPERTY: Type of test","PROPERTY: Test temperature ($^\\circ$C)","PROPERTY: YS (MPa)","PROPERTY: UTS (MPa)","PROPERTY: Elongation (%)","PROPERTY: Elongation plastic (%)","PROPERTY: Exp. Young modulus (GPa)","PROPERTY: Calculated Young modulus (GPa)","PROPERTY: O content (wppm)","PROPERTY: N content (wppm)","PROPERTY: C content (wppm)","REFERENCE: doi","REFERENCE: year","REFERENCE: title"],"rows":[[27,"Al0.25 Co1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.9,138,"C",25,158,null,null,null,null,192,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Al0.5 Co1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.4,212,"C",25,346,null,null,null,null,null,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Al0.75 Co1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7,385,"C",25,794,null,null,null,null,null,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Al1 Co1 Fe1 Ni1","BCC","CAST","BCC",null,null,6.6,456,"C",25,964,null,null,null,null,162,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Co1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.5,125,"C",25,204,null,null,null,null,207,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Co1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.5,125,"C",25,209,null,null,null,null,207,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Co1 Fe1 Ni1 Si0.25","FCC","CAST","FCC",null,null,7.7,149,"C",25,196,null,null,null,null,186,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Co1 Fe1 Ni1 Si0.5","FCC+Sec.","CAST","other",null,null,7.1,287,"C",25,476,null,null,null,null,null,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[27,"Co1 Fe1 Ni1 Si0.75","FCC+Sec.","CAST","other",null,null,6.6,570,"C",25,1301,null,null,null,null,null,null,null,null,"10.1016/j.jmmm.2014.07.023",2014,"Effects of Al and Si addition on the structure and properties of CoFeNi equal atomic ratio alloy"],[37,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",24,null,8.2,null,"T",25,274,708,39,null,null,226,null,null,null,"10.1016/j.actamat.2014.08.026",2014,"Temperature dependence of the mechanical properties of equiatomic solid solution alloys with face-centered cubic crystal structures"],[37,"Co1 Cr1 Mn1 Ni1","FCC","WROUGHT","FCC",36,null,8.1,null,"T",25,282,694,44,null,null,222,null,null,null,"10.1016/j.actamat.2014.08.026",2014,"Temperature dependence of the mechanical properties of equiatomic solid solution alloys with face-centered cubic crystal structures"],[37,"Co1 Cr1 Ni1","FCC","WROUGHT","FCC",41,null,8.3,null,"T",25,300,860,60,null,null,231,null,null,null,"10.1016/j.actamat.2014.08.026",2014,"Temperature dependence of the mechanical properties of equiatomic solid solution alloys with face-centered cubic crystal structures"],[37,"Co1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",48,null,8.2,null,"T",25,170,550,41,null,null,204,null,null,null,"10.1016/j.actamat.2014.08.026",2014,"Temperature dependence of the mechanical properties of equiatomic solid solution alloys with face-centered cubic crystal structures"],[37,"Co1 Fe1 Ni1","FCC","WROUGHT","FCC",28,null,8.5,null,"T",25,211,513,31,null,null,207,null,null,null,"10.1016/j.actamat.2014.08.026",2014,"Temperature dependence of the mechanical properties of equiatomic solid solution alloys with face-centered cubic crystal structures"],[37,"Co1 Mn1 Ni1","FCC","WROUGHT","FCC",32,null,8.4,null,"T",25,231,653,38,null,null,202,null,null,null,"10.1016/j.actamat.2014.08.026",2014,"Temperature dependence of the mechanical properties of equiatomic solid solution alloys with face-centered cubic crystal structures"],[37,"Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",30,null,8.1,null,"T",25,221,602,36,null,null,203,null,null,null,"10.1016/j.actamat.2014.08.026",2014,"Temperature dependence of the mechanical properties of equiatomic solid solution alloys with face-centered cubic crystal structures"],[46,"Co1 Cr1 Fe1 Nb0.103 Ni1","FCC+Sec.","CAST","other",175,null,8.2,null,"T",25,318,622,19,null,null,null,null,null,null,"10.1016/j.intermet.2015.01.004",2015,"Effects of Nb additions on the microstructure and mechanical property of CoCrFeNi high-entropy alloys"],[46,"Co1 Cr1 Fe1 Nb0.155 Ni1","FCC+Sec.","CAST","other",175,null,8.2,null,"T",25,322,744,23,null,null,null,null,null,null,"10.1016/j.intermet.2015.01.004",2015,"Effects of Nb additions on the microstructure and mechanical property of CoCrFeNi high-entropy alloys"],[46,"Co1 Cr1 Fe1 Nb0.206 Ni1","FCC+Sec.","CAST","other",175,null,8.2,null,"T",25,403,807,9,null,null,null,null,null,null,"10.1016/j.intermet.2015.01.004",2015,"Effects of Nb additions on the microstructure and mechanical property of CoCrFeNi high-entropy alloys"],[46,"Co1 Cr1 Fe1 Nb0.309 Ni1","FCC+Sec.","CAST","other",175,null,8.2,null,"T",25,479,879,4,null,null,null,null,null,null,"10.1016/j.intermet.2015.01.004",2015,"Effects of Nb additions on the microstructure and mechanical property of CoCrFeNi high-entropy alloys"],[46,"Co1 Cr1 Fe1 Nb0.412 Ni1","FCC+Sec.","CAST","other",175,null,8.2,null,"T",25,638,1004,1,null,null,null,null,null,null,"10.1016/j.intermet.2015.01.004",2015,"Effects of Nb additions on the microstructure and mechanical property of CoCrFeNi high-entropy alloys"],[46,"Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",175,null,8.2,null,"T",25,148,413,48,null,null,226,null,null,null,"10.1016/j.intermet.2015.01.004",2015,"Effects of Nb additions on the microstructure and mechanical property of CoCrFeNi high-entropy alloys"],[54,"Al0.25 Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,7.7,110,null,25,null,null,null,null,null,213,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.25 Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,7.7,113,null,25,null,null,null,null,null,213,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.375 Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,7.5,131,null,25,null,null,null,null,null,207,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.375 Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,7.5,196,null,25,null,null,null,null,null,207,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.5 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,7.3,159,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.5 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,7.3,209,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.75 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,7,388,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.75 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,7,280,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.875 Co1 Cr1 Fe1 Ni1","BCC","ANNEAL","BCC",null,null,6.9,538,null,25,null,null,null,null,null,188,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al0.875 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,6.9,361,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co0.8 Cr0.8 Fe0.8 Ni0.8","BCC","ANNEAL","BCC",null,null,6.5,487,null,25,null,null,null,null,null,177,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co0.8 Cr0.8 Fe0.8 Ni0.8","BCC","ANNEAL","BCC",null,null,6.5,499,null,25,null,null,null,null,null,177,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co0.667 Cr0.667 Fe0.667 Ni0.667","BCC","ANNEAL","BCC",null,null,6.2,484,null,25,null,null,null,null,null,171,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co0.667 Cr0.667 Fe0.667 Ni0.667","BCC","ANNEAL","BCC",null,null,6.2,517,null,25,null,null,null,null,null,171,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co0.5 Cr0.5 Fe0.5 Ni0.5","BCC","ANNEAL","BCC",null,null,5.9,509,null,25,null,null,null,null,null,160,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co0.5 Cr0.5 Fe0.5 Ni0.5","BCC","ANNEAL","BCC",null,null,5.9,512,null,25,null,null,null,null,null,160,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co1 Cr1 Fe1 Ni1","BCC","ANNEAL","BCC",null,null,6.7,484,null,25,null,null,null,null,null,184,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Al1 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,6.7,433,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,8.2,116,null,25,null,null,null,null,null,226,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[54,"Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,8.2,113,null,25,null,null,null,null,null,226,null,null,null,"10.1016/j.jallcom.2009.08.090",2009,"Microstructure and mechanical property of as-cast, -homogenized, and -deformed AlxCoCrFeNi (0≤x≤2) high-entropy alloys"],[63,"Al0.5 Co1 Cr1 Fe1 Mo0.5 Ni1","FCC+Sec.","CAST","other",null,null,7.7,425,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11837-013-0753-6",2013,"Effect of Aluminum Content on Microstructure and Mechanical Properties of Al x CoCrFeMo0.5Ni High-Entropy Alloys"],[63,"Al1 Co0.667 Cr0.667 Fe0.667 Mo0.333 Ni0.667","B2+Sec.","CAST","other",null,null,6.6,655,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11837-013-0753-6",2013,"Effect of Aluminum Content on Microstructure and Mechanical Properties of Al x CoCrFeMo0.5Ni High-Entropy Alloys"],[63,"Al1 Co0.5 Cr0.5 Fe0.5 Mo0.25 Ni0.5","BCC+B2","CAST","other",null,null,6.3,605,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11837-013-0753-6",2013,"Effect of Aluminum Content on Microstructure and Mechanical Properties of Al x CoCrFeMo0.5Ni High-Entropy Alloys"],[63,"Al1 Co1 Cr1 Fe1 Mo0.5 Ni1","B2+Sec.","CAST","other",null,null,7.1,715,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11837-013-0753-6",2013,"Effect of Aluminum Content on Microstructure and Mechanical Properties of Al x CoCrFeMo0.5Ni High-Entropy Alloys"],[63,"Co1 Cr1 Fe1 Mo0.5 Ni1","FCC+Sec.","CAST","other",null,null,8.5,210,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11837-013-0753-6",2013,"Effect of Aluminum Content on Microstructure and Mechanical Properties of Al x CoCrFeMo0.5Ni High-Entropy Alloys"],[66,"Al0.5 Co1 Cr1 Fe1 Ni1 Ti1","BCC+Laves","CAST","other",null,null,6.6,null,"C",25,null,1600,null,10,107,null,null,null,null,"10.1016/j.msea.2008.12.053",2009,"Microstructure and mechanical properties of CoCrFeNiTiAlx high-entropy alloys"],[66,"Al1 Co0.667 Cr0.667 Fe0.667 Ni0.667 Ti0.667","BCC+BCC","CAST","other",null,null,5.9,null,"C",25,null,2110,null,10,133,null,null,null,null,"10.1016/j.msea.2008.12.053",2009,"Microstructure and mechanical properties of CoCrFeNiTiAlx high-entropy alloys"],[66,"Al1 Co0.5 Cr0.5 Fe0.5 Ni0.5 Ti0.5","BCC+BCC","CAST","other",null,null,5.6,643,"C",25,null,1030,null,5,94,null,null,null,null,"10.1016/j.msea.2008.12.053",2009,"Microstructure and mechanical properties of CoCrFeNiTiAlx high-entropy alloys"],[66,"Al1 Co1 Cr1 Fe1 Ni1 Ti1","BCC+Laves","CAST","other",0.15,null,6.2,null,"C",25,null,2280,null,6,148,null,null,null,null,"10.1016/j.msea.2008.12.053",2009,"Microstructure and mechanical properties of CoCrFeNiTiAlx high-entropy alloys"],[66,"Co1 Cr1 Fe1 Ni1 Ti1","FCC","CAST","FCC",null,null,7.2,null,"C",25,null,2020,null,9,135,195,null,null,null,"10.1016/j.msea.2008.12.053",2009,"Microstructure and mechanical properties of CoCrFeNiTiAlx high-entropy alloys"],[1,"Al0.133 Co1 Cr0.667 Fe0.667 Ni1 Ti0.667","FCC+Sec.","ANNEAL","other",null,null,7.2,717,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2011.06.041",2011,"Microstructure and wear behavior of AlxCo1.5CrFeNi1.5Tiy high-entropy alloys"],[1,"Al0.133 Co1 Cr0.667 Fe0.667 Ni1 Ti0.333","FCC","ANNEAL","FCC",null,null,7.6,487,null,25,null,null,null,null,null,201,null,null,null,"10.1016/j.actamat.2011.06.041",2011,"Microstructure and wear behavior of AlxCo1.5CrFeNi1.5Tiy high-entropy alloys"],[1,"Co1 Cr0.667 Fe0.667 Ni1 Ti0.667","FCC+Sec.","ANNEAL","other",null,null,7.4,654,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2011.06.041",2011,"Microstructure and wear behavior of AlxCo1.5CrFeNi1.5Tiy high-entropy alloys"],[1,"Co1 Cr0.667 Fe0.667 Ni1 Ti0.333","FCC","ANNEAL","other",null,null,7.8,509,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2011.06.041",2011,"Microstructure and wear behavior of AlxCo1.5CrFeNi1.5Tiy high-entropy alloys"],[2,"Al0.7 Co0.3 Cr1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,624,"C",25,2033,2635,8,null,null,null,null,null,null,"10.1016/j.matdes.2013.04.061",2013,"Alloying behavior, microstructure and mechanical properties in a FeNiCrCo0.3Al0.7 high entropy alloy"],[3,"Al0.875 Co1 Cr1 Fe1 Ni1","FCC+BCC",null,"other",null,null,6.9,null,null,25,null,null,null,null,null,null,null,null,null,"10.1016/C2013-0-14235-3",2014,"High Entropy Alloys"],[4,"Al1 Co0.667 Cr0.667 Fe0.667 Ni0.667","BCC","CAST","BCC",null,null,6.2,402,null,25,null,null,null,null,null,171,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Al1 Co0.4 Cr0.4 Fe0.4 Ni0.4","BCC","CAST","BCC",null,null,5.6,487,null,25,null,null,null,null,null,152,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Al1 Co0.5 Cr0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,5.9,432,null,25,null,null,null,null,null,160,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Al1 Co0.333 Cr0.333 Fe0.333 Ni0.333","BCC","CAST","BCC",null,null,5.3,506,null,25,null,null,null,null,null,145,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Al1 Co1 Cr1 Fe1 Ni1","BCC","CAST","BCC",null,null,6.7,395,null,25,null,null,null,null,null,184,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,6.8,342,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.3,286,null,25,null,null,null,null,null,206,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,8.1,296,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[4,"Cr1 Cu1 Fe1 Mo1 Ni1","FCC","CAST","FCC",null,null,8.7,263,null,25,null,null,null,null,null,236,null,null,null,"10.1016/j.jallcom.2008.07.124",2009,"Effect of alloying elements on microstructure and properties of multiprincipal elements high-entropy alloys"],[5,"Al1 Co1 Cr1 Fe1 Ni1","BCC","CAST","BCC",null,null,6.7,null,"C",25,1251,2004,null,33,null,184,null,null,null,"10.1016/j.msea.2008.01.064",2008,"Microstructure and compressive properties of AlCrFeCoNi high entropy alloy"],[6,"Al1 C0.1 Co1 Cr1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,6.7,null,"C",25,957,2550,null,11,213,null,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[6,"Al1 C0.2 Co1 Cr1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,6.6,null,"C",25,906,2386,null,9,151,null,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[6,"Al1 C0.3 Co1 Cr1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,6.5,null,"C",25,867,2178,null,8,137,null,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[6,"Al1 C0.4 Co1 Cr1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,6.5,null,"C",25,1056,2375,null,7,156,null,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[6,"Al1 C0.5 Co1 Cr1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,6.4,null,"C",25,1060,2250,null,6,181,null,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[6,"Al0.667 C1 Co0.667 Cr0.667 Fe0.667 Ni0.667","BCC+Sec.","CAST","other",null,null,5.9,null,"C",25,1255,2083,null,6,73,null,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[6,"Al1 C1 Co1 Cr1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,6.2,null,"C",25,1251,2166,null,7,75,null,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[6,"Al1 Co1 Cr1 Fe1 Ni1","BCC","CAST","BCC",null,null,6.7,null,"C",25,1138,null,null,11,null,184,null,null,null,"10.1016/j.jallcom.2010.10.047",2011,"Microstructure and compressive properties of multiprincipal component AlCoCrFeNiCx alloys"],[7,"Al1 Co1 Cr1 Fe1 Mo0.1 Ni1","BCC","CAST","BCC",null,null,6.8,null,"C",25,1804,2280,null,9,null,188,null,null,null,"10.1016/j.msea.2010.07.028",2010,"Microstructures and compressive properties of multicomponent AlCoCrFeNiMox alloys"],[7,"Al1 Co1 Cr1 Fe1 Mo0.2 Ni1","BCC+Sec.","CAST","other",null,null,6.9,null,"C",25,2456,2953,null,3,null,null,null,null,null,"10.1016/j.msea.2010.07.028",2010,"Microstructures and compressive properties of multicomponent AlCoCrFeNiMox alloys"],[7,"Al1 Co1 Cr1 Fe1 Mo0.3 Ni1","BCC+Sec.","CAST","other",null,null,7,null,"C",25,2649,3208,null,3,null,null,null,null,null,"10.1016/j.msea.2010.07.028",2010,"Microstructures and compressive properties of multicomponent AlCoCrFeNiMox alloys"],[7,"Al1 Co1 Cr1 Fe1 Mo0.4 Ni1","BCC+Sec.","CAST","other",null,null,7,null,"C",25,2670,3161,null,3,null,null,null,null,null,"10.1016/j.msea.2010.07.028",2010,"Microstructures and compressive properties of multicomponent AlCoCrFeNiMox alloys"],[7,"Al1 Co1 Cr1 Fe1 Mo0.5 Ni1","BCC+Sec.","CAST","other",null,null,7.1,null,"C",25,2757,3036,null,3,null,null,null,null,null,"10.1016/j.msea.2010.07.028",2010,"Microstructures and compressive properties of multicomponent AlCoCrFeNiMox alloys"],[7,"Al1 Co1 Cr1 Fe1 Ni1","BCC","CAST","BCC",null,null,6.7,null,"C",25,1051,null,null,null,null,184,null,null,null,"10.1016/j.msea.2010.07.028",2010,"Microstructures and compressive properties of multicomponent AlCoCrFeNiMox alloys"],[8,"Al1 Co1 Cr1 Fe1 Nb0.1 Ni1","BCC","CAST","BCC",null,null,6.8,569,"C",25,1641,3285,null,17,null,182,null,null,null,"10.1016/j.msea.2011.10.110",2012,"Effect of Nb addition on the microstructure and properties of AlCoCrFeNi high-entropy alloy"],[8,"Al1 Co1 Cr1 Fe1 Nb0.25 Ni1","BCC+Sec.","CAST","other",null,null,6.8,668,"C",25,1959,3008,null,11,null,null,null,null,null,"10.1016/j.msea.2011.10.110",2012,"Effect of Nb addition on the microstructure and properties of AlCoCrFeNi high-entropy alloy"],[8,"Al1 Co1 Cr1 Fe1 Nb0.5 Ni1","BCC+Sec.","CAST","other",null,null,7,747,"C",25,2473,3170,null,4,null,null,null,null,null,"10.1016/j.msea.2011.10.110",2012,"Effect of Nb addition on the microstructure and properties of AlCoCrFeNi high-entropy alloy"],[8,"Al1 Co1 Cr1 Fe1 Nb0.75 Ni1","BCC+Sec.","CAST","other",null,null,7,null,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2011.10.110",2012,"Effect of Nb addition on the microstructure and properties of AlCoCrFeNi high-entropy alloy"],[8,"Al1 Co1 Cr1 Fe1 Ni1","BCC","CAST","BCC",null,null,6.7,520,"C",25,1373,3531,null,25,null,184,null,null,null,"10.1016/j.msea.2011.10.110",2012,"Effect of Nb addition on the microstructure and properties of AlCoCrFeNi high-entropy alloy"],[9,"Al0.8 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.3,270,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200900171",2010,"Microstructure and Mechanical Properties of New AlCoxCrFeMo0.5Ni High-Entropy Alloys"],[9,"Al1 Co0.5 Cr1 Fe1 Mo0.5 Ni1","BCC+Sec.","CAST","other",null,null,7,801,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200900171",2010,"Microstructure and Mechanical Properties of New AlCoxCrFeMo0.5Ni High-Entropy Alloys"],[9,"Al0.667 Co1 Cr0.667 Fe0.667 Mo0.333 Ni0.667","BCC+Sec.","CAST","other",null,null,7.2,741,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200900171",2010,"Microstructure and Mechanical Properties of New AlCoxCrFeMo0.5Ni High-Entropy Alloys"],[9,"Al0.5 Co1 Cr0.5 Fe0.5 Mo0.25 Ni0.5","FCC+BCC+Sec.","CAST","other",200,null,7.3,586,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200900171",2010,"Microstructure and Mechanical Properties of New AlCoxCrFeMo0.5Ni High-Entropy Alloys"],[9,"Al1 Co1 Cr1 Fe1 Mo0.5 Ni1","BCC+Sec.","CAST","other",null,null,7.1,796,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200900171",2010,"Microstructure and Mechanical Properties of New AlCoxCrFeMo0.5Ni High-Entropy Alloys"],[10,"Al1 Co1 Cr1 Fe1 Mo0.5 Ni1","BCC+Sec.","CAST","other",null,null,7.1,755,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.wear.2009.10.013",2010,"Effect of iron content on wear behavior of AlCoCrFexMo0.5Ni high-entropy alloys"],[10,"Al1 Co1 Cr1 Fe0.6 Mo0.5 Ni1","BCC+Sec.","CAST","other",null,null,7.1,754,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.wear.2009.10.013",2010,"Effect of iron content on wear behavior of AlCoCrFexMo0.5Ni high-entropy alloys"],[10,"Al0.667 Co0.667 Cr0.667 Fe1 Mo0.333 Ni0.667","BCC+Sec.","CAST","other",null,null,7.2,635,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.wear.2009.10.013",2010,"Effect of iron content on wear behavior of AlCoCrFexMo0.5Ni high-entropy alloys"],[10,"Al0.5 Co0.5 Cr0.5 Fe1 Mo0.25 Ni0.5","BCC+Sec.","CAST","other",null,null,7.2,639,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.wear.2009.10.013",2010,"Effect of iron content on wear behavior of AlCoCrFexMo0.5Ni high-entropy alloys"],[11,"Al1 Co1 Cr1 Fe1 Mo0.5","B2+Sec.","CAST","other",null,null,6.8,857,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.intermet.2012.09.008",2013,"On microstructure and mechanical performance of AlCoCrFeMo0.5Nix high-entropy alloys"],[11,"Al1 Co1 Cr1 Fe1 Mo0.5 Ni1","B2+Sec.","CAST","other",null,null,7.1,730,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.intermet.2012.09.008",2013,"On microstructure and mechanical performance of AlCoCrFeMo0.5Nix high-entropy alloys"],[11,"Al1 Co1 Cr1 Fe1 Mo0.5 Ni0.5","B2+Sec.","CAST","other",null,null,7,708,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.intermet.2012.09.008",2013,"On microstructure and mechanical performance of AlCoCrFeMo0.5Nix high-entropy alloys"],[11,"Al0.667 Co0.667 Cr0.667 Fe0.667 Mo0.333 Ni1","FCC+B2+Sec.","CAST","other",null,null,7.2,586,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.intermet.2012.09.008",2013,"On microstructure and mechanical performance of AlCoCrFeMo0.5Nix high-entropy alloys"],[11,"Al0.5 Co0.5 Cr0.5 Fe0.5 Mo0.25 Ni1","FCC+B2+Sec.","CAST","other",null,null,7.3,395,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.intermet.2012.09.008",2013,"On microstructure and mechanical performance of AlCoCrFeMo0.5Nix high-entropy alloys"],[12,"Al1 Co1 Cr1 Fe1 Ni1","BCC","CAST","BCC",null,null,6.7,null,"C",25,1110,null,null,null,null,184,null,null,null,"10.1016/j.msea.2010.07.049",2010,"Synthesis and properties of multiprincipal component AlCoCrFeNiSix alloys"],[12,"Al1 Co1 Cr1 Fe1 Ni1 Si1","BCC+Sec.","CAST","other",null,null,5.7,null,"C",25,2411,2950,null,1,null,null,null,null,null,"10.1016/j.msea.2010.07.049",2010,"Synthesis and properties of multiprincipal component AlCoCrFeNiSix alloys"],[12,"Al1 Co1 Cr1 Fe1 Ni1 Si0.2","BCC","CAST","BCC",null,null,6.5,null,"C",25,1265,2173,null,14,null,176,null,null,null,"10.1016/j.msea.2010.07.049",2010,"Synthesis and properties of multiprincipal component AlCoCrFeNiSix alloys"],[12,"Al1 Co1 Cr1 Fe1 Ni1 Si0.4","BCC","CAST","BCC",null,null,6.2,null,"C",25,1481,2444,null,13,null,169,null,null,null,"10.1016/j.msea.2010.07.049",2010,"Synthesis and properties of multiprincipal component AlCoCrFeNiSix alloys"],[12,"Al1 Co1 Cr1 Fe1 Ni1 Si0.6","BCC","CAST","BCC",null,null,6,null,"C",25,1834,2195,null,3,null,162,null,null,null,"10.1016/j.msea.2010.07.049",2010,"Synthesis and properties of multiprincipal component AlCoCrFeNiSix alloys"],[12,"Al1 Co1 Cr1 Fe1 Ni1 Si0.8","BCC+Sec.","CAST","other",null,null,5.8,null,"C",25,2179,2664,null,2,null,null,null,null,null,"10.1016/j.msea.2010.07.049",2010,"Synthesis and properties of multiprincipal component AlCoCrFeNiSix alloys"],[13,"Al0.25 Co1 Cr1 Cu0.5 Fe1 Ni1 Ti0.5","FCC","CAST","FCC",null,null,7.4,null,null,25,null,null,null,null,null,192,null,null,null,"10.1016/j.jallcom.2008.11.059",2009,"Atomic packing efficiency and phase transition in a high entropy alloy"],[13,"Al0.25 Co1 Cr1 Cu0.75 Fe1 Ni1 Ti0.5","FCC","CAST","FCC",null,null,7.5,null,"C",25,750,1970,null,39,103,189,null,null,null,"10.1016/j.jallcom.2008.11.059",2009,"Atomic packing efficiency and phase transition in a high entropy alloy"],[13,"Al0.5 Co1 Cr1 Cu0.5 Fe1 Ni1 Ti0.5","FCC+BCC","CAST","other",null,null,7.1,null,"C",25,1580,2389,null,17,161,null,null,null,null,"10.1016/j.jallcom.2008.11.059",2009,"Atomic packing efficiency and phase transition in a high entropy alloy"],[13,"Al0.75 Co1 Cr1 Cu0.25 Fe1 Ni1 Ti0.5","FCC+BCC","CAST","other",null,null,6.8,null,"C",25,1900,2697,null,12,164,null,null,null,null,"10.1016/j.jallcom.2008.11.059",2009,"Atomic packing efficiency and phase transition in a high entropy alloy"],[13,"Co1 Cr1 Cu1 Fe1 Ni1 Ti0.5","FCC","CAST","FCC",null,null,7.8,null,"C",25,700,1650,null,29,93,194,null,null,null,"10.1016/j.jallcom.2008.11.059",2009,"Atomic packing efficiency and phase transition in a high entropy alloy"],[14,"Al1 Co1 Cr1 Fe1 Ni1 Ti1","BCC+BCC","CAST","other",null,null,6.2,null,"C",25,1860,2580,10.7,9,90,null,null,null,null,"10.1063/1.2734517",2007,"Solid solution alloys of AlCoCrFeNiTix with excellent room-temperature mechanical properties"],[14,"Al1 Co1 Cr1 Fe1 Ni1 Ti0.5","BCC+BCC","CAST","other",null,null,6.4,178,"C",25,2260,3140,24.4,23,178,null,null,null,null,"10.1063/1.2734517",2007,"Solid solution alloys of AlCoCrFeNiTix with excellent room-temperature mechanical properties"],[14,"Al0.667 Co0.667 Cr0.667 Fe0.667 Ni0.667 Ti1","BCC+Sec.","CAST","other",null,null,6.1,null,"C",25,2220,2720,6.6,5,160,null,null,null,null,"10.1063/1.2734517",2007,"Solid solution alloys of AlCoCrFeNiTix with excellent room-temperature mechanical properties"],[15,"Al1 Co1 Cr1 Fe1 Ni1 Ti1 V1 Zr1",null,"CAST","other",null,null,6.3,780,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Al1 Co1 Fe1 Ni1 Ti1 V1 Zr1","BCC","CAST","BCC",null,null,6.2,790,null,25,null,null,null,null,null,129,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Al1 Fe1 Ni1 Ti1 V1 Zr1","BCC","CAST","BCC",null,null,5.9,800,null,25,null,null,null,null,null,119,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Co1 Cr1 Cu1 Fe1 Ni1 Ti1 V1 Zr1",null,"CAST","other",null,null,7.1,680,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Co1 Cr1 Fe1 Mo1 Ni1 Ti1 V1 Zr1",null,"CAST","other",null,null,7.3,850,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Co1 Cu1 Fe1 Ni1 Ti1 V1 Zr1",null,"CAST","other",null,null,7.1,630,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Co1 Fe1 Mo1 Ni1 Ti1 V1 Zr1",null,"CAST","other",null,null,7.3,790,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Cu1 Fe1 Ni1 Ti1 V1 Zr1",null,"CAST","other",null,null,6.8,590,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[15,"Fe1 Mo1 Ni1 Ti1 V1 Zr1","BCC+Sec.","CAST","other",null,null,7.1,740,null,25,null,null,null,null,null,null,null,null,null,"10.1002/adem.200300567",2004,"Nanostructured High-Entropy Alloys with Multiple Principal Elements: Novel Alloy Design Concepts and Outcomes"],[16,"Al0.1 Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,7.9,180,null,25,null,null,null,null,null,216,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.2 Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,7.7,171,"T",25,220,null,56,null,null,212,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.38 Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,7.5,182,"T",25,244,null,45,null,null,205,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.43 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.4,183,"T",25,285,null,35,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.49 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.4,220,"T",25,331,null,29,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.56 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.3,278,"T",25,526,null,16,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.62 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.2,405,"T",25,833,null,5,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.68 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.2,486,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.75 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.1,530,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.81 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7,539,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.88 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7,533,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al0.95 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,6.9,535,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Al1 Co0.8 Cr0.8 Fe0.8 Mn0.8 Ni0.8","BCC","CAST","BCC",null,null,6.6,539,null,25,null,null,null,null,null,180,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[16,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8,176,"T",25,208,null,62,null,null,220,null,null,null,"10.1016/j.actamat.2013.09.037",2014,"Effects of Al addition on structural evolution and tensile properties of the FeCoNiCrMn high-entropy alloy system"],[17,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",300,null,8,144,"C",25,null,null,null,null,null,220,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","ANNEAL","FCC",350,null,8,135,"C",25,230,null,null,75,null,220,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V0.25","FCC","CAST","FCC",120,null,7.9,151,"C",25,null,null,null,null,null,215,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V0.25","FCC+Sec.","ANNEAL","other",130,null,7.9,144,"C",25,200,null,null,75,null,null,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V0.5","FCC","CAST","FCC",220,null,7.8,186,"C",25,null,null,null,null,null,210,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V0.5","FCC","ANNEAL","FCC",250,null,7.8,275,"C",25,620,null,null,75,null,210,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V0.75","FCC+Sec.","ANNEAL","other",500,null,7.7,380,"C",25,740,1325,null,7.8,null,null,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V0.75","FCC+Sec.","CAST","other",500,null,7.7,342,"C",25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V1","FCC+Sec.","ANNEAL","other",12,null,7.7,636,"C",25,1660,1845,null,0.5,null,null,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[17,"Co1 Cr1 Fe1 Mn1 Ni1 V1","FCC+Sec.","CAST","other",null,null,7.7,650,"C",25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2014.12.157",2015,"Effect of V content on microstructure and mechanical properties of the CoCrFeMnNiVx high entropy alloys"],[18,"Al0.2 Cr0.667 Fe1 Mn0.667 Ni0.333","FCC+BCC","CAST","other",null,null,7.2,297,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2010.05.052",2010,"Microstructure and properties of age-hardenable AlxCrFe1.5MnNi0.5 alloys"],[18,"Al0.333 Cr0.667 Fe1 Mn0.667 Ni0.333","BCC","CAST","BCC",null,null,7,396,null,25,null,null,null,null,null,201,null,null,null,"10.1016/j.msea.2010.05.052",2010,"Microstructure and properties of age-hardenable AlxCrFe1.5MnNi0.5 alloys"],[19,"Al1 Cr1 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.3,472,"C",25,1406,2927,null,29,null,null,null,null,null,"10.1016/j.jallcom.2013.03.253",2013,"Microstructure and mechanical properties of multi-component AlCrFeNiMox high-entropy alloys"],[19,"Al1 Cr1 Fe1 Mo1 Ni1","B2+Sec.","CAST","other",null,null,7.2,905,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2013.03.253",2013,"Microstructure and mechanical properties of multi-component AlCrFeNiMox high-entropy alloys"],[19,"Al1 Cr1 Fe1 Mo0.2 Ni1","BCC+B2","CAST","other",null,null,6.5,549,"C",25,1487,3222,null,29,null,null,null,null,null,"10.1016/j.jallcom.2013.03.253",2013,"Microstructure and mechanical properties of multi-component AlCrFeNiMox high-entropy alloys"],[19,"Al1 Cr1 Fe1 Mo0.5 Ni1","BCC+B2","CAST","other",null,null,6.8,622,"C",25,1749,2644,null,13,null,null,null,null,null,"10.1016/j.jallcom.2013.03.253",2013,"Microstructure and mechanical properties of multi-component AlCrFeNiMox high-entropy alloys"],[19,"Al1 Cr1 Fe1 Mo0.8 Ni1","B2+Sec.","CAST","other",null,null,7,854,"C",25,1513,1513,null,0,null,null,null,null,null,"10.1016/j.jallcom.2013.03.253",2013,"Microstructure and mechanical properties of multi-component AlCrFeNiMox high-entropy alloys"],[20,"Al0.3 Co1 Cr1 Cu1 Fe1","FCC",null,"FCC",null,null,7.7,180,null,25,null,null,null,null,null,194,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al0.5 Co1 Cr1 Cu1 Fe1","FCC",null,"FCC",null,null,7.4,207,null,25,null,null,null,null,null,187,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al0.8 Co1 Cr1 Cu1 Fe1","FCC+BCC",null,"other",null,null,7,271,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.769 Cr0.769 Cu0.769 Fe0.769","FCC+BCC",null,"other",null,null,6.5,476,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.667 Cr0.667 Cu0.667 Fe0.667","FCC+BCC",null,"other",null,null,6.3,510,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.556 Cr0.556 Cu0.556 Fe0.556","FCC+BCC",null,"other",null,null,6.1,557,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5","FCC+BCC",null,"other",null,null,5.9,567,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.435 Cr0.435 Cu0.435 Fe0.435","FCC+BCC",null,"other",null,null,5.7,603,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.4 Cr0.4 Cu0.4 Fe0.4","FCC+BCC",null,"other",null,null,5.6,624,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.357 Cr0.357 Cu0.357 Fe0.357","BCC",null,"BCC",null,null,5.5,657,null,25,null,null,null,null,null,139,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co0.333 Cr0.333 Cu0.333 Fe0.333","BCC",null,"BCC",null,null,5.4,644,null,25,null,null,null,null,null,136,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Al1 Co1 Cr1 Cu1 Fe1","FCC+BCC",null,"other",null,null,6.8,407,null,25,null,null,null,null,null,null,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[20,"Co1 Cr1 Cu1 Fe1","FCC",null,"FCC",null,null,8.2,134,null,25,null,null,null,null,null,208,null,null,null,"10.3166/acsm.31.633-648",2006,"Recent progress in high-entropy alloys"],[21,"Co1 Cr1 Cu0.5 Fe1 Ni1","FCC","CAST","FCC",null,null,8.3,172,null,25,null,null,null,null,null,215,null,null,null,"10.1016/j.intermet.2010.03.030",2010,"Effect of aging treatment on microstructure and properties of high-entropy Cu0.5CoCrFeNi alloy"],[22,"Co1 Cr1 Cu1 Fe1 Ni1 Ti1","FCC","CAST","FCC",null,null,7.4,null,"C",25,1272,1272,2,0,null,185,null,null,null,"10.1016/j.intermet.2006.08.005",2007,"Novel microstructure and properties of multicomponent CoCrCuFeNiTix alloys"],[22,"Co1 Cr1 Cu1 Fe1 Ni1 Ti0.5","FCC","CAST","FCC",null,null,7.8,null,"C",25,700,1650,22,21.6,null,194,null,null,null,"10.1016/j.intermet.2006.08.005",2007,"Novel microstructure and properties of multicomponent CoCrCuFeNiTix alloys"],[22,"Co1 Cr1 Cu1 Fe1 Ni1 Ti0.8","FCC+Sec.","CAST","other",null,null,7.6,null,"C",25,1042,1848,3,2.11,null,null,null,null,null,"10.1016/j.intermet.2006.08.005",2007,"Novel microstructure and properties of multicomponent CoCrCuFeNiTix alloys"],[23,"Al0.5 B0.2 Co1 Cr1 Cu1 Fe1 Ni1",null,"CAST","other",null,null,7.5,415,null,25,null,null,null,null,null,null,null,null,null,"10.1149/1.2744133",2007,"The Effect of Boron on the Corrosion Resistance of the High Entropy Alloys Al[sub 0.5]CoCrCuFeNiB[sub x]"],[23,"Al0.5 B0.6 Co1 Cr1 Cu1 Fe1 Ni1",null,"CAST","other",null,null,7.3,505,null,25,null,null,null,null,null,null,null,null,null,"10.1149/1.2744133",2007,"The Effect of Boron on the Corrosion Resistance of the High Entropy Alloys Al[sub 0.5]CoCrCuFeNiB[sub x]"],[23,"Al0.5 B1 Co1 Cr1 Cu1 Fe1 Ni1",null,"CAST","other",null,null,7.1,736,null,25,null,null,null,null,null,null,null,null,null,"10.1149/1.2744133",2007,"The Effect of Boron on the Corrosion Resistance of the High Entropy Alloys Al[sub 0.5]CoCrCuFeNiB[sub x]"],[23,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1",null,"CAST","other",null,null,7.6,300,null,25,null,null,null,null,null,null,null,null,null,"10.1149/1.2744133",2007,"The Effect of Boron on the Corrosion Resistance of the High Entropy Alloys Al[sub 0.5]CoCrCuFeNiB[sub x]"],[24,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,225,null,25,null,null,null,null,null,189,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 Ti1","FCC+BCC+Sec.","CAST","other",null,null,7,636,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 Ti0.2","FCC","CAST","FCC",null,null,7.5,272,null,25,null,null,null,null,null,185,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 Ti0.4","FCC","CAST","FCC",null,null,7.3,321,null,25,null,null,null,null,null,182,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 Ti0.6","FCC+BCC","CAST","other",null,null,7.2,458,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 Ti0.8","FCC+BCC+Sec.","CAST","other",null,null,7.1,590,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.417 Co0.833 Cr0.833 Cu0.833 Fe0.833 Ni0.833 Ti1","FCC+BCC+Sec.","CAST","other",null,null,6.9,646,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.357 Co0.714 Cr0.714 Cu0.714 Fe0.714 Ni0.714 Ti1","FCC+BCC+B2+Sec.","CAST","other",null,null,6.8,664,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.312 Co0.625 Cr0.625 Cu0.625 Fe0.625 Ni0.625 Ti1","FCC+BCC+B2+Sec.","CAST","other",null,null,6.7,657,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.278 Co0.556 Cr0.556 Cu0.556 Fe0.556 Ni0.556 Ti1","FCC+BCC+B2+Sec.","CAST","other",null,null,6.6,667,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[24,"Al0.25 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5 Ti1","FCC+BCC+B2+Sec.","CAST","other",null,null,6.5,696,null,25,null,null,null,null,null,null,null,null,null,"10.2320/matertrans.47.1395",2006,"Microstructure and Properties of AlSUB0.5/SUBCoCrCuFeNiTiISUBx/SUB/I (Ix/I=0&ndash;2.0) High-Entropy Alloys"],[25,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,215,null,25,null,null,null,null,null,189,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 V0.2","FCC","CAST","FCC",null,null,7.6,204,null,25,null,null,null,null,null,186,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 V0.4","FCC+BCC","CAST","other",null,null,7.5,231,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 V0.6","FCC+BCC+Sec.","CAST","other",null,null,7.5,328,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 V0.8","FCC+BCC+Sec.","CAST","other",null,null,7.4,447,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1 V1","FCC+BCC+Sec.","CAST","other",null,null,7.4,639,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.417 Co0.833 Cr0.833 Cu0.833 Fe0.833 Ni0.833 V1","BCC","CAST","BCC",null,null,7.3,579,null,25,null,null,null,null,null,177,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.357 Co0.714 Cr0.714 Cu0.714 Fe0.714 Ni0.714 V1","BCC","CAST","BCC",null,null,7.3,577,null,25,null,null,null,null,null,175,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.312 Co0.625 Cr0.625 Cu0.625 Fe0.625 Ni0.625 V1","BCC","CAST","BCC",null,null,7.2,594,null,25,null,null,null,null,null,174,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.278 Co0.556 Cr0.556 Cu0.556 Fe0.556 Ni0.556 V1","BCC","CAST","BCC",null,null,7.2,597,null,25,null,null,null,null,null,172,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[25,"Al0.25 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5 V1","BCC","CAST","BCC",null,null,7.2,587,null,25,null,null,null,null,null,171,null,null,null,"10.1007/s11661-006-0081-3",2006,"Effect of vanadium addition on the microstructure, hardness, and wear resistance of Al0.5CoCrCuFeNi high-entropy alloy"],[26,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,472,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2012.07.003",2012,"Effect of elemental interaction on microstructure and mechanical properties of FeCoNiCuAl alloys"],[26,"Al1 Co1 Cu1 Fe1 Nd1 Ni1","Other","CAST","other",null,null,7,578,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2012.07.003",2012,"Effect of elemental interaction on microstructure and mechanical properties of FeCoNiCuAl alloys"],[26,"Al1 Co1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7,536,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2012.07.003",2012,"Effect of elemental interaction on microstructure and mechanical properties of FeCoNiCuAl alloys"],[26,"Al1 Co1 Cu1 Fe1 Ni1 Si1","FCC+BCC","CAST","other",null,null,5.9,682,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2012.07.003",2012,"Effect of elemental interaction on microstructure and mechanical properties of FeCoNiCuAl alloys"],[26,"Al1 Co1 Cu1 Fe1 Ni1 Ti1","FCC+BCC","CAST","other",null,null,6.5,626,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2012.07.003",2012,"Effect of elemental interaction on microstructure and mechanical properties of FeCoNiCuAl alloys"],[26,"Al1 Co1 Cu1 Fe1 Ni1 Zr1","FCC+BCC+Sec.","CAST","other",null,null,6.9,472,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2012.07.003",2012,"Effect of elemental interaction on microstructure and mechanical properties of FeCoNiCuAl alloys"],[28,"Al1 Co1 Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC+Sec.","CAST","other",null,null,7.1,null,"C",25,1005,null,15,null,null,null,null,null,null,"10.1016/j.msea.2008.08.025",2008,"Effects of Mn, Ti and V on the microstructure and properties of AlCrFeCoNiCu high entropy alloy"],[28,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,null,"C",25,1303,null,24,null,null,null,null,null,null,"10.1016/j.msea.2008.08.025",2008,"Effects of Mn, Ti and V on the microstructure and properties of AlCrFeCoNiCu high entropy alloy"],[28,"Al1 Co1 Cr1 Cu1 Fe1 Ni1 Ti1","FCC+BCC+BCC","CAST","other",null,null,6.6,null,"C",25,1234,null,9,null,null,null,null,null,null,"10.1016/j.msea.2008.08.025",2008,"Effects of Mn, Ti and V on the microstructure and properties of AlCrFeCoNiCu high entropy alloy"],[28,"Al1 Co1 Cr1 Cu1 Fe1 Ni1 V1","FCC+BCC","CAST","other",null,null,6.9,null,"C",25,1469,null,16,null,null,null,null,null,null,"10.1016/j.msea.2008.08.025",2008,"Effects of Mn, Ti and V on the microstructure and properties of AlCrFeCoNiCu high entropy alloy"],[29,"Al1 Co1 Cr1 Cu1 Ni1 Ti1","BCC",null,"BCC",null,null,6.4,null,"C",25,null,1495,null,8,36,157,null,null,null,"10.1016/j.matdes.2009.09.016",2010,"Effect of rare earth Y addition on the microstructure and mechanical properties of high entropy AlCoCrCuNiTi alloys"],[29,"Al1 Co1 Cr1 Cu1 Ni1 Ti1 Y1","Other","CAST","other",null,null,5.8,null,"C",25,null,1192,null,4,37,null,null,null,null,"10.1016/j.matdes.2009.09.016",2010,"Effect of rare earth Y addition on the microstructure and mechanical properties of high entropy AlCoCrCuNiTi alloys"],[29,"Al1 Co1 Cr1 Cu1 Ni1 Ti1 Y0.5","Other","CAST","other",null,null,6.1,null,"C",25,null,1025,null,3,36,null,null,null,null,"10.1016/j.matdes.2009.09.016",2010,"Effect of rare earth Y addition on the microstructure and mechanical properties of high entropy AlCoCrCuNiTi alloys"],[29,"Al1 Co1 Cr1 Cu1 Ni1 Ti1 Y0.8","Other","CAST","other",null,null,5.9,null,"C",25,null,1325,null,5,38,null,null,null,null,"10.1016/j.matdes.2009.09.016",2010,"Effect of rare earth Y addition on the microstructure and mechanical properties of high entropy AlCoCrCuNiTi alloys"],[30,"Co1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.6,null,"T",25,null,480,null,15,null,187,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[30,"Co1 Cu1 Fe1 Ni1 Sn0.02","FCC","CAST","FCC",null,null,8.6,null,"T",25,null,548,null,17,null,185,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[30,"Co1 Cu1 Fe1 Ni1 Sn0.04","FCC+Sec.","CAST","other",null,null,8.6,null,"T",25,null,594,null,18,null,null,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[30,"Co1 Cu1 Fe1 Ni1 Sn0.05","FCC+Sec.","CAST","other",null,null,8.6,null,"T",25,null,615,null,20,null,null,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[30,"Co1 Cu1 Fe1 Ni1 Sn0.07","FCC+Sec.","CAST","other",null,null,8.6,null,"T",25,null,632,null,19,null,null,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[30,"Co1 Cu1 Fe1 Ni1 Sn0.1","FCC+Sec.","CAST","other",null,null,8.6,null,"T",25,null,602,null,5,null,null,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[30,"Co1 Cu1 Fe1 Ni1 Sn0.2","FCC+Sec.","CAST","other",null,null,8.5,null,"T",25,null,261,null,2,null,null,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[30,"Co1 Cu1 Fe1 Ni1 Sn0.5","FCC+Sec.","CAST","other",null,null,8.3,null,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2012.03.080",2012,"Microstructure and the properties of FeCoCuNiSnx high entropy alloys"],[31,"Co1 Cu1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8.4,208,"T",25,null,478,null,14,null,189,null,null,null,"10.1016/j.matdes.2012.08.019",2013,"Microstructure and tensile properties of FeMnNiCuCoSnx high entropy alloys"],[31,"Co1 Cu1 Fe1 Mn1 Ni1 Sn0.03","FCC","CAST","FCC",null,null,8.4,192,"T",25,null,465,null,18,null,187,null,null,null,"10.1016/j.matdes.2012.08.019",2013,"Microstructure and tensile properties of FeMnNiCuCoSnx high entropy alloys"],[31,"Co1 Cu1 Fe1 Mn1 Ni1 Sn0.05","FCC+Sec.","CAST","other",null,null,8.4,205,"T",25,null,475,null,12,null,null,null,null,null,"10.1016/j.matdes.2012.08.019",2013,"Microstructure and tensile properties of FeMnNiCuCoSnx high entropy alloys"],[31,"Co1 Cu1 Fe1 Mn1 Ni1 Sn0.08","FCC+Sec.","CAST","other",null,null,8.3,219,"T",25,null,425,null,7,null,null,null,null,null,"10.1016/j.matdes.2012.08.019",2013,"Microstructure and tensile properties of FeMnNiCuCoSnx high entropy alloys"],[31,"Co1 Cu1 Fe1 Mn1 Ni1 Sn0.1","FCC+Sec.","CAST","other",null,null,8.3,253,"T",25,null,470,null,6,null,null,null,null,null,"10.1016/j.matdes.2012.08.019",2013,"Microstructure and tensile properties of FeMnNiCuCoSnx high entropy alloys"],[31,"Co1 Cu1 Fe1 Mn1 Ni1 Sn0.2","FCC+Sec.","CAST","other",null,null,8.3,319,"T",25,null,368,null,2,null,null,null,null,null,"10.1016/j.matdes.2012.08.019",2013,"Microstructure and tensile properties of FeMnNiCuCoSnx high entropy alloys"],[32,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,6.8,495,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2011.11.063",2012,"Microstructure and properties of AlCrFeCuNix (0.6≤x≤1.4) high-entropy alloys"],[32,"Al1 Cr1 Cu1 Fe1 Ni0.6","FCC+BCC","ANNEAL","other",null,null,6.6,496,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2011.11.063",2012,"Microstructure and properties of AlCrFeCuNix (0.6≤x≤1.4) high-entropy alloys"],[32,"Al1 Cr1 Cu1 Fe1 Ni0.8","FCC+BCC","ANNEAL","other",null,null,6.7,486,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2011.11.063",2012,"Microstructure and properties of AlCrFeCuNix (0.6≤x≤1.4) high-entropy alloys"],[32,"Al0.833 Cr0.833 Cu0.833 Fe0.833 Ni1","FCC+BCC","ANNEAL","other",null,null,6.8,407,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2011.11.063",2012,"Microstructure and properties of AlCrFeCuNix (0.6≤x≤1.4) high-entropy alloys"],[32,"Al0.714 Cr0.714 Cu0.714 Fe0.714 Ni1","FCC+BCC","ANNEAL","other",null,null,6.9,367,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2011.11.063",2012,"Microstructure and properties of AlCrFeCuNix (0.6≤x≤1.4) high-entropy alloys"],[32,"Co1 Cr1 Cu1 Fe1 Ni1","FCC",null,"FCC",null,null,8.3,286,"C",25,230,888,51,null,null,206,null,null,null,"10.1016/j.msea.2011.11.063",2012,"Microstructure and properties of AlCrFeCuNix (0.6≤x≤1.4) high-entropy alloys"],[33,"Al0.1 Cr0.5 Cu0.5 Fe0.5 Ni1","FCC",null,"FCC",null,null,8,null,null,25,null,null,null,null,null,197,null,null,null,"10.1063/1.3587228",2011,"Effect of valence electron concentration on stability of fcc or bcc phase in high entropy alloys"],[33,"Al0.2 Cr0.5 Cu0.5 Fe0.5 Ni1","FCC",null,"FCC",null,null,7.8,null,null,25,null,null,null,null,null,190,null,null,null,"10.1063/1.3587228",2011,"Effect of valence electron concentration on stability of fcc or bcc phase in high entropy alloys"],[33,"Al0.3 Cr0.5 Cu0.5 Fe0.5 Ni1","FCC",null,"FCC",null,null,7.5,null,null,25,null,null,null,null,null,185,null,null,null,"10.1063/1.3587228",2011,"Effect of valence electron concentration on stability of fcc or bcc phase in high entropy alloys"],[33,"Al0.4 Cr0.5 Cu0.5 Fe0.5 Ni1","FCC",null,"FCC",null,null,7.3,null,null,25,null,null,null,null,null,179,null,null,null,"10.1063/1.3587228",2011,"Effect of valence electron concentration on stability of fcc or bcc phase in high entropy alloys"],[33,"Al0.6 Cr0.5 Cu0.5 Fe0.5 Ni1","FCC+BCC",null,"other",null,null,6.9,null,null,25,null,null,null,null,null,null,null,null,null,"10.1063/1.3587228",2011,"Effect of valence electron concentration on stability of fcc or bcc phase in high entropy alloys"],[33,"Al0.5 Cr0.5 Cu0.5 Fe0.5 Ni1","FCC+BCC",null,"other",null,null,7.1,null,null,25,null,null,null,null,null,null,null,null,null,"10.1063/1.3587228",2011,"Effect of valence electron concentration on stability of fcc or bcc phase in high entropy alloys"],[34,"Al1 Cr1 Cu1 Fe1 Ni1 Ti1","BCC+BCC+Sec.","ANNEAL","other",null,null,6.3,null,"C",25,null,1219,null,null,null,null,null,null,null,"10.1016/j.jallcom.2011.02.108",2011,"Microstructure and property of AlTiCrFeNiCu high-entropy alloy"],[35,"Al1 Cu0.889 Fe0.667 Ni0.889 Ti1","FCC","CAST","FCC",null,null,5.9,516,"C",25,980,1326,null,7,145,131,null,null,null,"10.1016/j.apsusc.2015.07.207",2015,"Effect of iron content on the structure and mechanical properties of Al25Ti25Ni25Cu25 and (AlTi)60-xNi20Cu20Fex (x=15, 20) high-entropy alloys"],[35,"Al1 Cu0.889 Fe0.667 Ni0.889 Ti1","FCC","CAST","FCC",null,null,5.9,516,"C",25,980,1326,null,7,145,131,null,null,null,"10.1016/j.apsusc.2015.07.207",2015,"Effect of iron content on the structure and mechanical properties of Al25Ti25Ni25Cu25 and (AlTi)60-xNi20Cu20Fex (x=15, 20) high-entropy alloys"],[35,"Al1 Cu1 Fe1 Ni1 Ti1","FCC","CAST","FCC",null,null,6.1,516,"C",25,1074,1617,null,8,146,137,null,null,null,"10.1016/j.apsusc.2015.07.207",2015,"Effect of iron content on the structure and mechanical properties of Al25Ti25Ni25Cu25 and (AlTi)60-xNi20Cu20Fex (x=15, 20) high-entropy alloys"],[35,"Al1 Cu1 Ni1 Ti1","FCC","CAST","FCC",null,null,5.7,537,"C",25,300,536,null,0.85,108,122,null,null,null,"10.1016/j.apsusc.2015.07.207",2015,"Effect of iron content on the structure and mechanical properties of Al25Ti25Ni25Cu25 and (AlTi)60-xNi20Cu20Fex (x=15, 20) high-entropy alloys"],[36,"Al1 Cu0.062 Li0.062 Mg0.062 Zn0.062","FCC+Sec.","CAST","other",null,null,2.9,null,"C",25,488,879,17,null,null,null,null,null,null,"10.1007/s11837-015-1402-z",2015,"Erratum to: Phase Stability of Low-Density, Multiprincipal Component Alloys Containing Aluminum, Magnesium, and Lithium"],[36,"Al1 Li0.062 Mg0.062 Sn0.062 Zn0.062","FCC+Sec.","CAST","other",null,null,3,null,"C",25,415,836,16,null,null,null,null,null,null,"10.1007/s11837-015-1402-z",2015,"Erratum to: Phase Stability of Low-Density, Multiprincipal Component Alloys Containing Aluminum, Magnesium, and Lithium"],[36,"Al1 Cu0.2 Li0.5 Mg1 Zn0.5","Other","CAST","other",null,null,2.7,null,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11837-015-1402-z",2015,"Erratum to: Phase Stability of Low-Density, Multiprincipal Component Alloys Containing Aluminum, Magnesium, and Lithium"],[36,"Al1 Cu0.5 Li0.5 Mg1 Sn0.2","Other","CAST","other",null,null,3,null,null,25,null,null,null,null,null,null,null,null,null,"10.1007/s11837-015-1402-z",2015,"Erratum to: Phase Stability of Low-Density, Multiprincipal Component Alloys Containing Aluminum, Magnesium, and Lithium"],[36,"Al1 Li0.5 Mg1 Sn0.2 Zn0.5","FCC+Sec.","CAST","other",null,null,2.9,null,"C",25,546,546,null,null,null,null,null,null,null,"10.1007/s11837-015-1402-z",2015,"Erratum to: Phase Stability of Low-Density, Multiprincipal Component Alloys Containing Aluminum, Magnesium, and Lithium"],[36,"Al1 Li1 Mg1 Sn1 Zn1","FCC+HCP+Sec.","CAST","other",null,null,3.9,null,"C",25,600,615,1,null,null,null,null,null,null,"10.1007/s11837-015-1402-z",2015,"Erratum to: Phase Stability of Low-Density, Multiprincipal Component Alloys Containing Aluminum, Magnesium, and Lithium"],[38,"Al0.667 Li0.667 Mg0.333 Sc0.667 Ti1","FCC+HCP",null,"other",null,null,2.7,591,null,25,null,null,null,null,null,null,null,null,null,"10.1080/21663831.2014.985855",2014,"A Novel Low-Density, High-Hardness, High-entropy Alloy with Close-packed Single-phase Nanocrystalline Structures"],[39,"Al0.214 Nb0.714 Ta0.571 Ti1 V0.143 Zr0.929","BCC","OTHER","BCC",180,null,7.7,500,"C",25,1965,2061,5,null,null,107,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.214 Nb0.714 Ta0.571 Ti1 V0.143 Zr0.929","BCC","OTHER","BCC",null,null,7.7,null,"C",800,678,693,50,null,null,107,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.214 Nb0.714 Ta0.571 Ti1 V0.143 Zr0.929","BCC","OTHER","BCC",null,null,7.7,null,"C",1000,166,189,50,null,null,107,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.214 Nb0.714 Ta0.714 Ti1 Zr0.929","BCC","OTHER","BCC",200,null,8.1,490,"C",25,1965,2054,5,null,null,110,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.214 Nb0.714 Ta0.714 Ti1 Zr0.929","BCC","OTHER","BCC",null,null,8.1,null,"C",800,362,397,50,null,null,110,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.214 Nb0.714 Ta0.714 Ti1 Zr0.929","BCC","OTHER","BCC",null,null,8.1,null,"C",1000,236,247,50,null,null,110,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.4 Hf0.6 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",140,null,9.1,500,"C",25,1841,2269,10,null,null,107,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.4 Hf0.6 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.1,null,"C",800,796,834,50,null,null,107,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.4 Hf0.6 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.1,null,"C",1000,298,455,50,null,null,107,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.333 Nb0.667 Ta0.533 Ti1 V0.133 Zr0.667","BCC","OTHER","BCC",100,null,7.6,530,"C",25,2035,2105,5,null,null,109,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.333 Nb0.667 Ta0.533 Ti1 V0.133 Zr0.667","BCC","OTHER","BCC",null,null,7.6,null,"C",800,796,851,50,null,null,109,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.333 Nb0.667 Ta0.533 Ti1 V0.133 Zr0.667","BCC","OTHER","BCC",null,null,7.6,null,"C",1000,220,225,50,null,null,109,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+B2","OTHER","other",75,null,7.1,591,"C",25,2000,2368,10,null,null,null,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+B2","OTHER","other",null,null,7.1,null,"C",800,1597,1810,11,null,null,null,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+B2","OTHER","other",null,null,7.1,null,"C",1000,745,772,50,null,null,null,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+B2","OTHER","other",null,null,7.1,null,"C",1200,250,null,50,null,null,null,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.667 Nb1 Ta0.333 Ti1 Zr0.333","BCC","OTHER","BCC",2000,null,6.8,408,"C",25,1280,1367,4,null,null,105,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.667 Nb1 Ta0.333 Ti1 Zr0.333","BCC","OTHER","BCC",null,null,6.8,null,"C",800,728,1000,12,null,null,105,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[39,"Al0.667 Nb1 Ta0.333 Ti1 Zr0.333","BCC","OTHER","BCC",null,null,6.8,null,"C",1000,403,415,50,null,null,105,null,null,null,"10.1007/s11837-014-1066-0",2014,"Microstructure and Properties of Aluminum-Containing Refractory High-Entropy Alloys"],[40,"Al0.3 Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",105,null,9.5,353,"C",25,1188,null,50,null,63,105,null,null,null,"10.1016/j.jallcom.2014.11.064",2015,"Effect of Al addition on mechanical properties and microstructure of refractory AlxHfNbTaTiZr alloys"],[40,"Al0.5 Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",86,null,9.3,396,"C",25,1302,null,46,null,97,104,null,null,null,"10.1016/j.jallcom.2014.11.064",2015,"Effect of Al addition on mechanical properties and microstructure of refractory AlxHfNbTaTiZr alloys"],[40,"Al0.75 Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",89,null,9.1,427,"C",25,1415,2050,30,null,102,103,null,null,null,"10.1016/j.jallcom.2014.11.064",2015,"Effect of Al addition on mechanical properties and microstructure of refractory AlxHfNbTaTiZr alloys"],[40,"Al1 Hf1 Nb1 Ta1 Ti1 Zr1","BCC+Sec.","CAST","other",92,null,8.9,441,"C",25,1489,1900,21,null,103,null,null,null,null,"10.1016/j.jallcom.2014.11.064",2015,"Effect of Al addition on mechanical properties and microstructure of refractory AlxHfNbTaTiZr alloys"],[40,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",120,null,9.9,295,"C",25,1073,null,50,null,55,107,null,null,null,"10.1016/j.jallcom.2014.11.064",2015,"Effect of Al addition on mechanical properties and microstructure of refractory AlxHfNbTaTiZr alloys"],[41,"Al0.25 Mo1 Nb1 Ti1 V1","BCC","CAST","BCC",null,null,7.1,460,"C",25,1250,1500,13,null,null,161,null,null,null,"10.3390/e16020870",2014,"Microstructures and Crackling Noise of AlxNbTiMoV High Entropy Alloys"],[41,"Al0.5 Mo1 Nb1 Ti1 V1","BCC","CAST","BCC",null,null,6.8,487,"C",25,1625,1800,11,null,null,156,null,null,null,"10.3390/e16020870",2014,"Microstructures and Crackling Noise of AlxNbTiMoV High Entropy Alloys"],[41,"Al0.75 Mo1 Nb1 Ti1 V1","BCC","CAST","BCC",null,null,6.6,517,"C",25,1260,1300,8,null,null,151,null,null,null,"10.3390/e16020870",2014,"Microstructures and Crackling Noise of AlxNbTiMoV High Entropy Alloys"],[41,"Al1 Mo1 Nb1 Ti1 V1","BCC","CAST","BCC",null,null,6.4,537,"C",25,1375,1400,3,null,null,147,null,null,null,"10.3390/e16020870",2014,"Microstructures and Crackling Noise of AlxNbTiMoV High Entropy Alloys"],[41,"Mo1 Nb1 Ti1 V1","BCC","CAST","BCC",null,null,7.3,441,"C",25,1200,2050,26,null,null,167,null,null,null,"10.3390/e16020870",2014,"Microstructures and Crackling Noise of AlxNbTiMoV High Entropy Alloys"],[42,"Al0.25 Nb1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,8.8,null,"C",25,1330,null,null,null,null,130,null,null,null,"10.1016/j.proeng.2012.03.043",2012,"Microstructure and Compressive Properties of NbTiVTaAlx High Entropy Alloys"],[42,"Al0.5 Nb1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,8.5,null,"C",25,1014,null,null,null,null,127,null,null,null,"10.1016/j.proeng.2012.03.043",2012,"Microstructure and Compressive Properties of NbTiVTaAlx High Entropy Alloys"],[42,"Al1 Nb1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,7.9,null,"C",25,993,null,null,null,null,122,null,null,null,"10.1016/j.proeng.2012.03.043",2012,"Microstructure and Compressive Properties of NbTiVTaAlx High Entropy Alloys"],[42,"Nb1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,9.2,null,"C",25,1092,null,null,null,null,134,null,null,null,"10.1016/j.proeng.2012.03.043",2012,"Microstructure and Compressive Properties of NbTiVTaAlx High Entropy Alloys"],[43,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",350,null,5.5,448,"C",25,1020,1318,null,5,null,104,null,null,null,"10.1016/j.matlet.2014.11.162",2015,"Structure and mechanical properties of a light-weight AlNbTiV high entropy alloy"],[43,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.5,null,"C",600,810,1050,null,12,null,104,null,null,null,"10.1016/j.matlet.2014.11.162",2015,"Structure and mechanical properties of a light-weight AlNbTiV high entropy alloy"],[43,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.5,null,"C",800,685,null,null,50,null,104,null,null,null,"10.1016/j.matlet.2014.11.162",2015,"Structure and mechanical properties of a light-weight AlNbTiV high entropy alloy"],[43,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.5,null,"C",1000,158,null,null,50,null,104,null,null,null,"10.1016/j.matlet.2014.11.162",2015,"Structure and mechanical properties of a light-weight AlNbTiV high entropy alloy"],[44,"Cr1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+BCC+Sec.","OTHER","other",null,null,8,540,"C",25,1595,2046,5,null,null,null,null,null,null,"10.1016/j.msea.2011.09.033",2011,"Microstructure and properties of a refractory NbCrMo0.5Ta0.5TiZr alloy"],[44,"Cr1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+BCC+Sec.","OTHER","other",null,null,8,null,"C",800,983,1100,5.5,null,null,null,null,null,null,"10.1016/j.msea.2011.09.033",2011,"Microstructure and properties of a refractory NbCrMo0.5Ta0.5TiZr alloy"],[44,"Cr1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+BCC+Sec.","OTHER","other",null,null,8,null,"C",1000,546,630,50,null,null,null,null,null,null,"10.1016/j.msea.2011.09.033",2011,"Microstructure and properties of a refractory NbCrMo0.5Ta0.5TiZr alloy"],[44,"Cr1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+BCC+Sec.","OTHER","other",null,null,8,null,"C",1200,170,190,50,null,null,null,null,null,null,"10.1016/j.msea.2011.09.033",2011,"Microstructure and properties of a refractory NbCrMo0.5Ta0.5TiZr alloy"],[45,"Cr1 Nb1 Ti1 V1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,482,"C",25,1298,null,3,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Cr1 Nb1 Ti1 V1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,null,"C",600,1230,null,10,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Cr1 Nb1 Ti1 V1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,null,"C",800,615,null,50,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Cr1 Nb1 Ti1 V1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,null,"C",1000,259,null,50,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Cr1 Nb1 Ti1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,418,"C",25,1260,null,6,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Cr1 Nb1 Ti1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,null,"C",600,1035,null,50,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Cr1 Nb1 Ti1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,null,"C",800,300,null,50,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Cr1 Nb1 Ti1 Zr1","BCC+Sec.","OTHER","other",null,null,6.6,null,"C",1000,115,null,50,null,null,null,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb0.5 Ti0.5 V1 Zr0.5","BCC","OTHER","BCC",550,null,6.4,304,"C",25,918,null,50,null,null,105,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb0.5 Ti0.5 V1 Zr0.5","BCC","OTHER","BCC",null,null,6.4,null,"C",600,571,null,50,null,null,105,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb0.5 Ti0.5 V1 Zr0.5","BCC","OTHER","BCC",null,null,6.4,null,"C",800,240,null,50,null,null,105,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb0.5 Ti0.5 V1 Zr0.5","BCC","OTHER","BCC",null,null,6.4,null,"C",1000,72,null,50,null,null,105,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb1 Ti1 V1 Zr1","BCC","OTHER","BCC",550,null,6.5,335,"C",25,1105,null,50,null,null,100,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb1 Ti1 V1 Zr1","BCC","OTHER","BCC",null,null,6.5,null,"C",600,834,null,50,null,null,100,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb1 Ti1 V1 Zr1","BCC","OTHER","BCC",null,null,6.5,null,"C",800,187,null,50,null,null,100,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[45,"Nb1 Ti1 V1 Zr1","BCC","OTHER","BCC",null,null,6.5,null,"C",1000,58,null,50,null,null,100,null,null,null,"10.1016/j.msea.2012.12.018",2013,"Mechanical properties of low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system"],[47,"Hf0.5 Mo0.5 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,7.9,null,"C",25,1176,1538,null,null,null,115,null,null,null,"10.1016/j.intermet.2015.09.011",2016,"Microstructure and mechanical properties of in-situ MC-carbide particulates-reinforced refractory high-entropy Mo0.5NbHf0.5ZrTi matrix alloy composite"],[47,"C0.1 Hf0.5 Mo0.5 Nb1 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,7.8,null,"C",25,1183,2139,38,null,null,null,null,null,null,"10.1016/j.intermet.2015.09.011",2016,"Microstructure and mechanical properties of in-situ MC-carbide particulates-reinforced refractory high-entropy Mo0.5NbHf0.5ZrTi matrix alloy composite"],[47,"C0.3 Hf0.5 Mo0.5 Nb1 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,7.7,null,"C",25,1201,1965,33,null,null,null,null,null,null,"10.1016/j.intermet.2015.09.011",2016,"Microstructure and mechanical properties of in-situ MC-carbide particulates-reinforced refractory high-entropy Mo0.5NbHf0.5ZrTi matrix alloy composite"],[48,"Hf1 Mo0.25 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,395,"C",25,1112,null,50,null,96,115,null,null,null,"10.1016/j.matlet.2016.03.133",2016,"Solution strengthening of ductile refractory HfMoxNbTaTiZr high-entropy alloys"],[48,"Hf1 Mo0.5 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,480,"C",25,1317,null,50,null,102,123,null,null,null,"10.1016/j.matlet.2016.03.133",2016,"Solution strengthening of ductile refractory HfMoxNbTaTiZr high-entropy alloys"],[48,"Hf1 Mo0.75 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,492,"C",25,1373,null,50,null,109,130,null,null,null,"10.1016/j.matlet.2016.03.133",2016,"Solution strengthening of ductile refractory HfMoxNbTaTiZr high-entropy alloys"],[48,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,505,"C",25,1512,1800,12,null,115,137,null,null,null,"10.1016/j.matlet.2016.03.133",2016,"Solution strengthening of ductile refractory HfMoxNbTaTiZr high-entropy alloys"],[48,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,335,"C",25,1015,null,50,null,85,107,null,null,null,"10.1016/j.matlet.2016.03.133",2016,"Solution strengthening of ductile refractory HfMoxNbTaTiZr high-entropy alloys"],[49,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,505,"C",25,1512,null,12,null,null,137,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[49,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",800,1007,null,23,null,null,137,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[49,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",1000,814,null,30,null,null,137,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[49,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",1200,556,null,30,null,null,137,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[49,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,542,"C",25,1600,1743,4,null,null,143,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[49,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"C",800,1045,1446,19,null,null,143,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[49,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"C",1000,855,null,30,null,null,143,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[49,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"C",1200,404,null,30,null,null,143,null,null,null,"10.1016/j.intermet.2015.03.013",2015,"Enhanced mechanical properties of HfMoTaTiZr and HfMoNbTaTiZr refractory high-entropy alloys"],[50,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","ANNEAL","BCC",null,null,8.7,null,"C",25,1575,1640,9,null,null,128,null,null,null,"10.1016/j.matdes.2015.05.019",2015,"Microstructure and mechanical properties of refractory MoNbHfZrTi high-entropy alloy"],[50,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","ANNEAL","BCC",null,null,8.7,null,"C",800,825,1095,50,null,null,128,null,null,null,"10.1016/j.matdes.2015.05.019",2015,"Microstructure and mechanical properties of refractory MoNbHfZrTi high-entropy alloy"],[50,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","ANNEAL","BCC",null,null,8.7,null,"C",1000,635,654,50,null,null,128,null,null,null,"10.1016/j.matdes.2015.05.019",2015,"Microstructure and mechanical properties of refractory MoNbHfZrTi high-entropy alloy"],[50,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","ANNEAL","BCC",null,null,8.7,null,"C",1200,187,194,50,null,null,128,null,null,null,"10.1016/j.matdes.2015.05.019",2015,"Microstructure and mechanical properties of refractory MoNbHfZrTi high-entropy alloy"],[50,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.7,null,"C",25,1719,1803,10,null,null,128,null,null,null,"10.1016/j.matdes.2015.05.019",2015,"Microstructure and mechanical properties of refractory MoNbHfZrTi high-entropy alloy"],[51,"Hf1 Nb1 Si0.5 Ti1 V1","BCC+Sec.","CAST","other",null,null,7.8,null,"C",800,875,null,50,null,null,null,null,null,null,"10.1016/j.matlet.2016.03.092",2016,"Microstructure and mechanical properties of a refractory HfNbTiVSi0.5 high-entropy alloy composite"],[51,"Hf1 Nb1 Si0.5 Ti1 V1","BCC+Sec.","CAST","other",null,null,7.8,null,"C",1000,240,null,50,null,null,null,null,null,null,"10.1016/j.matlet.2016.03.092",2016,"Microstructure and mechanical properties of a refractory HfNbTiVSi0.5 high-entropy alloy composite"],[51,"Hf1 Nb1 Si0.5 Ti1 V1","BCC+Sec.","CAST","other",null,null,7.8,490,"C",25,1399,1608,11,null,null,null,null,null,null,"10.1016/j.matlet.2016.03.092",2016,"Microstructure and mechanical properties of a refractory HfNbTiVSi0.5 high-entropy alloy composite"],[52,"Hf0.333 Nb0.333 Ta0.333 Ti1 Zr0.667","BCC","CAST","BCC",null,null,8.2,301,"T",25,903,990,19,null,null,103,null,null,null,"10.1063/1.4966659",2016,"Alloy design for intrinsically ductile refractory high-entropy alloys"],[53,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"C",25,929,null,50,null,null,107,null,null,null,"10.1007/s10853-012-6260-2",2012,"Microstructure and elevated temperature properties of a refractory TaNbHfZrTi alloy"],[53,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"C",400,790,null,50,null,null,107,null,null,null,"10.1007/s10853-012-6260-2",2012,"Microstructure and elevated temperature properties of a refractory TaNbHfZrTi alloy"],[53,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"C",600,675,null,50,null,null,107,null,null,null,"10.1007/s10853-012-6260-2",2012,"Microstructure and elevated temperature properties of a refractory TaNbHfZrTi alloy"],[53,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"C",800,535,null,50,null,null,107,null,null,null,"10.1007/s10853-012-6260-2",2012,"Microstructure and elevated temperature properties of a refractory TaNbHfZrTi alloy"],[53,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"C",1000,295,null,50,null,null,107,null,null,null,"10.1007/s10853-012-6260-2",2012,"Microstructure and elevated temperature properties of a refractory TaNbHfZrTi alloy"],[53,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"C",1200,92,null,50,null,null,107,null,null,null,"10.1007/s10853-012-6260-2",2012,"Microstructure and elevated temperature properties of a refractory TaNbHfZrTi alloy"],[55,"Hf1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.4,null,"T",25,879,969,null,14.9,null,89,null,null,null,"10.1016/j.matlet.2014.05.134",2014,"A refractory Hf25Nb25Ti25Zr25 high-entropy alloy with excellent structural stability and tensile properties"],[56,"Mo1 Nb1 Ta1 V1","BCC","ANNEAL","BCC",200,null,10.7,504,"C",25,1525,2400,21,null,null,186,null,null,null,"10.3390/e18050189",2016,"MoNbTaV Medium-Entropy Alloy"],[57,"Mo1 Nb1 Ta1 V1 W1","BCC","CAST","BCC",80,null,12.4,536,"C",25,1246,1270,2,null,null,229,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 V1 W1","BCC","CAST","BCC",null,null,12.4,null,"C",600,862,null,13,null,null,229,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 V1 W1","BCC","CAST","BCC",null,null,12.4,null,"C",800,846,null,17,null,null,229,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 V1 W1","BCC","CAST","BCC",null,null,12.4,null,"C",1000,842,null,19,null,null,229,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 V1 W1","BCC","CAST","BCC",null,null,12.4,null,"C",1200,735,null,7.5,null,null,229,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 V1 W1","BCC","CAST","BCC",null,null,12.4,null,"C",1400,656,null,40,null,null,229,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 V1 W1","BCC","CAST","BCC",null,null,12.4,null,"C",1600,477,null,40,null,null,229,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",200,null,13.7,454,"C",25,1058,1211,2,null,null,250,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",null,null,13.7,null,"C",600,561,null,40,null,null,250,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",null,null,13.7,null,"C",800,552,null,40,null,null,250,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",null,null,13.7,null,"C",1000,548,null,40,null,null,250,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",null,null,13.7,null,"C",1200,506,null,40,null,null,250,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",null,null,13.7,null,"C",1400,421,null,40,null,null,250,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[57,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",null,null,13.7,null,"C",1600,405,null,40,null,null,250,null,null,null,"10.1016/j.intermet.2011.01.004",2011,"Mechanical properties of Nb25Mo25Ta25W25 and V20Nb20Mo20Ta20W20 refractory high entropy alloys"],[58,"Mo0.3 Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,6.7,null,"C",25,1289,null,null,42,null,114,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo0.5 Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,6.8,null,"C",25,1473,null,null,32,null,122,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo0.7 Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,6.9,null,"C",25,1706,null,null,32,null,130,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb0.769 Ti0.769 V0.769 Zr0.769","BCC","ANNEAL","BCC",null,null,7.3,null,"C",25,1496,null,null,30,null,150,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb0.667 Ti0.667 V0.667 Zr0.667","BCC","ANNEAL","BCC",null,null,7.4,null,"C",25,1603,null,null,20,null,156,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb0.588 Ti0.588 V0.588 Zr0.588","BCC","ANNEAL","BCC",null,null,7.5,null,"C",25,1645,null,null,15,null,161,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb0.5 Ti0.5 V0.5 Zr0.5","BCC","ANNEAL","BCC",null,null,7.6,null,"C",25,1765,null,null,12,null,169,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,7.1,null,"C",25,1779,null,null,32,null,141,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb1 Ti1 V0.3 Zr1","BCC","ANNEAL","BCC",null,null,7.2,null,"C",25,1455,null,null,25,null,142,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo0.1 Nb1 Ti1 V0.3 Zr1","BCC","ANNEAL","BCC",null,null,6.6,null,"C",25,932,null,null,45,null,102,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo0.3 Nb1 Ti1 V0.3 Zr1","BCC","ANNEAL","BCC",null,null,6.8,null,"C",25,1312,null,null,50,null,112,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo0.5 Nb1 Ti1 V0.3 Zr1","BCC","ANNEAL","BCC",null,null,6.9,null,"C",25,1301,null,null,43,null,122,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo0.7 Nb1 Ti1 V0.3 Zr1","BCC","ANNEAL","BCC",null,null,7.1,null,"C",25,1436,null,null,27,null,130,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb0.769 Ti0.769 V0.231 Zr0.769","BCC","ANNEAL","BCC",null,null,7.4,null,"C",25,1603,null,null,20,null,153,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Mo1 Nb0.667 Ti0.667 V0.2 Zr0.667","BCC","ANNEAL","BCC",null,null,7.5,null,"C",25,1576,null,null,8,null,159,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Nb1 Ti1 V0.3 Zr1","BCC","ANNEAL","BCC",null,null,6.5,null,"C",25,866,null,null,45,null,96,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[58,"Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,6.5,null,"C",25,1104,null,null,50,null,100,null,null,null,"10.1016/j.matdes.2015.06.072",2015,"Phase composition and solid solution strengthening effect in TiZrNbMoV high-entropy alloys"],[59,"Mo1 Nb1 Ti1 V0.25 Zr1","BCC","CAST","BCC",null,null,7.3,null,"C",25,1776,3893,30,null,null,142,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[59,"Mo1 Nb1 Ti1 V0.5 Zr1","BCC","CAST","BCC",null,null,7.2,null,"C",25,1647,3307,28,null,null,142,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[59,"Mo1 Nb1 Ti1 V0.75 Zr1","BCC","CAST","BCC",null,null,7.2,null,"C",25,1708,3929,29,null,null,141,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[59,"Mo1 Nb1 Ti1 V1 Zr1","BCC","CAST","BCC",null,null,7.1,null,"C",25,1786,3828,26,null,null,141,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[59,"Mo0.667 Nb0.667 Ti0.667 V1 Zr0.667","BCC","CAST","BCC",null,null,7.1,null,"C",25,1735,3300,20,null,null,140,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[59,"Mo0.5 Nb0.5 Ti0.5 V1 Zr0.5","BCC","CAST","BCC",null,null,7,null,"C",25,1538,3176,23,null,null,139,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[59,"Mo0.333 Nb0.333 Ti0.333 V1 Zr0.333","BCC","CAST","BCC",null,null,6.9,null,"C",25,1418,2508,24,null,null,138,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[59,"Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,7.3,null,"C",25,1592,3450,34,null,null,143,null,null,null,"10.1007/s11837-012-0366-5",2012,"Alloy Design and Properties Optimization of High-Entropy Alloys"],[60,"Co1 Fe1 Mo0.2 Ni1 V1","FCC+Sec.","CAST","other",null,null,8,267,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co1 Fe1 Mo0.4 Ni1 V1","FCC+Sec.","CAST","other",null,null,8.1,402,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co1 Fe1 Mo0.6 Ni1 V1","FCC+Sec.","CAST","other",null,null,8.2,557,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co1 Fe1 Mo0.8 Ni1 V1","FCC+Sec.","CAST","other",null,null,8.3,606,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co0.833 Fe0.833 Mo0.833 Ni1 V0.833","FCC+Sec.","CAST","other",null,null,8.4,602,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co0.714 Fe0.714 Mo0.714 Ni1 V0.714","FCC+Sec.","CAST","other",null,null,8.5,538,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co0.625 Fe0.625 Mo0.625 Ni1 V0.625","FCC+Sec.","CAST","other",null,null,8.5,520,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co0.556 Fe0.556 Mo0.556 Ni1 V0.556","FCC+Sec.","CAST","other",null,null,8.5,510,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co0.5 Fe0.5 Mo0.5 Ni1 V0.5","FCC+Sec.","CAST","other",null,null,8.5,382,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co1 Fe1 Mo1 Ni1 V1","FCC+Sec.","CAST","other",null,null,8.4,625,null,25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[60,"Co1 Fe1 Ni1 V1","FCC","CAST","FCC",null,null,7.8,238,null,25,null,null,null,null,null,184,null,null,null,"10.1016/j.jallcom.2015.07.185",2015,"Effect of Mo and Ni elements on microstructure evolution and mechanical properties of the CoFeNixVMoy high entropy alloys"],[61,"Al0.125 Co1 Cr1 Cu1 Fe1 Mn1 Ni1 Ti1 V1","FCC+BCC","CAST","other",null,null,7.2,null,"C",25,1862,2431,1,null,164,null,null,null,null,"10.1016/j.msea.2006.11.049",2007,"Microstructure and compressive properties of multicomponent Alx(TiVCrMnFeCoNiCu)100−x high-entropy alloys"],[61,"Al0.25 Co1 Cr1 Cu1 Fe1 Mn1 Ni1 Ti1 V1","BCC","CAST","BCC",null,null,7.1,null,"C",25,1465,2016,2,null,190,174,null,null,null,"10.1016/j.msea.2006.11.049",2007,"Microstructure and compressive properties of multicomponent Alx(TiVCrMnFeCoNiCu)100−x high-entropy alloys"],[61,"Al0.667 Co1 Cr1 Cu1 Fe1 Mn1 Ni1 Ti1 V1","BCC+Sec.","CAST","other",null,null,6.8,null,"C",25,1461,1461,1,null,163,null,null,null,null,"10.1016/j.msea.2006.11.049",2007,"Microstructure and compressive properties of multicomponent Alx(TiVCrMnFeCoNiCu)100−x high-entropy alloys"],[61,"Co1 Cr1 Cu1 Fe1 Mn1 Ni1 Ti1 V1","FCC+BCC+Sec.","CAST","other",null,null,7.3,null,"C",25,1312,1312,1,null,74,null,null,null,null,"10.1016/j.msea.2006.11.049",2007,"Microstructure and compressive properties of multicomponent Alx(TiVCrMnFeCoNiCu)100−x high-entropy alloys"],[62,"Cu1 Mn1 Ni1 Zn0.333","FCC","CAST","FCC",null,null,8.3,147,"C",25,215,null,60,null,null,167,null,null,null,"10.1016/j.jallcom.2015.07.285",2015,"High entropy brasses and bronzes – Microstructure, phase evolution and properties"],[62,"Cu1 Mn1 Ni1 Zn0.25","FCC","CAST","FCC",null,null,8.3,109,"C",25,140,null,65,null,null,169,null,null,null,"10.1016/j.jallcom.2015.07.285",2015,"High entropy brasses and bronzes – Microstructure, phase evolution and properties"],[62,"Al0.111 Cu1 Mn1 Ni1","FCC+Sec.","CAST","other",null,null,8.1,241,"C",25,515,null,40,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.285",2015,"High entropy brasses and bronzes – Microstructure, phase evolution and properties"],[62,"Cu1 Mn1 Ni1 Sn0.111","FCC+Sec.","CAST","other",null,null,8.3,318,"C",25,630,null,20,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.285",2015,"High entropy brasses and bronzes – Microstructure, phase evolution and properties"],[62,"Al0.053 Cu1 Mn1 Ni1","FCC","CAST","FCC",null,null,8.3,166,"C",25,330,null,45,null,null,173,null,null,null,"10.1016/j.jallcom.2015.07.285",2015,"High entropy brasses and bronzes – Microstructure, phase evolution and properties"],[62,"Cu1 Mn1 Ni1 Sn0.053","FCC+Sec.","CAST","other",null,null,8.4,205,"C",25,380,null,63,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.285",2015,"High entropy brasses and bronzes – Microstructure, phase evolution and properties"],[64,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",1000,150,null,null,null,null,189,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",25,388,null,null,null,null,189,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",300,411,null,null,null,null,189,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",500,421,null,null,null,null,189,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",700,426,null,null,null,null,189,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",900,230,null,null,null,null,189,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",1100,80,null,null,null,null,189,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",1000,116,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",1100,79,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",25,1620,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",600,805,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",500,1120,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",700,567,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",900,214,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","BCC","CAST","BCC",null,null,6.3,null,"C",800,302,null,null,null,null,156,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,null,"C",1000,47,null,null,null,null,null,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,null,"C",25,948,null,null,null,null,null,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,null,"C",600,561,null,null,null,null,null,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,null,"C",700,307,null,null,null,null,null,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,null,"C",800,172,null,null,null,null,null,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[64,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,null,"C",900,98,null,null,null,null,null,null,null,null,"10.1007/s11661-005-0218-9",2005,"Mechanical performance of the Al x CoCrCuFeNi high-entropy alloy system with multiprincipal elements"],[65,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","CAST","BCC",null,null,6.6,null,"C",25,null,null,null,null,null,170,92,null,null,"10.1016/j.jallcom.2015.11.050",2016,"Microstructure and mechanical properties at elevated temperatures of a new Al-containing refractory high-entropy alloy Nb-Mo-Cr-Ti-Al"],[65,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","CAST","BCC",null,null,6.6,null,"C",400,1080,1100,2,null,null,170,92,null,null,"10.1016/j.jallcom.2015.11.050",2016,"Microstructure and mechanical properties at elevated temperatures of a new Al-containing refractory high-entropy alloy Nb-Mo-Cr-Ti-Al"],[65,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","CAST","BCC",null,null,6.6,null,"C",600,1060,1170,3,null,null,170,92,null,null,"10.1016/j.jallcom.2015.11.050",2016,"Microstructure and mechanical properties at elevated temperatures of a new Al-containing refractory high-entropy alloy Nb-Mo-Cr-Ti-Al"],[65,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","CAST","BCC",null,null,6.6,null,"C",800,860,1000,2,null,null,170,92,null,null,"10.1016/j.jallcom.2015.11.050",2016,"Microstructure and mechanical properties at elevated temperatures of a new Al-containing refractory high-entropy alloy Nb-Mo-Cr-Ti-Al"],[65,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","CAST","BCC",null,null,6.6,null,"C",1000,594,630,15,null,null,170,92,null,null,"10.1016/j.jallcom.2015.11.050",2016,"Microstructure and mechanical properties at elevated temperatures of a new Al-containing refractory high-entropy alloy Nb-Mo-Cr-Ti-Al"],[65,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","CAST","BCC",null,null,6.6,null,"C",1200,105,116,24,null,null,170,92,null,null,"10.1016/j.jallcom.2015.11.050",2016,"Microstructure and mechanical properties at elevated temperatures of a new Al-containing refractory high-entropy alloy Nb-Mo-Cr-Ti-Al"],[67,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.6,null,"C",600,930,null,null,null,null,170,92,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.6,null,"C",800,875,null,null,null,null,170,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.6,null,"C",1000,550,null,null,null,null,170,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.6,null,"C",1200,150,null,null,null,null,170,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,null,6,null,"C",400,1070,null,null,null,null,189,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,null,6,null,"C",600,1020,null,null,null,null,189,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,null,6,null,"C",800,875,null,null,null,null,189,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,null,6,null,"C",1000,375,null,null,null,null,189,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,null,6,null,"C",1200,100,null,null,null,null,189,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.5,null,"C",600,520,null,null,null,null,151,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.5,null,"C",800,500,null,null,null,null,151,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.5,null,"C",1000,540,null,null,null,null,151,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[67,"Al1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,6.5,null,"C",1200,200,null,null,null,null,151,null,null,null,"10.1007/s11661-017-4386-1",2017,"Contribution of Lattice Distortion to Solid Solution Strengthening in a Series of Refractory High Entropy Alloys"],[68,"Al1 Nb1 Ti1 V1","B2","ANNEAL","other",null,null,5.5,null,"C",600,780,1005,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1","B2","ANNEAL","other",null,null,5.5,null,"C",800,560,700,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1","B2","ANNEAL","other",null,null,5.5,null,"C",25,1000,1280,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr1","B2+Laves+Sec.","ANNEAL","other",null,null,5.8,null,"C",600,1155,1385,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr1","B2+Laves+Sec.","ANNEAL","other",null,null,5.8,null,"C",800,550,600,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr1","B2+Laves+Sec.","ANNEAL","other",null,null,5.8,null,"C",25,1500,1675,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.1","B2+Sec.","ANNEAL","other",null,null,5.5,null,"C",600,975,1085,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.1","B2+Sec.","ANNEAL","other",null,null,5.5,null,"C",800,865,920,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.1","B2+Sec.","ANNEAL","other",null,null,5.5,null,"C",25,1290,1395,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.25","B2+Sec.","ANNEAL","other",null,null,5.6,null,"C",600,1065,1260,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.25","B2+Sec.","ANNEAL","other",null,null,5.6,null,"C",800,855,865,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.25","B2+Sec.","ANNEAL","other",null,null,5.6,null,"C",25,1360,1480,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.5","B2+Laves+Sec.","ANNEAL","other",null,null,5.6,null,"C",600,1135,1425,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.5","B2+Laves+Sec.","ANNEAL","other",null,null,5.6,null,"C",800,675,740,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al1 Nb1 Ti1 V1 Zr0.5","B2+Laves+Sec.","ANNEAL","other",null,null,5.6,null,"C",25,1485,null,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al0.667 Nb0.667 Ti0.667 V0.667 Zr1","B2+Laves+Sec.","ANNEAL","other",null,null,5.8,null,"C",600,1195,1195,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al0.667 Nb0.667 Ti0.667 V0.667 Zr1","B2+Laves+Sec.","ANNEAL","other",null,null,5.8,null,"C",800,180,395,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[68,"Al0.667 Nb0.667 Ti0.667 V0.667 Zr1","B2+Laves+Sec.","ANNEAL","other",null,null,5.8,null,"C",25,1535,1550,null,null,null,null,null,null,null,"10.1016/j.msea.2017.08.019",2017,"Structure and mechanical properties of B2 ordered refractory AlNbTiVZr x (x = 0–1.5) high-entropy alloys"],[69,"Co1 Cr1 Mo1 Nb1","BCC+Laves","CAST","other",null,null,8.8,null,"C",25,null,1419.6,null,null,null,null,null,null,null,"10.1007/s11665-017-2799-z",2017,"Microstructure and Mechanical Properties of a Refractory CoCrMoNbTi High-Entropy Alloy"],[69,"Co1 Cr1 Mo1 Nb1 Ti1","BCC+Laves","CAST","other",null,null,7.8,null,"C",25,null,1096.8,null,null,null,null,null,null,null,"10.1007/s11665-017-2799-z",2017,"Microstructure and Mechanical Properties of a Refractory CoCrMoNbTi High-Entropy Alloy"],[69,"Co1 Cr1 Mo1 Nb1 Ti0.2","BCC+Laves","CAST","other",null,null,8.5,null,"C",25,null,1905.6,null,null,null,null,null,null,null,"10.1007/s11665-017-2799-z",2017,"Microstructure and Mechanical Properties of a Refractory CoCrMoNbTi High-Entropy Alloy"],[69,"Co1 Cr1 Mo1 Nb1 Ti0.4","BCC","CAST","BCC",null,null,8.3,null,"C",25,null,1771.3,null,null,null,212,null,null,null,"10.1007/s11665-017-2799-z",2017,"Microstructure and Mechanical Properties of a Refractory CoCrMoNbTi High-Entropy Alloy"],[69,"Co1 Cr1 Mo1 Nb1 Ti0.5","BCC+Laves","CAST","other",null,null,8.2,null,"C",25,null,1609.2,null,null,null,null,null,null,null,"10.1007/s11665-017-2799-z",2017,"Microstructure and Mechanical Properties of a Refractory CoCrMoNbTi High-Entropy Alloy"],[70,"Cr1 Hf1 Nb1 Ti1 Zr1","BCC+Laves","ANNEAL","other",null,null,8.2,null,"C",25,1457,1908,null,null,null,null,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Cr1 Hf1 Nb1 Ti1 Zr1","BCC+Laves","ANNEAL","other",null,null,8.2,null,"C",25,1420,1885,4.2,null,null,null,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Cr1 Hf1 Nb1 Ti1 Zr1","BCC+Laves","ANNEAL","other",null,null,8.2,null,"C",25,1328,1884,null,null,null,null,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Cr1 Hf1 Nb1 Ti1 Zr1","BCC+Laves","ANNEAL","other",null,null,8.2,null,"C",25,1322,1726,null,null,null,null,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Cr1 Hf1 Nb1 Ti1 Zr1","BCC+Laves","CAST","other",null,null,8.2,null,"C",25,1375,2130,3,null,null,null,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Hf1 Nb1 Ti1 V1 Zr1","BCC+Laves","ANNEAL","other",null,null,8.1,null,"C",25,1157,1468,null,null,null,null,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Hf1 Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,8.1,null,"C",25,1253,1393,null,null,null,95,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Hf1 Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,8.1,null,"C",25,1140,1463,null,null,null,95,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Hf1 Nb1 Ti1 V1 Zr1","BCC","ANNEAL","BCC",null,null,8.1,null,"C",25,1120,1376,30,null,null,95,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[70,"Hf1 Nb1 Ti1 V1 Zr1","BCC+Laves","CAST","other",null,null,8.1,null,"C",25,1170,1463,29.6,null,null,null,null,null,null,"10.1016/j.ijrmhm.2014.07.009",2014,"Experimental and theoretical study of Ti20Zr20Hf20Nb20X20 (X=V or Cr) refractory high-entropy alloys"],[71,"Cr1 Ta1 Ti0.17 V1 W1","BCC+Laves","POWDER","other",null,null,12.6,715,"C",25,2034,null,1.5,null,null,null,null,null,null,"10.1016/j.matchemphys.2017.06.054",2018,"The effect of Ti on the sintering and mechanical properties of refractory high-entropy alloy TixWTaVCr fabricated via spark plasma sintering for fusion plasma-facing materials"],[71,"Cr1 Ta1 Ti0.17 V1 W1","BCC+Laves","POWDER","other",null,null,12.6,null,"C",1200,750,null,1.5,null,null,null,null,null,null,"10.1016/j.matchemphys.2017.06.054",2018,"The effect of Ti on the sintering and mechanical properties of refractory high-entropy alloy TixWTaVCr fabricated via spark plasma sintering for fusion plasma-facing materials"],[71,"Cr1 Ta1 Ti0.3 V1 W1","BCC+Laves","POWDER","other",null,null,12.3,780,"C",25,2050,null,1.5,null,null,null,null,null,null,"10.1016/j.matchemphys.2017.06.054",2018,"The effect of Ti on the sintering and mechanical properties of refractory high-entropy alloy TixWTaVCr fabricated via spark plasma sintering for fusion plasma-facing materials"],[71,"Cr1 Ta1 Ti0.3 V1 W1","BCC+Laves","POWDER","other",null,null,12.3,null,"C",1200,586,null,1.5,null,null,null,null,null,null,"10.1016/j.matchemphys.2017.06.054",2018,"The effect of Ti on the sintering and mechanical properties of refractory high-entropy alloy TixWTaVCr fabricated via spark plasma sintering for fusion plasma-facing materials"],[71,"Cr1 Ta1 V1 W1","BCC+Laves","POWDER","other",null,null,13,715,"C",25,2327,null,1.5,null,null,null,null,null,null,"10.1016/j.matchemphys.2017.06.054",2018,"The effect of Ti on the sintering and mechanical properties of refractory high-entropy alloy TixWTaVCr fabricated via spark plasma sintering for fusion plasma-facing materials"],[71,"Cr1 Ta1 V1 W1","BCC+Laves","POWDER","other",null,null,13,null,"C",1200,979,null,1.5,null,null,null,null,null,null,"10.1016/j.matchemphys.2017.06.054",2018,"The effect of Ti on the sintering and mechanical properties of refractory high-entropy alloy TixWTaVCr fabricated via spark plasma sintering for fusion plasma-facing materials"],[72,"Hf0.26 Nb1 Ta1 Ti0.578 Zr0.416","BCC","CAST","BCC",null,null,10.4,null,"C",60,795,null,null,null,null,122,null,null,null,"10.1080/14786435.2015.1028506",2015,"Plastic deformation properties of Zr–Nb–Ti–Ta–Hf high-entropy alloys"],[72,"Hf0.26 Nb1 Ta1 Ti0.578 Zr0.416","BCC","CAST","BCC",null,null,10.4,null,"C",100,765,null,null,null,null,122,null,null,null,"10.1080/14786435.2015.1028506",2015,"Plastic deformation properties of Zr–Nb–Ti–Ta–Hf high-entropy alloys"],[72,"Hf0.26 Nb1 Ta1 Ti0.578 Zr0.416","BCC","CAST","BCC",null,null,10.4,null,"C",200,650,null,null,null,null,122,null,null,null,"10.1080/14786435.2015.1028506",2015,"Plastic deformation properties of Zr–Nb–Ti–Ta–Hf high-entropy alloys"],[72,"Hf0.26 Nb1 Ta1 Ti0.578 Zr0.416","BCC","CAST","BCC",null,null,10.4,null,"C",300,590,null,null,null,null,122,null,null,null,"10.1080/14786435.2015.1028506",2015,"Plastic deformation properties of Zr–Nb–Ti–Ta–Hf high-entropy alloys"],[72,"Hf0.26 Nb1 Ta1 Ti0.578 Zr0.416","BCC","CAST","BCC",null,null,10.4,null,"C",25,845,null,null,null,null,122,null,null,null,"10.1080/14786435.2015.1028506",2015,"Plastic deformation properties of Zr–Nb–Ti–Ta–Hf high-entropy alloys"],[73,"Hf0.5 Mo0.5 Nb1 Si0.1 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,7.7,442,"C",25,1350,2150,null,28,null,null,null,null,null,"10.1016/j.jallcom.2015.11.091",2016,"Microstructure and mechanical properties of refractory high entropy (Mo 0.5 NbHf 0.5 ZrTi) BCC /M 5 Si 3 in-situ compound"],[73,"Hf0.5 Mo0.5 Nb1 Si0.3 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,7.5,494,"C",25,1370,1950,null,23,null,null,null,null,null,"10.1016/j.jallcom.2015.11.091",2016,"Microstructure and mechanical properties of refractory high entropy (Mo 0.5 NbHf 0.5 ZrTi) BCC /M 5 Si 3 in-situ compound"],[73,"Hf0.5 Mo0.5 Nb1 Si0.5 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,7.2,524,"C",25,1600,1900,null,23,null,null,null,null,null,"10.1016/j.jallcom.2015.11.091",2016,"Microstructure and mechanical properties of refractory high entropy (Mo 0.5 NbHf 0.5 ZrTi) BCC /M 5 Si 3 in-situ compound"],[73,"Hf0.5 Mo0.5 Nb1 Si0.7 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,7,580,"C",25,1550,1900,null,12,null,null,null,null,null,"10.1016/j.jallcom.2015.11.091",2016,"Microstructure and mechanical properties of refractory high entropy (Mo 0.5 NbHf 0.5 ZrTi) BCC /M 5 Si 3 in-situ compound"],[73,"Hf0.5 Mo0.5 Nb1 Si0.9 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,6.8,640,"C",25,1650,2000,null,9,null,null,null,null,null,"10.1016/j.jallcom.2015.11.091",2016,"Microstructure and mechanical properties of refractory high entropy (Mo 0.5 NbHf 0.5 ZrTi) BCC /M 5 Si 3 in-situ compound"],[73,"Hf0.5 Mo0.5 Nb1 Ti1 Zr1","BCC+Sec.","CAST","other",null,null,7.9,400,"C",25,1150,1500,null,25,null,null,null,null,null,"10.1016/j.jallcom.2015.11.091",2016,"Microstructure and mechanical properties of refractory high entropy (Mo 0.5 NbHf 0.5 ZrTi) BCC /M 5 Si 3 in-situ compound"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-268.8,2250,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-268.8,2210,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-196,1880,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-196,1750,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-153,1640,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-153,1550,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-103,1380,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-103,1370,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-43,1200,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-43,1180,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",72,1040,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",72,1020,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",25,1150,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",25,1100,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-268.8,2390,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-196,1920,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-153,1370,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-103,1250,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",-43,1020,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",72,640,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[74,"Hf0.5 Nb0.667 Ta0.333 Ti1 Zr0.833","BCC","WROUGHT","BCC",null,null,8.4,null,"C",25,890,null,null,null,null,100,null,null,null,"10.1016/j.msea.2017.10.073",2018,"Mechanical properties and thermally activated plasticity of the Ti30Zr25Hf15Nb20Ta10 high entropy alloy at temperatures 4.2–350 K"],[75,"Hf1 Mo0.5 Nb1 Si0.3 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,8.5,null,"C",1000,398,430,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.3 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,8.5,null,"C",25,1617,2016,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.3 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,8.5,null,"C",1200,166,187,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.5 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,8.2,null,"C",1000,614,696,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.5 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,8.2,null,"C",25,1787,2052,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.5 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,8.2,null,"C",1200,188,219,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.7 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,7.9,null,"C",1000,673,741,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.7 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,7.9,null,"C",25,2134,2242,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Si0.7 Ti1 V0.5","BCC+Sec.","CAST","other",null,null,7.9,null,"C",1200,235,268,null,null,null,null,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Ti1 V0.5","BCC","CAST","BCC",null,null,9,null,"C",1000,368,394,null,null,null,126,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Ti1 V0.5","BCC","CAST","BCC",null,null,9,null,"C",1200,60,null,null,null,null,126,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[75,"Hf1 Mo0.5 Nb1 Ti1 V0.5","BCC","CAST","BCC",null,null,9,null,"C",25,1260,null,null,null,null,126,null,null,null,"10.1016/j.jallcom.2016.10.014",2017,"Microstructure and mechanical properties of refractory HfMo0.5NbTiV0.5Six high-entropy composites"],[76,"Hf0.787 Nb0.142 Ta0.142 Ti1 Zr0.787","BCC","WROUGHT","BCC",null,null,8.5,null,"T",25,540,995,null,null,null,92,null,null,null,"10.1080/21663831.2016.1221861",2016,"Design and tensile properties of a bcc Ti-rich high-entropy alloy with transformation-induced plasticity"],[77,"Al0.25 Nb1 Ta1 Ti1 Zr1","BCC+B2","OTHER","other",null,null,8.6,null,"C",1000,366,408,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al0.25 Nb1 Ta1 Ti1 Zr1","BCC+B2","OTHER","other",null,null,8.6,null,"C",25,1745,1830,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al0.5 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+B2","OTHER","other",null,null,7.6,null,"C",1000,579,646,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al0.5 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+B2","OTHER","other",null,null,7.6,null,"C",25,2350,2460,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+B2","OTHER","other",null,null,7.1,null,"C",25,2197,2370,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr0.5","B2","OTHER","other",null,null,7.2,null,"C",1000,935,1019,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr0.5","B2","OTHER","other",null,null,7.2,null,"C",25,null,1320,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al1 Nb1 Ta0.5 Ti1 Zr0.5","B2","OTHER","other",null,null,6.9,null,"C",1000,535,542,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[77,"Al1 Nb1 Ta0.5 Ti1 Zr0.5","B2","OTHER","other",null,null,6.9,null,"C",25,1352,1357,null,null,null,null,null,null,null,"10.1016/j.matdes.2017.11.033",2018,"Compositional variation effects on the microstructure and properties of a refractory high-entropy superalloy AlMo0.5NbTa0.5TiZr"],[78,"Hf1 Nb1 Si0.5 Ti1 V1 Zr1","BCC+Laves+Sec.","ANNEAL","other",null,null,7.5,null,"C",400,1273,1335,14.5,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.849.76",2016,"Microstructure and Mechanical Properties of a New Refractory HfNbSisub0.5/subTiVZr High Entropy Alloy"],[78,"Hf1 Nb1 Si0.5 Ti1 V1 Zr1","BCC+Laves+Sec.","ANNEAL","other",null,null,7.5,null,"C",600,597,651,50,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.849.76",2016,"Microstructure and Mechanical Properties of a New Refractory HfNbSisub0.5/subTiVZr High Entropy Alloy"],[78,"Hf1 Nb1 Si0.5 Ti1 V1 Zr1","BCC+Laves+Sec.","ANNEAL","other",null,null,7.5,null,"C",800,102,107,50,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.849.76",2016,"Microstructure and Mechanical Properties of a New Refractory HfNbSisub0.5/subTiVZr High Entropy Alloy"],[78,"Hf1 Nb1 Si0.5 Ti1 V1 Zr1","BCC+Laves+Sec.","ANNEAL","other",null,null,7.5,null,"C",25,1483,1611,9.5,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.849.76",2016,"Microstructure and Mechanical Properties of a New Refractory HfNbSisub0.5/subTiVZr High Entropy Alloy"],[78,"Hf1 Nb1 Si0.5 Ti1 V1 Zr1","BCC+Laves+Sec.","CAST","other",null,null,7.5,null,"C",600,920,1252,50,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.849.76",2016,"Microstructure and Mechanical Properties of a New Refractory HfNbSisub0.5/subTiVZr High Entropy Alloy"],[78,"Hf1 Nb1 Si0.5 Ti1 V1 Zr1","BCC+Laves+Sec.","CAST","other",null,null,7.5,null,"C",800,371,427,50,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.849.76",2016,"Microstructure and Mechanical Properties of a New Refractory HfNbSisub0.5/subTiVZr High Entropy Alloy"],[78,"Hf1 Nb1 Si0.5 Ti1 V1 Zr1","BCC+Laves+Sec.","CAST","other",null,null,7.5,null,"C",25,1540,1643,16.8,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.849.76",2016,"Microstructure and Mechanical Properties of a New Refractory HfNbSisub0.5/subTiVZr High Entropy Alloy"],[79,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",0.05,null,9.9,420,"T",25,1900,null,7.9,null,null,107,null,null,null,"10.1016/j.actamat.2017.09.035",2018,null],[79,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC+HCP","OTHER","other",0.05,null,9.9,530,"T",25,1520,null,1.7,null,null,null,null,null,null,"10.1016/j.actamat.2017.09.035",2018,null],[79,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC+BCC+HCP","OTHER","other",0.1,null,9.9,340,"T",25,795,null,5,null,null,null,null,null,null,"10.1016/j.actamat.2017.09.035",2018,null],[79,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",100,null,9.9,309,"T",25,830,840,9.2,null,null,107,null,null,null,"10.1016/j.actamat.2017.09.035",2018,null],[80,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,366,"T",25,1202,1295,4.7,null,null,107,null,null,null,"10.1016/j.jallcom.2015.07.209",2015,"Microstructure and properties of a refractory high-entropy alloy after cold working"],[80,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC+BCC","WROUGHT","other",null,null,9.9,null,"T",25,1303,1334,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.07.209",2015,"Microstructure and properties of a refractory high-entropy alloy after cold working"],[80,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,351,"T",25,1145,1262,9.7,null,null,107,null,null,null,"10.1016/j.jallcom.2015.07.209",2015,"Microstructure and properties of a refractory high-entropy alloy after cold working"],[81,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,"T",25,958,974,null,null,null,107,null,null,null,"10.1016/j.matlet.2016.08.060",2016,"Simultaneously increasing the strength and ductility of a refractory high-entropy alloy via grain refining"],[81,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,"T",25,944,959,null,null,null,107,null,null,null,"10.1016/j.matlet.2016.08.060",2016,"Simultaneously increasing the strength and ductility of a refractory high-entropy alloy via grain refining"],[81,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,"T",25,940,953,null,null,null,107,null,null,null,"10.1016/j.matlet.2016.08.060",2016,"Simultaneously increasing the strength and ductility of a refractory high-entropy alloy via grain refining"],[82,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,"T",25,940,1080,null,null,null,107,null,null,null,"10.1016/j.actamat.2017.09.062",2018,"Study of a bcc multi-principal element alloy: Tensile and simple shear properties and underlying deformation mechanisms"],[83,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",25,905,null,null,null,null,107,null,null,null,"10.1016/j.msea.2015.08.024",2015,"On the room temperature deformation mechanisms of a TiZrHfNbTa refractory high-entropy alloy"],[84,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",25,890,null,null,null,null,107,null,null,null,"10.1016/j.matchar.2015.11.018",2016,"Mechanical behavior and microstructure of Ti20Hf20Zr20Ta20Nb20 high-entropy alloy loaded under quasi-static and dynamic compression conditions"],[85,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"T",25,828,862,null,null,null,107,null,null,null,"10.1016/j.msea.2015.12.017",2016,"Elastic and plastic properties of as-cast equimolar TiHfZrTaNb high-entropy alloy"],[85,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"T",25,827,856,null,null,null,107,null,null,null,"10.1016/j.msea.2015.12.017",2016,"Elastic and plastic properties of as-cast equimolar TiHfZrTaNb high-entropy alloy"],[85,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"T",25,820,869,null,null,null,107,null,null,null,"10.1016/j.msea.2015.12.017",2016,"Elastic and plastic properties of as-cast equimolar TiHfZrTaNb high-entropy alloy"],[85,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"T",25,803,810,null,null,null,107,null,null,null,"10.1016/j.msea.2015.12.017",2016,"Elastic and plastic properties of as-cast equimolar TiHfZrTaNb high-entropy alloy"],[86,"Al0.2 Mo1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,9.3,null,"C",25,1021,1249,null,null,null,183,null,null,null,"10.4028/www.scientific.net/MSF.898.638",2017,"Microstructure and Mechanical Properties of VTaTiMoAlsubx/sub Refractory High Entropy Alloys"],[86,"Al0.6 Mo1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,8.7,null,"C",25,962,1033,null,null,null,173,null,null,null,"10.4028/www.scientific.net/MSF.898.638",2017,"Microstructure and Mechanical Properties of VTaTiMoAlsubx/sub Refractory High Entropy Alloys"],[86,"Al1 Mo1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,8.2,null,"C",25,null,735,null,null,null,165,null,null,null,"10.4028/www.scientific.net/MSF.898.638",2017,"Microstructure and Mechanical Properties of VTaTiMoAlsubx/sub Refractory High Entropy Alloys"],[86,"Mo1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,9.6,null,"C",25,1221,1454,null,null,null,189,null,null,null,"10.4028/www.scientific.net/MSF.898.638",2017,"Microstructure and Mechanical Properties of VTaTiMoAlsubx/sub Refractory High Entropy Alloys"],[87,"Hf1 Nb1 Ta1 Zr1","BCC+HCP","ANNEAL","other",null,null,11.1,571,"C",25,2310,null,0.3,null,null,null,null,null,null,"10.1016/j.actamat.2016.01.018",2016,"Structural-disorder and its effect on mechanical properties in single-phase TaNbHfZr high-entropy alloy"],[87,"Hf1 Nb1 Ta1 Zr1","BCC+HCP","ANNEAL","other",null,null,11.1,536,"C",25,2100,2200,3.5,null,null,null,null,null,null,"10.1016/j.actamat.2016.01.018",2016,"Structural-disorder and its effect on mechanical properties in single-phase TaNbHfZr high-entropy alloy"],[87,"Hf1 Nb1 Ta1 Zr1","BCC+HCP","ANNEAL","other",null,null,11.1,null,"C",25,2020,2030,1.7,null,null,null,null,null,null,"10.1016/j.actamat.2016.01.018",2016,"Structural-disorder and its effect on mechanical properties in single-phase TaNbHfZr high-entropy alloy"],[87,"Hf1 Nb1 Ta1 Zr1","BCC+HCP","ANNEAL","other",null,null,11.1,550,"C",25,1950,2010,2.2,null,null,null,null,null,null,"10.1016/j.actamat.2016.01.018",2016,"Structural-disorder and its effect on mechanical properties in single-phase TaNbHfZr high-entropy alloy"],[87,"Hf1 Nb1 Ta1 Zr1","BCC","CAST","BCC",null,null,11.1,365,"C",25,1315,1800,21.6,null,null,105,null,null,null,"10.1016/j.actamat.2016.01.018",2016,"Structural-disorder and its effect on mechanical properties in single-phase TaNbHfZr high-entropy alloy"],[88,"Hf1 Ta0.4 Ti1 Zr1","BCC+HCP","CAST","other",null,null,9.2,null,"T",25,400,1100,null,null,null,null,null,null,null,"10.1002/adma.201701678",2017,"Phase-Transformation Ductilization of Brittle High-Entropy Alloys via Metastability Engineering"],[88,"Hf1 Ta0.5 Ti1 Zr1","BCC+HCP","CAST","other",null,null,9.4,null,"T",25,700,1100,null,null,null,null,null,null,null,"10.1002/adma.201701678",2017,"Phase-Transformation Ductilization of Brittle High-Entropy Alloys via Metastability Engineering"],[88,"Hf1 Ta0.6 Ti1 Zr1","BCC+HCP","CAST","other",null,null,9.6,null,"T",25,800,1100,null,null,null,null,null,null,null,"10.1002/adma.201701678",2017,"Phase-Transformation Ductilization of Brittle High-Entropy Alloys via Metastability Engineering"],[88,"Hf1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"T",25,1500,1500,null,null,null,107,null,null,null,"10.1002/adma.201701678",2017,"Phase-Transformation Ductilization of Brittle High-Entropy Alloys via Metastability Engineering"],[89,"Mo1 Nb1 Ta1 Ti0.25 W1","BCC","CAST","BCC",null,null,13.1,478.3,"C",25,1109,1197,null,null,null,242,null,null,null,"10.1016/j.msea.2017.12.004",2018,"Microstructures and mechanical properties of Ti NbMoTaW refractory high-entropy alloys"],[89,"Mo1 Nb1 Ta1 Ti0.5 W1","BCC","CAST","BCC",null,null,12.6,481.2,"C",25,1211,1578,null,null,null,234,null,null,null,"10.1016/j.msea.2017.12.004",2018,"Microstructures and mechanical properties of Ti NbMoTaW refractory high-entropy alloys"],[89,"Mo1 Nb1 Ta1 Ti0.75 W1","BCC","CAST","BCC",null,null,12.2,495.6,"C",25,1304,1593,null,null,null,228,null,null,null,"10.1016/j.msea.2017.12.004",2018,"Microstructures and mechanical properties of Ti NbMoTaW refractory high-entropy alloys"],[89,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,null,11.8,507.5,"C",25,1455,1910,null,null,null,222,null,null,null,"10.1016/j.msea.2017.12.004",2018,"Microstructures and mechanical properties of Ti NbMoTaW refractory high-entropy alloys"],[89,"Mo1 Nb1 Ta1 W1","BCC","CAST","BCC",null,null,13.7,null,"C",25,996,1148,null,null,null,250,null,null,null,"10.1016/j.msea.2017.12.004",2018,"Microstructures and mechanical properties of Ti NbMoTaW refractory high-entropy alloys"],[90,"Mo1 Nb1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,9.4,null,"C",25,1400,2450,null,null,null,171,null,null,null,"10.1016/j.jallcom.2016.11.188",2017,"Mechanical properties of refractory high-entropy alloys: Experiments and modeling"],[91,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,null,11,null,"C",600,973,1590,null,null,null,209,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,null,11,null,"C",800,791.3,1180,null,null,null,209,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,null,11,null,"C",1000,752.8,1105,null,null,null,209,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,null,11,null,"C",1200,659,696,null,null,null,209,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,null,11,null,"C",25,1515,2135,null,null,null,209,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,null,11.8,null,"C",600,689,1681,null,null,null,222,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,null,11.8,null,"C",800,674,1618,null,null,null,222,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,null,11.8,null,"C",1000,620,1426,null,null,null,222,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,null,11.8,null,"C",1200,586,814,null,null,null,222,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[91,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,null,11.8,null,"C",25,1343,2005,null,null,null,222,null,null,null,"10.1016/j.intermet.2017.01.007",2017,"Effect of Ti additions on mechanical properties of NbMoTaW and VNbMoTaW refractory high entropy alloys"],[92,"Mo1 Nb1 Ta1 Ti1 Zr1","BCC+BCC","CAST","other",null,null,9.1,null,"C",25,1390,null,null,null,null,null,null,null,null,"10.1016/j.msec.2016.12.057",2017,"TiZrNbTaMo high-entropy alloy designed for orthopedic implants: As-cast microstructure and mechanical properties"],[93,"Mo1 Nb1 Ta1 Ti1 Zr1","BCC+BCC","ANNEAL","other",null,null,9.1,null,"C",25,1100,1200,null,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.10.028",2017,"Novel TiNbTaZrMo high-entropy alloys for metallic biomaterials"],[93,"Mo1 Nb1 Ta1 Ti1 Zr1","BCC+BCC","CAST","other",null,null,9.1,null,"C",25,1375,1450,null,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.10.028",2017,"Novel TiNbTaZrMo high-entropy alloys for metallic biomaterials"],[94,"Mo1 Nb1 Ta1 V1 W1","BCC","POWDER","BCC",null,null,12.4,null,"C",25,2612,3472,null,null,null,229,null,null,null,"10.1016/j.msea.2017.12.021",2018,"Ultra-high strength WNbMoTaV high-entropy alloys with fine grain structure fabricated by powder metallurgical process"],[95,"Nb1 Ta1 Ti1 V1","BCC","CAST","BCC",null,null,9.2,null,"C",25,965,null,null,null,null,134,null,null,null,"10.1016/j.msea.2016.07.102",2016,"NbTaV-(Ti,W) refractory high-entropy alloys: Experiments and modeling"],[95,"Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,null,11.1,null,"C",25,1420,1800,20,null,null,187,null,null,null,"10.1016/j.msea.2016.07.102",2016,"NbTaV-(Ti,W) refractory high-entropy alloys: Experiments and modeling"],[95,"Nb1 Ta1 V1 W1","BCC","CAST","BCC",null,null,12.9,null,"C",25,1530,1700,12,null,null,206,null,null,null,"10.1016/j.msea.2016.07.102",2016,"NbTaV-(Ti,W) refractory high-entropy alloys: Experiments and modeling"],[96,"Al0.25 Cr0.5 Nb0.5 Ti1 V0.25","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",600,930,null,null,null,null,null,null,null,null,"10.1016/j.matlet.2016.11.030",2017,"Precipitation-strengthened refractory Al 0.5 CrNbTi 2 V 0.5 high entropy alloy"],[96,"Al0.25 Cr0.5 Nb0.5 Ti1 V0.25","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",800,445,null,null,null,null,null,null,null,null,"10.1016/j.matlet.2016.11.030",2017,"Precipitation-strengthened refractory Al 0.5 CrNbTi 2 V 0.5 high entropy alloy"],[96,"Al0.25 Cr0.5 Nb0.5 Ti1 V0.25","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",1000,90,null,null,null,null,null,null,null,null,"10.1016/j.matlet.2016.11.030",2017,"Precipitation-strengthened refractory Al 0.5 CrNbTi 2 V 0.5 high entropy alloy"],[96,"Al0.25 Cr0.5 Nb0.5 Ti1 V0.25","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",25,1340,1700,null,null,null,null,null,null,null,"10.1016/j.matlet.2016.11.030",2017,"Precipitation-strengthened refractory Al 0.5 CrNbTi 2 V 0.5 high entropy alloy"],[96,"Al0.25 Cr0.5 Nb0.5 Ti1 V0.25","BCC","CAST","BCC",null,null,5.8,null,"C",25,1240,null,null,null,null,134,null,null,null,"10.1016/j.matlet.2016.11.030",2017,"Precipitation-strengthened refractory Al 0.5 CrNbTi 2 V 0.5 high entropy alloy"],[97,"Al1 Cr0.5 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.6,null,"C",600,1005,1045,null,null,null,119,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Cr0.5 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.6,null,"C",800,640,685,null,null,null,119,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Cr0.5 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.6,null,"C",1000,40,80,null,null,null,119,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Cr0.5 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.6,null,"C",25,1300,1430,null,null,null,119,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al0.667 Cr1 Nb0.667 Ti0.667 V0.667","BCC+Laves","ANNEAL","other",null,null,5.9,null,"C",600,1370,1480,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al0.667 Cr1 Nb0.667 Ti0.667 V0.667","BCC+Laves","ANNEAL","other",null,null,5.9,null,"C",800,970,1045,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al0.667 Cr1 Nb0.667 Ti0.667 V0.667","BCC+Laves","ANNEAL","other",null,null,5.9,null,"C",1000,75,105,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al0.667 Cr1 Nb0.667 Ti0.667 V0.667","BCC+Laves","ANNEAL","other",null,null,5.9,null,"C",25,1700,1700,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Cr1 Nb1 Ti1 V1","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",600,1015,1060,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Cr1 Nb1 Ti1 V1","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",800,860,895,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Cr1 Nb1 Ti1 V1","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",1000,65,95,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Cr1 Nb1 Ti1 V1","BCC+Laves","ANNEAL","other",null,null,5.8,null,"C",25,1550,1570,null,null,null,null,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.5,null,"C",600,780,1005,null,null,null,104,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.5,null,"C",800,560,700,null,null,null,104,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.5,null,"C",1000,110,null,null,null,null,104,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[97,"Al1 Nb1 Ti1 V1","BCC","ANNEAL","BCC",null,null,5.5,null,"C",25,1000,1280,null,null,null,104,null,null,null,"10.1016/j.jallcom.2015.08.224",2015,"Structure and mechanical properties of the AlCrxNbTiV (x = 0, 0.5, 1, 1.5) high entropy alloys"],[140,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC+B2+Sec.","WROUGHT","other",0.71,null,7.6,null,"T",25,900,1074,25,null,null,null,null,null,null,"10.1016/j.scriptamat.2018.08.048",2019,"Fatigue behavior of ultrafine grained triplex Al0.3CoCrFeNi high entropy alloy"],[151,"Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",75,null,7.8,null,"C",25,219,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.05 Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.6,null,"C",25,354,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.15 Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.4,null,"C",25,550,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.25 Cr0.5 Fe1 Mn0.625 Ni0.375","B2+BCC","CAST","other",175,null,7.2,null,"C",25,994,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.35 Cr0.5 Fe1 Mn0.625 Ni0.375","B2+BCC","CAST","other",175,null,6.9,null,"C",25,942,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.8,null,"C",400,132,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.05 Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.6,null,"C",400,260,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.15 Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.4,null,"C",400,469,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.25 Cr0.5 Fe1 Mn0.625 Ni0.375","B2+BCC","CAST","other",null,null,7.2,null,"C",400,943,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.35 Cr0.5 Fe1 Mn0.625 Ni0.375","B2+BCC","CAST","other",null,null,6.9,null,"C",400,833,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.8,null,"C",600,115,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.05 Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.6,null,"C",600,155,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.15 Cr0.5 Fe1 Mn0.625 Ni0.375","FCC+BCC","CAST","other",null,null,7.4,null,"C",600,272,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.25 Cr0.5 Fe1 Mn0.625 Ni0.375","B2+BCC","CAST","other",null,null,7.2,null,"C",600,359,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[151,"Al0.35 Cr0.5 Fe1 Mn0.625 Ni0.375","B2+BCC","CAST","other",null,null,6.9,null,"C",600,326,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.093",2019,"Effect of Al on structure and mechanical properties of Fe-Mn-Cr-Ni-Al non-equiatomic high entropy alloys with high Fe content"],[198,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",4.4,null,8,null,"T",25,370,670,56,null,null,220,null,null,null,"10.1016/j.actamat.2013.06.018",2013,"The influences of temperature and microstructure on the tensile properties of a CoCrFeMnNi high-entropy alloy"],[174,"Co0.2 Cr0.2 Fe1 Mn0.6","FCC+HCP","ANNEAL","other",45,null,7.8,null,"T",25,250,730,50,null,null,null,null,null,null,"10.1016/j.actamat.2017.03.069",2017,"A TRIP-assisted dual-phase high-entropy alloy: Grain size and phase fraction effects on deformation behavior"],[174,"Co0.2 Cr0.2 Fe1 Mn0.6","FCC+HCP","WROUGHT","other",4.5,null,7.8,null,"T",25,330,870,75,null,null,null,null,null,null,"10.1016/j.actamat.2017.03.069",2017,"A TRIP-assisted dual-phase high-entropy alloy: Grain size and phase fraction effects on deformation behavior"],[174,"Co0.2 Cr0.2 Fe1 Mn0.6","FCC+HCP","WROUGHT","other",6,null,7.8,null,"T",25,280,800,65,null,null,null,null,null,null,"10.1016/j.actamat.2017.03.069",2017,"A TRIP-assisted dual-phase high-entropy alloy: Grain size and phase fraction effects on deformation behavior"],[174,"Co0.2 Cr0.2 Fe1 Mn0.6","FCC+HCP","WROUGHT","other",15,null,7.8,null,"T",25,350,830,63,null,null,null,null,null,null,"10.1016/j.actamat.2017.03.069",2017,"A TRIP-assisted dual-phase high-entropy alloy: Grain size and phase fraction effects on deformation behavior"],[176,"Co0.2 Cr0.2 Fe1 Mn0.6","FCC+HCP","WROUGHT","other",6.5,null,7.8,null,"T",25,298,1400,45,null,null,null,null,null,null,"10.1038/s41598-017-16509-9",2017,"Enhanced strength and ductility in a friction stir processing engineered dual phase high entropy alloy"],[176,"Co0.2 Cr0.2 Fe1 Mn0.6","FCC+HCP","WROUGHT","other",5.2,null,7.8,null,"T",25,400,1200,42,null,null,null,null,null,null,"10.1038/s41598-017-16509-9",2017,"Enhanced strength and ductility in a friction stir processing engineered dual phase high entropy alloy"],[183,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+B2",null,"other",0.7,null,7.6,null,"T",25,860,1050,48,null,null,null,null,null,null,"10.1016/j.matlet.2017.04.072",2017,"Formation of ultrafine-grained microstructure in Al 0.3 CoCrFeNi high entropy alloys with grain boundary precipitates"],[183,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+B2",null,"other",2.7,null,7.6,null,"T",25,690,930,51,null,null,null,null,null,null,"10.1016/j.matlet.2017.04.072",2017,"Formation of ultrafine-grained microstructure in Al 0.3 CoCrFeNi high entropy alloys with grain boundary precipitates"],[183,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+B2",null,"other",3.8,null,7.6,null,"T",25,550,800,64,null,null,null,null,null,null,"10.1016/j.matlet.2017.04.072",2017,"Formation of ultrafine-grained microstructure in Al 0.3 CoCrFeNi high entropy alloys with grain boundary precipitates"],[183,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC",null,"FCC",53,null,7.6,null,"T",25,260,620,87,null,null,210,null,null,null,"10.1016/j.matlet.2017.04.072",2017,"Formation of ultrafine-grained microstructure in Al 0.3 CoCrFeNi high entropy alloys with grain boundary precipitates"],[183,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+BCC","WROUGHT","other",null,null,7.6,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matlet.2017.04.072",2017,"Formation of ultrafine-grained microstructure in Al 0.3 CoCrFeNi high entropy alloys with grain boundary precipitates"],[179,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,7.6,null,null,null,null,null,null,null,null,210,null,null,null,"10.1016/j.scriptamat.2016.06.019",2016,"Stability of ordered L12 and B2 precipitates in face centered cubic based high entropy alloys - Al0.3CoFeCrNi and Al0.3CuFeCrNi2"],[179,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+L12","WROUGHT","other",null,null,7.6,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.06.019",2016,"Stability of ordered L12 and B2 precipitates in face centered cubic based high entropy alloys - Al0.3CoFeCrNi and Al0.3CuFeCrNi2"],[179,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+L12","WROUGHT","other",null,null,7.6,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.06.019",2016,"Stability of ordered L12 and B2 precipitates in face centered cubic based high entropy alloys - Al0.3CoFeCrNi and Al0.3CuFeCrNi2"],[178,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",144,null,7.6,null,"T",25,159,410,65,null,null,210,null,null,null,"10.1016/j.matdes.2017.02.072",2017,"Optimizing the coupled effects of Hall-Petch and precipitation strengthening in a Al 0.3 CoCrFeNi high entropy alloy"],[178,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+B2","WROUGHT","other",144,null,7.6,null,"T",25,215,520,43,null,null,null,null,null,null,"10.1016/j.matdes.2017.02.072",2017,"Optimizing the coupled effects of Hall-Petch and precipitation strengthening in a Al 0.3 CoCrFeNi high entropy alloy"],[178,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+L12","WROUGHT","other",144,null,7.6,null,"T",25,285,540,55,null,null,null,null,null,null,"10.1016/j.matdes.2017.02.072",2017,"Optimizing the coupled effects of Hall-Petch and precipitation strengthening in a Al 0.3 CoCrFeNi high entropy alloy"],[178,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",23,null,7.6,null,"T",25,263,589,60,null,null,210,null,null,null,"10.1016/j.matdes.2017.02.072",2017,"Optimizing the coupled effects of Hall-Petch and precipitation strengthening in a Al 0.3 CoCrFeNi high entropy alloy"],[178,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+L12+B2","WROUGHT","other",23,null,7.6,null,"T",25,490,840,45,null,null,null,null,null,null,"10.1016/j.matdes.2017.02.072",2017,"Optimizing the coupled effects of Hall-Petch and precipitation strengthening in a Al 0.3 CoCrFeNi high entropy alloy"],[184,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,7.6,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2017.03.066",2017,"Secondary phases in AlxCoCrFeNi high-entropy alloys: An in-situ TEM heating study and thermodynamic appraisal"],[185,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+L12","CAST","other",null,null,7.6,null,"T",25,175,335,58,null,null,null,null,null,null,"10.1016/j.jallcom.2008.12.088",2009,"Microstructure and tensile behaviors of FCC Al0.3CoCrFeNi high entropy alloy"],[185,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+L12","ANNEAL","other",null,null,7.6,null,"T",25,305,525,44,null,null,null,null,null,null,"10.1016/j.jallcom.2008.12.088",2009,"Microstructure and tensile behaviors of FCC Al0.3CoCrFeNi high entropy alloy"],[185,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+B2","ANNEAL","other",null,null,7.6,null,"T",25,280,570,44,null,null,null,null,null,null,"10.1016/j.jallcom.2008.12.088",2009,"Microstructure and tensile behaviors of FCC Al0.3CoCrFeNi high entropy alloy"],[186,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,7.6,null,null,null,null,null,null,null,null,210,null,null,null,"10.1016/j.jallcom.2013.11.084",2014,"Phases, microstructure and mechanical properties of AlxCoCrFeNi high-entropy alloys at elevated temperatures"],[186,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,7.6,null,null,null,null,null,null,null,null,210,null,null,null,"10.1016/j.jallcom.2013.11.084",2014,"Phases, microstructure and mechanical properties of AlxCoCrFeNi high-entropy alloys at elevated temperatures"],[187,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC+B2","ANNEAL","other",null,null,7.6,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2016.05.079",2016,"The effect of grain size on the annealing-induced phase transformation in an Al0·3CoCrFeNi high entropy alloy"],[190,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.7,null,"T",700,395,400,1,null,null,null,null,null,null,"10.1016/j.msea.2015.08.078",2015,"Tensile ductility of an AlCoCrFeNi multi-phase high-entropy alloy through hot isostatic pressing (HIP) and homogenization"],[188,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.7,null,"C",25,1380,null,10,null,null,null,null,null,null,"10.1016/j.jallcom.2016.05.034",2016,"Heat treatment impacts the micro-structure and mechanical properties of AlCoCrFeNi high entropy alloy"],[188,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC+Sec.","ANNEAL","other",null,null,6.7,null,"C",25,1430,null,1.6,null,null,null,null,null,null,"10.1016/j.jallcom.2016.05.034",2016,"Heat treatment impacts the micro-structure and mechanical properties of AlCoCrFeNi high entropy alloy"],[188,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","ANNEAL","other",null,null,6.7,null,"C",25,1690,null,2.7,null,null,null,null,null,null,"10.1016/j.jallcom.2016.05.034",2016,"Heat treatment impacts the micro-structure and mechanical properties of AlCoCrFeNi high entropy alloy"],[188,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","ANNEAL","other",null,null,6.7,null,"C",25,1220,null,13,null,null,null,null,null,null,"10.1016/j.jallcom.2016.05.034",2016,"Heat treatment impacts the micro-structure and mechanical properties of AlCoCrFeNi high entropy alloy"],[188,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","ANNEAL","other",null,null,6.7,null,"C",25,1450,null,20,null,null,null,null,null,null,"10.1016/j.jallcom.2016.05.034",2016,"Heat treatment impacts the micro-structure and mechanical properties of AlCoCrFeNi high entropy alloy"],[189,"Al1 Co1 Cr1 Fe1 Ni1","FCC+BCC","ANNEAL","other",null,null,6.7,null,"T",25,900,935,0.8,null,null,null,null,null,null,"10.1016/j.jallcom.2017.03.307",2017,"In-situ study of crack initiation and propagation in a dual phase AlCoCrFeNi high entropy alloy"],[191,"Al0.515 Cr0.515 Fe1 Ni1","FCC+BCC+B2","CAST","other",null,null,6.9,null,"T",25,796,1437,15.7,null,null,null,null,null,null,"10.1016/j.matlet.2016.01.096",2016,"A multi-component AlCrFe2Ni2 alloy with excellent mechanical properties"],[192,"Al0.511 Co0.511 Cr0.511 Fe0.511 Ni1","L12+B2",null,"other",null,null,7,null,"T",25,545,1030,16.5,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.465 Co0.465 Cr0.465 Fe0.465 Ni1","L12+B2",null,"other",null,null,7.1,null,"T",25,545,1050,16.5,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.451 Co0.451 Cr0.451 Fe0.451 Ni1","L12+B2",null,"other",null,null,7.1,null,"T",25,545,1110,20,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.511 Co0.511 Cr0.511 Fe0.511 Ni1",null,null,"other",null,null,7,null,"T",-70,580,1034,10.5,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.465 Co0.465 Cr0.465 Fe0.465 Ni1",null,null,"other",null,null,7.1,null,"T",-70,595,1168,15.8,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.451 Co0.451 Cr0.451 Fe0.451 Ni1",null,null,"other",null,null,7.1,null,"T",-70,570,1143,18,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.511 Co0.511 Cr0.511 Fe0.511 Ni1",null,null,"other",null,null,7,null,"T",-192,700,1000,4,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.465 Co0.465 Cr0.465 Fe0.465 Ni1",null,null,"other",null,null,7.1,null,"T",-192,700,1000,7,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.451 Co0.451 Cr0.451 Fe0.451 Ni1",null,null,"other",null,null,7.1,null,"T",-192,700,1000,9,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[195,"Al0.343 Co0.514 Cr0.514 Fe1 Ni0.514","BCC+B2","CAST","other",null,null,7.2,null,"T",25,866,1223,7.9,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.04.014",2016,null],[195,"Al0.343 Co0.514 Cr0.514 Fe1 Ni0.514",null,"CAST","other",null,null,7.2,null,"T",650,400,468,6.3,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.04.014",2016,null],[195,"Al0.343 Co0.514 Cr0.514 Fe1 Ni0.514",null,"CAST","other",null,null,7.2,null,"T",700,220,340,5.4,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.04.014",2016,null],[195,"Al0.343 Co0.514 Cr0.514 Fe1 Ni0.514",null,"CAST","other",null,null,7.2,null,"C",25,1166,null,30.2,null,null,null,null,null,null,"10.1016/j.scriptamat.2016.04.014",2016,null],[199,"Co1 Cr0.429 Fe0.571 Mo0.286 Ni0.571","FCC","CAST","FCC",null,null,8.6,null,"T",25,350,720,21.9,null,null,234,null,null,null,"10.1016/j.scriptamat.2017.05.019",2017,"Precipitation strengthening of ductile Cr 15 Fe 20 Co 35 Ni 20 Mo 10 alloys"],[199,"Co1 Cr0.429 Fe0.571 Mo0.286 Ni0.571","FCC+Sec.","WROUGHT","other",null,null,8.6,null,"T",25,1311,1410,12.1,null,null,null,null,null,null,"10.1016/j.scriptamat.2017.05.019",2017,"Precipitation strengthening of ductile Cr 15 Fe 20 Co 35 Ni 20 Mo 10 alloys"],[199,"Co1 Cr0.429 Fe0.571 Mo0.286 Ni0.571","FCC+Sec.","WROUGHT","other",null,null,8.6,null,"T",25,1212,1360,14.9,null,null,null,null,null,null,"10.1016/j.scriptamat.2017.05.019",2017,"Precipitation strengthening of ductile Cr 15 Fe 20 Co 35 Ni 20 Mo 10 alloys"],[199,"Co1 Cr0.429 Fe0.571 Mo0.286 Ni0.571","FCC+Sec.","WROUGHT","other",null,null,8.6,null,"T",25,1028,1249,18.3,null,null,null,null,null,null,"10.1016/j.scriptamat.2017.05.019",2017,"Precipitation strengthening of ductile Cr 15 Fe 20 Co 35 Ni 20 Mo 10 alloys"],[199,"Co1 Cr0.429 Fe0.571 Mo0.286 Ni0.571","FCC+Sec.","WROUGHT","other",null,null,8.6,null,"T",25,879,1194,25.4,null,null,null,null,null,null,"10.1016/j.scriptamat.2017.05.019",2017,"Precipitation strengthening of ductile Cr 15 Fe 20 Co 35 Ni 20 Mo 10 alloys"],[199,"Co1 Cr0.429 Fe0.571 Mo0.286 Ni0.571","FCC+Sec.","WROUGHT","other",null,null,8.6,null,"T",25,799,1127,28.2,null,null,null,null,null,null,"10.1016/j.scriptamat.2017.05.019",2017,"Precipitation strengthening of ductile Cr 15 Fe 20 Co 35 Ni 20 Mo 10 alloys"],[199,"Co1 Cr0.429 Fe0.571 Mo0.286 Ni0.571","FCC","WROUGHT","FCC",null,null,8.6,null,"T",25,350,918,62.4,null,null,234,null,null,null,"10.1016/j.scriptamat.2017.05.019",2017,"Precipitation strengthening of ductile Cr 15 Fe 20 Co 35 Ni 20 Mo 10 alloys"],[197,"Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.2,null,"T",25,155,472.4,58.9,null,null,226,null,null,null,"10.1016/j.actamat.2016.06.063",2016,"Ductile CoCrFeNiMox high entropy alloys strengthened by hard intermetallic phases"],[197,"Co1 Cr1 Fe1 Mo0.083 Ni1",null,"CAST","other",null,null,8.2,null,"T",25,198.8,479,51.1,null,null,null,null,null,null,"10.1016/j.actamat.2016.06.063",2016,"Ductile CoCrFeNiMox high entropy alloys strengthened by hard intermetallic phases"],[197,"Co1 Cr1 Fe1 Mo0.208 Ni1",null,"CAST","other",null,null,8.3,null,"T",25,254.7,589.6,55.1,null,null,null,null,null,null,"10.1016/j.actamat.2016.06.063",2016,"Ductile CoCrFeNiMox high entropy alloys strengthened by hard intermetallic phases"],[197,"Co1 Cr1 Fe1 Mo0.304 Ni1","FCC+Sec.","CAST","other",null,null,8.4,null,"T",25,305.3,709.7,49.3,null,null,null,null,null,null,"10.1016/j.actamat.2016.06.063",2016,"Ductile CoCrFeNiMox high entropy alloys strengthened by hard intermetallic phases"],[197,"Co1 Cr1 Fe1 Mo0.304 Ni1","FCC+Sec.","WROUGHT","other",3,null,8.4,null,"T",25,815.5,1186.5,18.9,null,null,null,null,null,null,"10.1016/j.actamat.2016.06.063",2016,"Ductile CoCrFeNiMox high entropy alloys strengthened by hard intermetallic phases"],[197,"Co1 Cr1 Fe1 Mo0.304 Ni1","FCC+Sec.","WROUGHT","other",3.1,null,8.4,null,"T",25,646.7,1042,32.5,null,null,null,null,null,null,"10.1016/j.actamat.2016.06.063",2016,"Ductile CoCrFeNiMox high entropy alloys strengthened by hard intermetallic phases"],[197,"Co1 Cr1 Fe1 Mo0.304 Ni1","FCC+Sec.","WROUGHT","other",1.6,null,8.4,null,"T",25,683.7,1066.6,30.4,null,null,null,null,null,null,"10.1016/j.actamat.2016.06.063",2016,"Ductile CoCrFeNiMox high entropy alloys strengthened by hard intermetallic phases"],[177,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",13,null,8,null,"T",25,243,558,50,null,null,220,null,null,null,"10.1080/02670836.2017.1342367",2017,"Controllable fabrication of a carbide-containing FeCoCrNiMn high-entropy alloy: microstructure and mechanical properties"],[177,"C0.02 Co1 Cr1 Fe1 Mn1 Ni1","FCC+Sec.","WROUGHT","other",8,null,8,null,"T",25,446,723,15,null,null,null,null,null,null,"10.1080/02670836.2017.1342367",2017,"Controllable fabrication of a carbide-containing FeCoCrNiMn high-entropy alloy: microstructure and mechanical properties"],[177,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",7,null,8,null,"T",25,249,621,41,null,null,220,null,null,null,"10.1080/02670836.2017.1342367",2017,"Controllable fabrication of a carbide-containing FeCoCrNiMn high-entropy alloy: microstructure and mechanical properties"],[177,"C0.02 Co1 Cr1 Fe1 Mn1 Ni1","FCC+Sec.","WROUGHT","other",4,null,8,null,"T",25,581,857,28,null,null,null,null,null,null,"10.1080/02670836.2017.1342367",2017,"Controllable fabrication of a carbide-containing FeCoCrNiMn high-entropy alloy: microstructure and mechanical properties"],[196,"Co1 Cr1 Fe1 Ni1","FCC","ANNEAL","FCC",null,null,8.2,null,"T",25,190,453,70,null,null,226,null,null,null,"10.1016/j.actamat.2015.08.076",2016,"A precipitation-hardened high-entropy alloy with outstanding tensile properties"],[196,"Al0.167 Co1 Cr1 Fe1 Ni1 Ti0.083","FCC","ANNEAL","FCC",289.7,null,7.8,null,"T",25,210,503,70,null,null,214,null,null,null,"10.1016/j.actamat.2015.08.076",2016,"A precipitation-hardened high-entropy alloy with outstanding tensile properties"],[196,"Al0.167 Co1 Cr1 Fe1 Ni1 Ti0.083","FCC+L12+Sec.","WROUGHT","other",15.7,null,7.8,null,"T",25,645,1094,39,null,null,null,null,null,null,"10.1016/j.actamat.2015.08.076",2016,"A precipitation-hardened high-entropy alloy with outstanding tensile properties"],[196,"Al0.167 Co1 Cr1 Fe1 Ni1 Ti0.083","FCC+L12+Sec.","WROUGHT","other",2.8,null,7.8,null,"T",25,1005,1273,17,null,null,null,null,null,null,"10.1016/j.actamat.2015.08.076",2016,"A precipitation-hardened high-entropy alloy with outstanding tensile properties"],[201,"Al0.242 Co0.515 Cr0.515 Cu0.242 Fe0.515 Ni1",null,"ANNEAL","other",null,null,7.7,168,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.ultramic.2015.06.009",2015,"Influence of W, Mo and Ti trace elements on the phase separation in Al8Co17Cr17Cu8Fe17Ni33 based high entropy alloy"],[201,"Al0.242 Co0.515 Cr0.515 Cu0.242 Fe0.515 Ni1",null,"ANNEAL","other",null,null,7.7,328,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.ultramic.2015.06.009",2015,"Influence of W, Mo and Ti trace elements on the phase separation in Al8Co17Cr17Cu8Fe17Ni33 based high entropy alloy"],[167,"Al0.242 Co0.515 Cr0.515 Cu0.242 Fe0.515 Ni1",null,"ANNEAL","other",null,null,7.7,206,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matchar.2018.06.036",2019,"New multiphase compositionally complex alloys driven by the high entropy alloy approach"],[201,"Al0.229 Co0.486 Cr0.4 Cu0.229 Fe0.486 Mo0.003 Ni1 Ti0.029 W0.003",null,"CAST","other",null,null,7.7,231,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.ultramic.2015.06.009",2015,"Influence of W, Mo and Ti trace elements on the phase separation in Al8Co17Cr17Cu8Fe17Ni33 based high entropy alloy"],[201,"Al0.229 Co0.486 Cr0.4 Cu0.229 Fe0.486 Mo0.003 Ni1 Ti0.029 W0.003",null,"ANNEAL","other",null,null,7.7,178,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.ultramic.2015.06.009",2015,"Influence of W, Mo and Ti trace elements on the phase separation in Al8Co17Cr17Cu8Fe17Ni33 based high entropy alloy"],[201,"Al0.229 Co0.486 Cr0.4 Cu0.229 Fe0.486 Mo0.003 Ni1 Ti0.029 W0.003",null,"ANNEAL","other",null,null,7.7,343,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.ultramic.2015.06.009",2015,"Influence of W, Mo and Ti trace elements on the phase separation in Al8Co17Cr17Cu8Fe17Ni33 based high entropy alloy"],[167,"Al0.229 Co0.486 Cr0.4 Cu0.229 Fe0.486 Mo0.003 Ni1 Ti0.029 W0.003",null,"ANNEAL","other",null,null,7.7,171,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matchar.2018.06.036",2019,"New multiphase compositionally complex alloys driven by the high entropy alloy approach"],[175,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167",null,"CAST","other",null,null,7.4,343,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11837-015-1484-7",2015,"High-Temperature Tensile Strength of Al10Co25Cr8Fe15Ni36Ti6 Compositionally Complex Alloy (High-Entropy Alloy)"],[175,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167",null,"ANNEAL","other",null,null,7.4,341,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11837-015-1484-7",2015,"High-Temperature Tensile Strength of Al10Co25Cr8Fe15Ni36Ti6 Compositionally Complex Alloy (High-Entropy Alloy)"],[175,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC+L12+Sec.","ANNEAL","other",null,null,7.4,345,"T",25,568,786,12,null,null,null,null,null,null,"10.1007/s11837-015-1484-7",2015,"High-Temperature Tensile Strength of Al10Co25Cr8Fe15Ni36Ti6 Compositionally Complex Alloy (High-Entropy Alloy)"],[175,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC+L12+Sec.","ANNEAL","other",null,null,7.4,345,"T",25,596,1039,20,null,null,null,null,null,null,"10.1007/s11837-015-1484-7",2015,"High-Temperature Tensile Strength of Al10Co25Cr8Fe15Ni36Ti6 Compositionally Complex Alloy (High-Entropy Alloy)"],[175,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC+L12+Sec.","ANNEAL","other",null,null,7.4,345,"T",700,487,702,18,null,null,null,null,null,null,"10.1007/s11837-015-1484-7",2015,"High-Temperature Tensile Strength of Al10Co25Cr8Fe15Ni36Ti6 Compositionally Complex Alloy (High-Entropy Alloy)"],[175,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC+L12+Sec.","ANNEAL","other",null,null,7.4,345,"T",700,486,624,11,null,null,null,null,null,null,"10.1007/s11837-015-1484-7",2015,"High-Temperature Tensile Strength of Al10Co25Cr8Fe15Ni36Ti6 Compositionally Complex Alloy (High-Entropy Alloy)"],[193,"Al0.885 Cr1 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.4,null,"C",25,770,null,37.1,null,null,null,null,null,null,"10.1016/j.msea.2016.11.019",2017,"Effects of aluminum on microstructure and compressive properties of Al-Cr-Fe-Ni eutectic multi-component alloys"],[193,"Al1 Cr1 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.3,null,"C",25,843.2,null,34.6,null,null,null,null,null,null,"10.1016/j.msea.2016.11.019",2017,"Effects of aluminum on microstructure and compressive properties of Al-Cr-Fe-Ni eutectic multi-component alloys"],[193,"Al1 Cr0.889 Fe0.889 Ni0.889","BCC+B2","CAST","other",null,null,6.1,null,"C",25,1044,null,27.1,null,null,null,null,null,null,"10.1016/j.msea.2016.11.019",2017,"Effects of aluminum on microstructure and compressive properties of Al-Cr-Fe-Ni eutectic multi-component alloys"],[193,"Al1 Cr0.828 Fe0.828 Ni0.828","BCC+B2","CAST","other",null,null,6,null,"C",25,906.8,null,31.4,null,null,null,null,null,null,"10.1016/j.msea.2016.11.019",2017,"Effects of aluminum on microstructure and compressive properties of Al-Cr-Fe-Ni eutectic multi-component alloys"],[193,"Al1 Cr0.767 Fe0.767 Ni0.767","BCC+B2","CAST","other",null,null,5.9,null,"C",25,1122.8,null,24.7,null,null,null,null,null,null,"10.1016/j.msea.2016.11.019",2017,"Effects of aluminum on microstructure and compressive properties of Al-Cr-Fe-Ni eutectic multi-component alloys"],[191,"Al0.5 Cr0.5 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.9,null,"C",25,null,null,17,null,null,null,null,null,null,"10.1016/j.matlet.2016.01.096",2016,"A multi-component AlCrFe2Ni2 alloy with excellent mechanical properties"],[168,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.7,null,"C",25,null,null,22.7,null,null,null,null,null,null,"10.1016/j.matdes.2013.12.048",2014,"Effect of vanadium addition on the microstructure and properties of AlCoCrFeNi high entropy alloy"],[182,"Al0.545 Cr0.545 Fe0.515 Ni1 Ti0.273","BCC+B2+L12","CAST","other",null,null,6.4,null,"C",25,null,null,16.3,null,null,null,null,null,null,"10.1016/j.jmst.2014.09.011",2015,"Mechanical Properties Improvement of AlCrFeNi2Ti0.5 High Entropy Alloy through Annealing Design and its Relationship with its Particle-reinforced Microstructures"],[182,"Al0.545 Cr0.545 Fe0.515 Ni1 Ti0.273","BCC+B2+L12","ANNEAL","other",null,null,6.4,null,"C",25,null,null,19.8,null,null,null,null,null,null,"10.1016/j.jmst.2014.09.011",2015,"Mechanical Properties Improvement of AlCrFeNi2Ti0.5 High Entropy Alloy through Annealing Design and its Relationship with its Particle-reinforced Microstructures"],[181,"Cr1 Fe1 Ni1 V0.5 W1","FCC+BCC+Sec.","CAST","other",null,null,10.9,null,"C",25,null,null,7.3,null,null,null,null,null,null,"10.1007/s11665-015-1767-8",2015,"Effects of Tungsten on Microstructure and Mechanical Properties of CrFeNiV0.5W x and CrFeNi2V0.5W x High-Entropy Alloys"],[169,"Al0.515 Cr0.515 Cu0.515 Fe0.515 Ni1","FCC+BCC+B2","CAST","other",null,null,7.1,null,"C",25,null,null,30,null,null,null,null,null,null,"10.1016/j.msea.2015.09.089",2016,"Strain rate effects on the dynamic mechanical properties of the AlCrCuFeNi2 high-entropy alloy"],[202,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","ANNEAL","FCC",null,null,8,null,"C",25,230,null,75,null,null,220,null,null,null,"10.1134/S0031918X17060084",2017,"Effect of heat treatment on the structure and hardness of high-entropy alloys CoCrFeNiMnV x  (x = 0.25, 0.5, 0.75, 1)"],[202,"Co1 Cr1 Fe1 Mn1 Ni1 V0.263","FCC+Sec.","ANNEAL","other",null,null,7.9,null,"C",25,200,null,75,null,null,null,null,null,null,"10.1134/S0031918X17060084",2017,"Effect of heat treatment on the structure and hardness of high-entropy alloys CoCrFeNiMnV x  (x = 0.25, 0.5, 0.75, 1)"],[202,"Co1 Cr1 Fe1 Mn1 Ni1 V0.5","FCC+Sec.","ANNEAL","other",null,null,7.8,null,"C",25,620,null,75,null,null,null,null,null,null,"10.1134/S0031918X17060084",2017,"Effect of heat treatment on the structure and hardness of high-entropy alloys CoCrFeNiMnV x  (x = 0.25, 0.5, 0.75, 1)"],[202,"Co1 Cr1 Fe1 Mn1 Ni1 V0.765","FCC+Sec.","ANNEAL","other",null,null,7.7,null,"C",25,780,null,8,null,null,null,null,null,null,"10.1134/S0031918X17060084",2017,"Effect of heat treatment on the structure and hardness of high-entropy alloys CoCrFeNiMnV x  (x = 0.25, 0.5, 0.75, 1)"],[202,"Co1 Cr1 Fe1 Mn1 Ni1 V1","FCC+Sec.","ANNEAL","other",null,null,7.7,null,"C",25,1700,null,1,null,null,null,null,null,null,"10.1134/S0031918X17060084",2017,"Effect of heat treatment on the structure and hardness of high-entropy alloys CoCrFeNiMnV x  (x = 0.25, 0.5, 0.75, 1)"],[170,"Al0.304 Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,null,"C",25,null,null,25,null,null,210,null,null,null,"10.1016/j.msea.2015.02.072",2015,"Comparative study of the microstructures and mechanical properties of direct laser fabricated and arc-melted Al x CoCrFeNi high entropy alloys"],[171,"Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.2,null,"C",25,null,null,83,null,null,226,null,null,null,"10.1016/j.jallcom.2013.12.210",2014,"Effect of Mn and V on structure and mechanical properties of high-entropy alloys based on CoCrFeNi system"],[192,"Al0.511 Co0.511 Cr0.511 Fe0.511 Ni1","FCC+B2","CAST","other",null,null,7,null,"T",25,545,1030,16.5,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.465 Co0.465 Cr0.465 Fe0.465 Ni1","FCC+B2","CAST","other",null,null,7.1,null,"T",25,545,1050,16.5,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[192,"Al0.451 Co0.451 Cr0.451 Fe0.451 Ni1","FCC+B2","CAST","other",null,null,7.1,null,"T",25,545,1110,20,null,null,null,null,null,null,"10.1016/j.actamat.2016.11.016",2017,"Directly cast bulk eutectic and near-eutectic high entropy alloys with balanced strength and ductility in a wide temperature range"],[194,"Al0.108 Co0.514 Cr0.514 Cu0.108 Fe0.514 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,719,1048,30,null,null,209,null,null,null,"10.1016/j.msea.2017.04.111",2017,null],[194,"Al0.108 Co0.514 Cr0.514 Cu0.108 Fe0.514 Ni1","FCC+L12","WROUGHT","other",null,null,8,null,"T",25,460,732,32,null,null,null,null,null,null,"10.1016/j.msea.2017.04.111",2017,null],[200,"Al0.242 Co0.515 Cr0.515 Cu0.242 Fe0.515 Ni1","Other","CAST","other",null,null,7.7,null,"T",25,357,459,9,null,null,null,null,null,null,"10.1007/s11837-013-0756-3",2013,"Microstructure and Tensile Behavior of Al8Co17Cr17Cu8Fe17Ni33 (at.%) High-Entropy Alloy"],[200,"Al0.242 Co0.515 Cr0.515 Cu0.242 Fe0.515 Ni1","Other","ANNEAL","other",null,null,7.7,null,"T",25,365,365,0.1,null,null,null,null,null,null,"10.1007/s11837-013-0756-3",2013,"Microstructure and Tensile Behavior of Al8Co17Cr17Cu8Fe17Ni33 (at.%) High-Entropy Alloy"],[200,"Al0.242 Co0.515 Cr0.515 Cu0.242 Fe0.515 Ni1","FCC+L12","ANNEAL","other",null,null,7.7,null,"T",25,215,489,39,null,null,null,null,null,null,"10.1007/s11837-013-0756-3",2013,"Microstructure and Tensile Behavior of Al8Co17Cr17Cu8Fe17Ni33 (at.%) High-Entropy Alloy"],[172,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","Other","WROUGHT","other",null,null,7.6,null,"T",25,1284,1344,7.6,null,null,null,null,null,null,"10.1016/j.actamat.2015.07.004",2015,"Fatigue behavior of a wrought Al0.5CoCrCuFeNi two-phase high-entropy alloy"],[173,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,7.6,null,"T",25,1290,1400,5,null,null,189,null,null,null,"10.1016/j.jallcom.2009.10.088",2010,"Effect of temperature on mechanical properties of Al0.5CoCrCuFeNi wrought alloy"],[173,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","Other","WROUGHT","other",null,null,7.6,null,"T",25,650,790,27,null,null,null,null,null,null,"10.1016/j.jallcom.2009.10.088",2010,"Effect of temperature on mechanical properties of Al0.5CoCrCuFeNi wrought alloy"],[180,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,7.1,null,"T",25,790,790,0.2,null,null,null,null,null,null,"10.1016/j.msea.2011.11.045",2012,"Tensile properties of an AlCrCuNiFeCo high-entropy alloy in as-cast and wrought conditions"],[180,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","BCC+Sec.","WROUGHT","other",null,null,7.1,null,"T",25,1040,1170,1,null,null,null,null,null,null,"10.1016/j.msea.2011.11.045",2012,"Tensile properties of an AlCrCuNiFeCo high-entropy alloy in as-cast and wrought conditions"],[190,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2","CAST","other",null,null,6.7,null,"T",700,395,400,1,null,null,null,null,null,null,"10.1016/j.msea.2015.08.078",2015,"Tensile ductility of an AlCoCrFeNi multi-phase high-entropy alloy through hot isostatic pressing (HIP) and homogenization"],[190,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC+Sec.","OTHER","other",null,null,6.7,null,"T",700,295,393,12,null,null,null,null,null,null,"10.1016/j.msea.2015.08.078",2015,"Tensile ductility of an AlCoCrFeNi multi-phase high-entropy alloy through hot isostatic pressing (HIP) and homogenization"],[180,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","BCC+Sec.","CAST","other",null,null,7.1,null,"T",700,350,360,5,null,null,null,null,null,null,"10.1016/j.msea.2011.11.045",2012,"Tensile properties of an AlCrCuNiFeCo high-entropy alloy in as-cast and wrought conditions"],[180,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","BCC+Sec.","WROUGHT","other",null,null,7.1,null,"T",700,63,91,63,null,null,null,null,null,null,"10.1016/j.msea.2011.11.045",2012,"Tensile properties of an AlCrCuNiFeCo high-entropy alloy in as-cast and wrought conditions"],[173,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,7.6,null,"T",700,180,185,5,null,null,189,null,null,null,"10.1016/j.jallcom.2009.10.088",2010,"Effect of temperature on mechanical properties of Al0.5CoCrCuFeNi wrought alloy"],[173,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","Other","WROUGHT","other",null,null,7.6,null,"T",700,160,180,12,null,null,null,null,null,null,"10.1016/j.jallcom.2009.10.088",2010,"Effect of temperature on mechanical properties of Al0.5CoCrCuFeNi wrought alloy"],[214,"Co1 Cr1 Fe1 Ni1 Ti0.2","FCC","WROUGHT","FCC",51,null,7.9,null,"T",25,700,1240,36,null,null,218,null,null,null,"10.1016/j.actamat.2018.11.049",2019,"Outstanding tensile properties of a precipitation-strengthened FeCoNiCrTi0.2 high-entropy alloy at room and cryogenic temperatures"],[214,"Co1 Cr1 Fe1 Ni1 Ti0.2","FCC","WROUGHT","FCC",null,null,7.9,null,"T",-196,860,1580,46,null,null,218,null,null,null,"10.1016/j.actamat.2018.11.049",2019,"Outstanding tensile properties of a precipitation-strengthened FeCoNiCrTi0.2 high-entropy alloy at room and cryogenic temperatures"],[222,"Al0.1 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",20,null,8,null,"T",25,1198,null,1.5,null,null,220,null,null,null,"10.1016/j.actamat.2018.12.012",2019,"Enhancement of strength-ductility trade-off in a high-entropy alloy through a heterogeneous structure"],[222,"Al0.1 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",20,null,8,null,"T",25,1403,null,1.2,null,null,220,null,null,null,"10.1016/j.actamat.2018.12.012",2019,"Enhancement of strength-ductility trade-off in a high-entropy alloy through a heterogeneous structure"],[222,"Al0.1 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",20,null,8,null,"T",25,711,928,30.3,null,null,220,null,null,null,"10.1016/j.actamat.2018.12.012",2019,"Enhancement of strength-ductility trade-off in a high-entropy alloy through a heterogeneous structure"],[222,"Al0.1 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",7,null,8,null,"T",25,356,null,42.4,null,null,220,null,null,null,"10.1016/j.actamat.2018.12.012",2019,"Enhancement of strength-ductility trade-off in a high-entropy alloy through a heterogeneous structure"],[236,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",0.5,null,8,null,"T",25,1120,null,1.1,null,null,220,null,null,null,"10.1016/j.msea.2018.10.094",2019,"Achieving high ductility in the 1.7 GPa grade CoCrFeMnNi high-entropy alloy at 77 K"],[236,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",-196,1692,null,10.3,null,null,220,null,null,null,"10.1016/j.msea.2018.10.094",2019,"Achieving high ductility in the 1.7 GPa grade CoCrFeMnNi high-entropy alloy at 77 K"],[245,"Al0.5 Cr0.5 Fe0.5 Ni1","FCC+BCC+B2","CAST","other",null,null,6.7,null,"T",25,774,1357,6.4,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.176",2019,"A new CrFeNi2Al eutectic high entropy alloy system with excellent mechanical properties"],[245,"Al0.429 Cr0.476 Fe0.476 Ni1","FCC+BCC+B2","CAST","other",null,null,6.9,null,"T",25,610,1173,9.1,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.176",2019,"A new CrFeNi2Al eutectic high entropy alloy system with excellent mechanical properties"],[245,"Al0.364 Cr0.455 Fe0.455 Ni1","FCC+BCC+B2","CAST","other",null,null,7,null,"T",25,479,956,12.7,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.176",2019,"A new CrFeNi2Al eutectic high entropy alloy system with excellent mechanical properties"],[245,"Al0.304 Cr0.435 Fe0.435 Ni1","FCC+BCC+B2","CAST","other",null,null,7.2,null,"T",25,461,835,30,null,null,null,null,null,null,"10.1016/j.jallcom.2018.08.176",2019,"A new CrFeNi2Al eutectic high entropy alloy system with excellent mechanical properties"],[245,"Al0.25 Cr0.417 Fe0.417 Ni1","FCC","CAST","FCC",null,null,7.3,null,"T",25,441,757,45,null,null,196,null,null,null,"10.1016/j.jallcom.2018.08.176",2019,"A new CrFeNi2Al eutectic high entropy alloy system with excellent mechanical properties"],[98,"Co1 Cr1 Ni1","FCC","WROUGHT","FCC",1.6,null,8.3,468,"T",25,1100,null,23,null,null,231,null,null,null,"10.1016/j.actamat.2018.12.015",2019,"Achieving ultra-high strength and ductility in equiatomic CrCoNi with partially recrystallized microstructures"],[98,"Co1 Cr1 Ni1","FCC","WROUGHT","FCC",2.3,null,8.3,326,"T",25,597,null,53,null,null,231,null,null,null,"10.1016/j.actamat.2018.12.015",2019,"Achieving ultra-high strength and ductility in equiatomic CrCoNi with partially recrystallized microstructures"],[98,"Co1 Cr1 Ni1","FCC","WROUGHT","FCC",24,null,8.3,223,"T",25,346,null,76,null,null,231,null,null,null,"10.1016/j.actamat.2018.12.015",2019,"Achieving ultra-high strength and ductility in equiatomic CrCoNi with partially recrystallized microstructures"],[105,"Al0.476 Co0.476 Cr0.476 Fe0.476 Ni1","FCC+B2","WROUGHT","other",null,null,7.1,null,"T",25,1263,null,21.4,null,null,null,null,null,null,"10.1038/s41467-019-08460-2",2019,"Enhanced strength–ductility synergy in ultrafine-grained eutectic high-entropy alloys by inheriting microstructural lamellae"],[105,"Al0.476 Co0.476 Cr0.476 Fe0.476 Ni1","FCC+B2","WROUGHT","other",null,null,7.1,null,"T",25,1154,null,25,null,null,null,null,null,null,"10.1038/s41467-019-08460-2",2019,"Enhanced strength–ductility synergy in ultrafine-grained eutectic high-entropy alloys by inheriting microstructural lamellae"],[105,"Al0.476 Co0.476 Cr0.476 Fe0.476 Ni1","FCC+B2","WROUGHT","other",null,null,7.1,null,"T",25,1490,null,16,null,null,null,null,null,null,"10.1038/s41467-019-08460-2",2019,"Enhanced strength–ductility synergy in ultrafine-grained eutectic high-entropy alloys by inheriting microstructural lamellae"],[115,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8,130.2,"C",25,210,null,57,null,null,220,null,null,null,"10.1016/j.scriptamat.2018.11.042",2019,"Annealing-induced abnormal hardening in a cold rolled CrMnFeCoNi high entropy alloy"],[115,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,394.1,"C",25,1150,null,6,null,null,220,null,null,null,"10.1016/j.scriptamat.2018.11.042",2019,"Annealing-induced abnormal hardening in a cold rolled CrMnFeCoNi high entropy alloy"],[115,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,430.7,"C",25,1300,null,0.7,null,null,220,null,null,null,"10.1016/j.scriptamat.2018.11.042",2019,"Annealing-induced abnormal hardening in a cold rolled CrMnFeCoNi high entropy alloy"],[125,"Nb1 Ta1 Ti1 V1","BCC","POWDER","BCC",24,null,9.2,427.9,"C",25,1108,null,28.3,null,null,134,null,null,null,"10.1016/j.jallcom.2018.10.230",2019,"Microstructures and mechanical properties of ductile NbTaTiV refractory high entropy alloy prepared by powder metallurgy"],[125,"Nb1 Ta1 Ti1 V1","BCC","POWDER","BCC",30,null,9.2,444.3,"C",25,1308,null,27.9,null,null,134,null,null,null,"10.1016/j.jallcom.2018.10.230",2019,"Microstructures and mechanical properties of ductile NbTaTiV refractory high entropy alloy prepared by powder metallurgy"],[125,"Nb1 Ta1 Ti1 V1","BCC","POWDER","BCC",38,null,9.2,510.9,"C",25,1373,null,29.5,null,null,134,4800,null,1900,"10.1016/j.jallcom.2018.10.230",2019,"Microstructures and mechanical properties of ductile NbTaTiV refractory high entropy alloy prepared by powder metallurgy"],[135,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",30,null,8,null,"T",25,300,676,52,null,null,220,null,null,null,"10.1016/j.jmst.2018.09.068",2019,"Fatigue behavior of CoCrFeMnNi high-entropy alloy under fully reversed cyclic deformation"],[135,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",0.65,null,8,null,"T",25,800,888,39.5,null,null,220,null,null,null,"10.1016/j.jmst.2018.09.068",2019,"Fatigue behavior of CoCrFeMnNi high-entropy alloy under fully reversed cyclic deformation"],[136,"Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",0.4,null,8,415,"C",25,1314,null,null,20.3,null,null,null,null,null,"10.1016/j.jallcom.2018.10.168",2019,"Microstructure and mechanical properties of FeCoCrNiMnAlx high-entropy alloys prepared by mechanical alloying and hot-pressed sintering"],[136,"Al0.1 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.9,432,"C",25,1631,null,null,12.6,null,null,null,null,null,"10.1016/j.jallcom.2018.10.168",2019,"Microstructure and mechanical properties of FeCoCrNiMnAlx high-entropy alloys prepared by mechanical alloying and hot-pressed sintering"],[136,"Al0.3 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",0.2,null,7.6,511,"C",25,1836,null,null,4.9,null,null,null,null,null,"10.1016/j.jallcom.2018.10.168",2019,"Microstructure and mechanical properties of FeCoCrNiMnAlx high-entropy alloys prepared by mechanical alloying and hot-pressed sintering"],[136,"Al0.5 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,7.4,553,"C",25,1932,null,null,3.8,null,null,null,null,null,"10.1016/j.jallcom.2018.10.168",2019,"Microstructure and mechanical properties of FeCoCrNiMnAlx high-entropy alloys prepared by mechanical alloying and hot-pressed sintering"],[136,"Al0.7 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",0.2,null,7.1,622,"C",25,2230,null,null,1.69,null,null,null,null,null,"10.1016/j.jallcom.2018.10.168",2019,"Microstructure and mechanical properties of FeCoCrNiMnAlx high-entropy alloys prepared by mechanical alloying and hot-pressed sintering"],[136,"Al1 Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,6.8,684,"C",25,null,null,null,0,null,null,null,null,null,"10.1016/j.jallcom.2018.10.168",2019,"Microstructure and mechanical properties of FeCoCrNiMnAlx high-entropy alloys prepared by mechanical alloying and hot-pressed sintering"],[137,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,8.2,null,"T",20,446,664,63,null,189,226,null,null,null,"10.1007/s40843-018-9373-y",2018,"Excellent ductility and serration feature of metastable CoCrFeNi high-entropy alloy at extremely low temperatures"],[137,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,8.2,null,"T",-73,488,790,72,null,194,226,null,null,null,"10.1007/s40843-018-9373-y",2018,"Excellent ductility and serration feature of metastable CoCrFeNi high-entropy alloy at extremely low temperatures"],[137,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,8.2,null,"T",-196,590,1070,78,null,204,226,null,null,null,"10.1007/s40843-018-9373-y",2018,"Excellent ductility and serration feature of metastable CoCrFeNi high-entropy alloy at extremely low temperatures"],[137,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,8.2,null,"T",-223,null,1152,68.6,null,null,226,null,null,null,"10.1007/s40843-018-9373-y",2018,"Excellent ductility and serration feature of metastable CoCrFeNi high-entropy alloy at extremely low temperatures"],[137,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,8.2,null,"T",-253,null,1240,62,null,null,226,null,null,null,"10.1007/s40843-018-9373-y",2018,"Excellent ductility and serration feature of metastable CoCrFeNi high-entropy alloy at extremely low temperatures"],[137,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,8.2,null,"T",-268.8,680,1251,61.6,null,221,226,null,null,null,"10.1007/s40843-018-9373-y",2018,"Excellent ductility and serration feature of metastable CoCrFeNi high-entropy alloy at extremely low temperatures"],[138,"Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.2,94.7,null,null,null,null,null,null,null,226,null,null,null,"10.1016/j.matdes.2019.107698",2019,"Microstructural evolution, electrochemical and corrosion properties of Al CoCrFeNiTihigh entropy alloys"],[138,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,117,null,null,null,null,null,null,null,210,null,null,null,"10.1016/j.matdes.2019.107698",2019,"Microstructural evolution, electrochemical and corrosion properties of Al CoCrFeNiTihigh entropy alloys"],[138,"Al0.6 Co1 Cr1 Fe1 Ni1","FCC+BCC+B2","CAST","other",null,null,7.2,228,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2019.107698",2019,"Microstructural evolution, electrochemical and corrosion properties of Al CoCrFeNiTihigh entropy alloys"],[138,"Al0.9 Co1 Cr1 Fe1 Ni1","FCC+BCC+B2","CAST","other",null,null,6.8,511,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2019.107698",2019,"Microstructural evolution, electrochemical and corrosion properties of Al CoCrFeNiTihigh entropy alloys"],[138,"Al0.9 Co1 Cr1 Fe1 Ni1 Ti0.5","BCC+B2+Sec.","CAST","other",null,null,6.5,623,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2019.107698",2019,"Microstructural evolution, electrochemical and corrosion properties of Al CoCrFeNiTihigh entropy alloys"],[139,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8,null,"C",25,216,null,null,null,null,220,null,null,null,"10.1016/j.jmst.2018.10.009",2019,"Strengthening FCC-CoCrFeMnNi high entropy alloys by Mo addition"],[139,"Co1 Cr1 Fe1 Mn1 Mo0.042 Ni1","FCC","CAST","FCC",null,null,8.1,null,"C",25,225,null,null,null,null,221,null,null,null,"10.1016/j.jmst.2018.10.009",2019,"Strengthening FCC-CoCrFeMnNi high entropy alloys by Mo addition"],[139,"Co1 Cr1 Fe1 Mn1 Mo0.087 Ni1","FCC+Sec.","CAST","other",null,null,8.1,null,"C",25,330,null,null,null,null,null,null,null,null,"10.1016/j.jmst.2018.10.009",2019,"Strengthening FCC-CoCrFeMnNi high entropy alloys by Mo addition"],[139,"Co1 Cr1 Fe1 Mn1 Mo0.136 Ni1","FCC+Sec.","CAST","other",null,null,8.1,null,"C",25,560,null,null,null,null,null,null,null,null,"10.1016/j.jmst.2018.10.009",2019,"Strengthening FCC-CoCrFeMnNi high entropy alloys by Mo addition"],[139,"Co1 Cr1 Fe1 Mn1 Mo0.19 Ni1","FCC+Sec.","CAST","other",null,null,8.1,null,"C",25,765,null,null,null,null,null,null,null,null,"10.1016/j.jmst.2018.10.009",2019,"Strengthening FCC-CoCrFeMnNi high entropy alloys by Mo addition"],[141,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC+L12","WROUGHT","other",null,null,7.6,null,"T",25,143,400,65,null,null,null,null,null,null,"10.1016/j.scriptamat.2018.10.023",2019,"Tensile yield strength of a single bulk Al0.3CoCrFeNi high entropy alloy can be tuned from 160 MPa to 1800 MPa"],[141,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC+L12","WROUGHT","other",null,null,7.6,null,"T",25,471,850,45,null,null,null,null,null,null,"10.1016/j.scriptamat.2018.10.023",2019,"Tensile yield strength of a single bulk Al0.3CoCrFeNi high entropy alloy can be tuned from 160 MPa to 1800 MPa"],[141,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC+B2+Sec.","WROUGHT","other",null,null,7.6,null,"T",25,841,1050,35,null,null,null,null,null,null,"10.1016/j.scriptamat.2018.10.023",2019,"Tensile yield strength of a single bulk Al0.3CoCrFeNi high entropy alloy can be tuned from 160 MPa to 1800 MPa"],[141,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC+BCC+B2+Sec.","WROUGHT","other",null,null,7.6,null,"T",25,161,1650,7,null,null,null,null,null,null,"10.1016/j.scriptamat.2018.10.023",2019,"Tensile yield strength of a single bulk Al0.3CoCrFeNi high entropy alloy can be tuned from 160 MPa to 1800 MPa"],[141,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC+BCC+B2+Sec.","WROUGHT","other",null,null,7.6,null,"T",25,187,1850,5,null,null,null,null,null,null,"10.1016/j.scriptamat.2018.10.023",2019,"Tensile yield strength of a single bulk Al0.3CoCrFeNi high entropy alloy can be tuned from 160 MPa to 1800 MPa"],[142,"Al1 Co1 Cr1 Ni1","BCC+B2","CAST","other",null,null,6.4,1183,"C",25,1750,null,16.7,null,null,null,null,null,null,"10.1016/j.jallcom.2018.11.057",2019,"Chemical evolution-induced strengthening on AlCoCrNi dual-phase high-entropy alloy with high specific strength"],[143,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",7.9,null,8,null,"T",25,378.7,700.1,54,null,null,220,null,null,null,"10.1016/j.msea.2019.01.050",2019,"Effects of carbon on the microstructures and mechanical properties of FeCoCrNiMn high entropy alloys"],[143,"C0.5 Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",4.7,null,7.6,null,"T",25,520.9,791.4,40.3,null,null,null,null,null,36380,"10.1016/j.msea.2019.01.050",2019,"Effects of carbon on the microstructures and mechanical properties of FeCoCrNiMn high entropy alloys"],[143,"C1 Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",3.5,null,7.3,null,"T",25,643.8,887.6,31.9,null,null,null,null,null,30300,"10.1016/j.msea.2019.01.050",2019,"Effects of carbon on the microstructures and mechanical properties of FeCoCrNiMn high entropy alloys"],[144,"Co1 Cr0.588 Fe0.176 Mo0.176 Ni1","FCC","WROUGHT","FCC",null,null,8.6,null,"T",25,null,null,8,null,null,231,null,null,null,"10.1016/j.ijplas.2018.10.005",2019,"Strength and ductility of CrFeCoNiMo alloy with hierarchical microstructures"],[144,"Co1 Cr0.588 Fe0.176 Mo0.176 Ni1","FCC","WROUGHT","FCC",0.7,null,8.6,null,"T",25,null,null,14,null,null,231,null,null,null,"10.1016/j.ijplas.2018.10.005",2019,"Strength and ductility of CrFeCoNiMo alloy with hierarchical microstructures"],[144,"Co1 Cr0.588 Fe0.176 Mo0.176 Ni1","FCC","WROUGHT","FCC",0.8,null,8.6,null,"T",25,1100,1300,32,null,null,231,null,null,null,"10.1016/j.ijplas.2018.10.005",2019,"Strength and ductility of CrFeCoNiMo alloy with hierarchical microstructures"],[144,"Co1 Cr0.588 Fe0.176 Mo0.176 Ni1","FCC","WROUGHT","FCC",1.2,null,8.6,null,"T",25,950,null,36,null,null,231,null,null,null,"10.1016/j.ijplas.2018.10.005",2019,"Strength and ductility of CrFeCoNiMo alloy with hierarchical microstructures"],[144,"Co1 Cr0.588 Fe0.176 Mo0.176 Ni1","FCC","WROUGHT","FCC",3.2,null,8.6,null,"T",25,649,null,42,null,null,231,null,null,null,"10.1016/j.ijplas.2018.10.005",2019,"Strength and ductility of CrFeCoNiMo alloy with hierarchical microstructures"],[144,"Co1 Cr0.588 Fe0.176 Mo0.176 Ni1","FCC","WROUGHT","FCC",18.2,null,8.6,null,"T",25,439,null,63,null,null,231,null,null,null,"10.1016/j.ijplas.2018.10.005",2019,"Strength and ductility of CrFeCoNiMo alloy with hierarchical microstructures"],[144,"Co1 Cr0.588 Fe0.176 Mo0.176 Ni1","FCC","WROUGHT","FCC",94.1,null,8.6,null,"T",25,309,null,82,null,null,231,null,null,null,"10.1016/j.ijplas.2018.10.005",2019,"Strength and ductility of CrFeCoNiMo alloy with hierarchical microstructures"],[145,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,386.2,666.3,57.5,null,null,220,null,null,null,"10.1016/j.jallcom.2018.08.115",2019,"Effects of (W, Cr) carbide on grain refinement and mechanical properties for CoCrFeMnNi high entropy alloys"],[146,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","CAST","other",null,null,8.1,null,"T",25,628,950,14.5,null,null,null,null,null,null,"10.1016/j.matchemphys.2018.09.033",2019,"Evaluation of the mechanical properties of the heat treated FeCrCuMnNi high entropy alloy"],[146,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","ANNEAL","other",null,null,8.1,null,"T",25,545,690,5,null,null,null,null,null,null,"10.1016/j.matchemphys.2018.09.033",2019,"Evaluation of the mechanical properties of the heat treated FeCrCuMnNi high entropy alloy"],[146,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","ANNEAL","other",null,null,8.1,null,"T",25,534,682,5.2,null,null,null,null,null,null,"10.1016/j.matchemphys.2018.09.033",2019,"Evaluation of the mechanical properties of the heat treated FeCrCuMnNi high entropy alloy"],[146,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","ANNEAL","other",null,null,8.1,null,"T",25,525,680,4.3,null,null,null,null,null,null,"10.1016/j.matchemphys.2018.09.033",2019,"Evaluation of the mechanical properties of the heat treated FeCrCuMnNi high entropy alloy"],[146,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","ANNEAL","other",null,null,8.1,null,"T",25,517,677,4.6,null,null,null,null,null,null,"10.1016/j.matchemphys.2018.09.033",2019,"Evaluation of the mechanical properties of the heat treated FeCrCuMnNi high entropy alloy"],[146,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","ANNEAL","other",null,null,8.1,null,"T",25,502,671,4.9,null,null,null,null,null,null,"10.1016/j.matchemphys.2018.09.033",2019,"Evaluation of the mechanical properties of the heat treated FeCrCuMnNi high entropy alloy"],[147,"Cr1 Mo1 Nb1 Ta1 V1 W1","BCC+Laves+Sec.","POWDER","other",0.54,null,11.7,1072,"C",25,null,null,null,null,null,null,7946,null,null,"10.1016/j.jallcom.2018.11.318",2019,"A fine-grained NbMoTaWVCr refractory high-entropy alloy with ultra-high strength: Microstructural evolution and mechanical properties"],[147,"Cr1 Mo1 Nb1 Ta1 V1 W1","BCC+Laves+Sec.","POWDER","other",1.24,null,11.7,1010,"C",25,3416,null,null,5.3,null,null,7946,null,null,"10.1016/j.jallcom.2018.11.318",2019,"A fine-grained NbMoTaWVCr refractory high-entropy alloy with ultra-high strength: Microstructural evolution and mechanical properties"],[147,"Cr1 Mo1 Nb1 Ta1 V1 W1","BCC+Laves+Sec.","POWDER","other",5.01,null,11.7,991.3,"C",25,3410,null,null,2,null,null,7946,null,null,"10.1016/j.jallcom.2018.11.318",2019,"A fine-grained NbMoTaWVCr refractory high-entropy alloy with ultra-high strength: Microstructural evolution and mechanical properties"],[147,"Cr1 Mo1 Nb1 Ta1 V1 W1","BCC+Laves+Sec.","POWDER","other",10.8,null,11.7,988.3,"C",25,3338,null,null,1.9,null,null,7946,null,null,"10.1016/j.jallcom.2018.11.318",2019,"A fine-grained NbMoTaWVCr refractory high-entropy alloy with ultra-high strength: Microstructural evolution and mechanical properties"],[148,"Cr1 Mo1 Nb1 Ta1 V1 W1",null,"CAST","other",null,null,11.7,704.6,"C",25,null,null,null,null,null,null,null,null,null,"10.1016/j.calphad.2015.09.007",2015,"Senary refractory high-entropy alloy Cr MoNbTaVW"],[149,"Co1 Cr1 Cu1 Fe1 Ni1","FCC+B2","CAST","other",null,null,8.3,160,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.vacuum.2018.12.009",2019,"The effect of Al content on microstructures and comprehensive properties in AlxCoCrCuFeNi high entropy alloys"],[149,"Al0.5 Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,160,null,null,null,null,null,null,null,189,null,null,null,"10.1016/j.vacuum.2018.12.009",2019,"The effect of Al content on microstructures and comprehensive properties in AlxCoCrCuFeNi high entropy alloys"],[149,"Al1 Co1 Cr1 Cu1 Fe1 Ni1","FCC+BCC","CAST","other",null,null,7.1,458,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.vacuum.2018.12.009",2019,"The effect of Al content on microstructures and comprehensive properties in AlxCoCrCuFeNi high entropy alloys"],[149,"Al1 Co0.667 Cr0.667 Cu0.667 Fe0.667 Ni0.667","FCC+BCC","CAST","other",null,null,6.6,498,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.vacuum.2018.12.009",2019,"The effect of Al content on microstructures and comprehensive properties in AlxCoCrCuFeNi high entropy alloys"],[149,"Al1 Co0.5 Cr0.5 Cu0.5 Fe0.5 Ni0.5","FCC+BCC","CAST","other",null,null,6.3,560,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.vacuum.2018.12.009",2019,"The effect of Al content on microstructures and comprehensive properties in AlxCoCrCuFeNi high entropy alloys"],[150,"Co1 Ni1 V1",null,"WROUGHT","other",2,null,7.8,null,"T",25,991,1359,38,null,null,null,null,null,null,"10.1002/adma.201807142",2018,"Ultrastrong Medium-Entropy Single-Phase Alloys Designed via Severe Lattice Distortion"],[150,"Co1 Ni1 V1",null,"WROUGHT","other",5.6,null,7.8,null,"T",25,767,1224,46,null,null,null,null,null,null,"10.1002/adma.201807142",2018,"Ultrastrong Medium-Entropy Single-Phase Alloys Designed via Severe Lattice Distortion"],[150,"Co1 Ni1 V1",null,"WROUGHT","other",18.7,null,7.8,null,"T",25,602,1123,51,null,null,null,null,null,null,"10.1002/adma.201807142",2018,"Ultrastrong Medium-Entropy Single-Phase Alloys Designed via Severe Lattice Distortion"],[150,"Co1 Ni1 V1",null,"WROUGHT","other",27.8,null,7.8,null,"T",25,517,1049,55,null,null,null,null,null,null,"10.1002/adma.201807142",2018,"Ultrastrong Medium-Entropy Single-Phase Alloys Designed via Severe Lattice Distortion"],[150,"Co1 Ni1 V1",null,"WROUGHT","other",122.2,null,7.8,null,"T",25,461,886,58,null,null,null,null,null,null,"10.1002/adma.201807142",2018,"Ultrastrong Medium-Entropy Single-Phase Alloys Designed via Severe Lattice Distortion"],[152,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC+B2+Sec.","WROUGHT","other",null,null,7.6,null,"T",25,840,1039,32,null,null,null,null,null,null,"10.1016/j.actamat.2018.12.010",2019,"Enhancing strength and strain hardenability via deformation twinning in fcc-based high entropy alloys reinforced with intermetallic compounds"],[152,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,7.6,null,"T",25,159,410,65,null,null,210,null,null,null,"10.1016/j.actamat.2018.12.010",2019,"Enhancing strength and strain hardenability via deformation twinning in fcc-based high entropy alloys reinforced with intermetallic compounds"],[153,"Al0.544 C0.001 Co0.69 Fe1 Ni0.69","B2+L12","CAST","other",null,null,7,null,"C",25,1202,null,32.6,null,null,null,null,null,11067,"10.1016/j.scriptamat.2018.09.008",2019,"A novel high-entropy alloy with a dendrite-composite microstructure and remarkable compression performance"],[154,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,null,null,null,null,null,null,107,null,null,null,"10.1016/j.matchar.2018.07.015",2019,"Body-centered cubic high-entropy alloys: From processing to underlying deformation mechanisms"],[155,"Co0.5 Cr0.5 Fe1 Ni1","FCC","WROUGHT","FCC",14.9,null,8.2,null,"T",25,214,612,77.3,null,null,219,null,null,null,"10.1080/21663831.2018.1553803",2018,"Novel Co-rich high entropy alloys with superior tensile properties"],[155,"Co0.875 Cr0.625 Fe1 Ni1","FCC","WROUGHT","FCC",11.6,null,8.3,null,"T",25,241,721,96.2,null,null,220,null,null,null,"10.1080/21663831.2018.1553803",2018,"Novel Co-rich high entropy alloys with superior tensile properties"],[155,"Co1 Cr0.556 Fe0.667 Ni0.667","FCC","WROUGHT","FCC",10.5,null,8.3,null,"T",25,299,841,76.1,null,null,222,null,null,null,"10.1080/21663831.2018.1553803",2018,"Novel Co-rich high entropy alloys with superior tensile properties"],[155,"Co1 Cr0.455 Fe0.364 Ni0.364","FCC+HCP","WROUGHT","other",5.6,null,8.3,null,"T",25,392,746,20.3,null,null,null,null,null,null,"10.1080/21663831.2018.1553803",2018,"Novel Co-rich high entropy alloys with superior tensile properties"],[155,"Co1 Cr0.385 Fe0.154 Ni0.154","FCC+HCP","WROUGHT","other",6.2,null,8.4,null,"T",25,482,857,13.3,null,null,null,null,null,null,"10.1080/21663831.2018.1553803",2018,"Novel Co-rich high entropy alloys with superior tensile properties"],[156,"Hf1 Nb1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,8.4,null,null,null,null,null,null,null,69.7,89,null,null,null,"10.1016/j.intermet.2019.04.003",2019,"Evaluating elastic properties of a body-centered cubic NbHfZrTi high-entropy alloy – A direct comparison between experiments and ab initio calculations"],[157,"Co1 Cr1 Fe1 Mn1 Ni1",null,null,"other",null,null,8,null,null,null,null,null,null,null,202,null,null,null,null,"10.1016/j.intermet.2014.11.005",2015,"Polycrystalline elastic moduli of a high-entropy alloy at cryogenic temperatures"],[158,"Co1 Cr1 Fe1 Mn1 Ni1",null,null,"other",null,null,8,null,null,null,null,null,null,null,202.2,null,null,null,null,"10.1557/adv.2017.76",2017,"Monocrystalline elastic constants of fcc-CrMnFeCoNi high entropy alloy"],[159,"Co1 Cr1 Fe1 Ni1",null,null,"other",null,null,8.2,null,null,null,null,null,null,null,204,null,null,null,null,"10.1016/j.msea.2016.05.082",2016,"Effects of the sp element additions on the microstructure and mechanical properties of NiCoFeCr based high entropy alloys"],[160,"Co1 Fe1 Ni1",null,null,"other",null,null,8.5,null,null,null,null,null,null,null,174,null,null,null,null,"10.1016/j.msea.2017.04.003",2017,"Intrinsic properties and strengthening mechanism of monocrystalline Ni-containing ternary concentrated solid solutions"],[161,"Co1 Cr1 Ni1",null,null,"other",null,null,8.3,null,null,null,null,null,null,null,222,null,null,null,null,"10.1016/j.msea.2017.06.086",2017,"Mechanical and microstructural characterization of powder metallurgy CoCrNi medium entropy alloy"],[162,"Co1 Cr1 Ni1",null,null,"other",null,null,8.3,null,null,null,null,null,null,null,229,null,null,null,null,"10.1038/ncomms10602",2016,"Exceptional damage-tolerance of a medium-entropy alloy CrCoNi at cryogenic temperatures"],[160,"Co1 Cr1 Ni1",null,null,"other",null,null,8.3,null,null,null,null,null,null,null,234,null,null,null,null,"10.1016/j.msea.2017.04.003",2017,"Intrinsic properties and strengthening mechanism of monocrystalline Ni-containing ternary concentrated solid solutions"],[163,"Al1 Co0.5 Cr1 Fe0.147 Ni0.294",null,null,"other",null,null,5.7,699,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.298 Cr0.319 Cu0.106 Fe0.277 Ni0.128",null,null,"other",null,null,5.1,728,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.289 Cr0.158 Cu0.237 Fe0.816 Ni0.132",null,null,"other",null,null,5.6,604,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.278 Cr0.861 Cu0.167 Fe0.25 Ni0.222",null,null,"other",null,null,5.6,790,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.395 Cr0.349 Fe0.326 Ni0.256",null,null,"other",null,null,5.3,806,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.255 Cr0.255 Cu0.106 Fe0.277 Ni0.234",null,null,"other",null,null,5.1,811,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.442 Cr0.442 Fe0.256 Ni0.186",null,null,"other",null,null,5.3,855,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.256 Cr0.795 Cu0.128 Fe0.205 Ni0.179",null,null,"other",null,null,5.4,802,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.277 Cr0.277 Cu0.106 Fe0.255 Ni0.213",null,null,"other",null,null,5.1,799,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.465 Cr0.512 Fe0.186 Ni0.163",null,null,"other",null,null,5.3,857,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.298 Cr0.404 Cu0.106 Fe0.213 Ni0.106",null,null,"other",null,null,5.1,855,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.465 Cr0.442 Fe0.209 Ni0.209",null,null,"other",null,null,5.3,816,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.304 Cr0.5 Cu0.109 Fe0.152 Ni0.109",null,null,"other",null,null,5.1,811,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.318 Cr0.5 Cu0.114 Fe0.205 Ni0.136",null,null,"other",null,null,5.2,790,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.442 Cr0.558 Fe0.186 Ni0.14",null,null,"other",null,null,5.2,855,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.419 Cr0.558 Fe0.233 Ni0.116",null,null,"other",null,null,5.2,856,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.298 Cr0.426 Cu0.106 Fe0.191 Ni0.106",null,null,"other",null,null,5.1,859,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.558 Cr0.512 Fe0.116 Ni0.14",null,null,"other",null,null,5.3,875,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.581 Cr0.512 Fe0.116 Ni0.116",null,null,"other",null,null,5.3,865,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.558 Cr0.535 Fe0.116 Ni0.116",null,null,"other",null,null,5.3,868,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.535 Cr0.488 Fe0.186 Ni0.116",null,null,"other",null,null,5.3,863,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.255 Cr0.426 Cu0.106 Fe0.234 Ni0.106",null,null,"other",null,null,5,843,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.349 Cr0.395 Cu0.116 Fe0.326 Ni0.14",null,null,"other",null,null,5.3,794,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.238 Cr0.714 Cu0.119 Fe0.167 Ni0.143",null,null,"other",null,null,5.2,783,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.234 Cr0.468 Cu0.106 Fe0.213 Ni0.106",null,null,"other",null,null,5,797,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.277 Cr0.426 Cu0.106 Fe0.213 Ni0.106",null,null,"other",null,null,5,805,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al0.853 Co0.647 Cr1 Fe0.147 Ni0.294",null,null,"other",null,null,6,699,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.149 Cr0.34 Cu0.191 Fe0.34 Ni0.106",null,null,"other",null,null,5.1,749,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.326 Cr0.488 Cu0.116 Fe0.279 Ni0.116",null,null,"other",null,null,5.3,828,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.383 Cr0.319 Cu0.106 Fe0.213 Ni0.106",null,null,"other",null,null,5.1,849,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.426 Cr0.383 Cu0.106 Fe0.106 Ni0.106","BCC+B2",null,"other",null,null,5.1,883,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.419 Cr0.465 Fe0.279 Ni0.163",null,null,"other",null,null,5.3,864,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.362 Cr0.34 Cu0.106 Fe0.213 Ni0.106",null,null,"other",null,null,5.1,841,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.558 Cr0.535 Fe0.116 Ni0.116",null,null,"other",null,null,5.3,868,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.581 Cr0.512 Fe0.116 Ni0.116",null,null,"other",null,null,5.3,865,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.457 Cr0.391 Cu0.109 Fe0.109 Ni0.109",null,null,"other",null,null,5.2,840,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.512 Cr0.512 Fe0.186 Ni0.116",null,null,"other",null,null,5.3,882,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.512 Cr0.535 Fe0.163 Ni0.116","BCC+B2",null,"other",null,null,5.3,883,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.465 Cr0.488 Fe0.209 Ni0.163",null,null,"other",null,null,5.3,828,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.512 Cr0.581 Fe0.116 Ni0.116",null,null,"other",null,null,5.2,848,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.404 Cr0.404 Cu0.106 Fe0.106 Ni0.106",null,null,"other",null,null,5.1,878,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[163,"Al1 Co0.674 Cr0.419 Fe0.116 Ni0.116",null,null,"other",null,null,5.3,782,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.03.010",2019,"Machine learning assisted design of high entropy alloys with desired property"],[164,"Co0.986 Cr0.952 Fe0.933 Mn0.914 Ni1","BCC","WROUGHT","BCC",null,null,8,null,"T",25,1290,1415,1.1,null,null,220,null,null,null,"10.1016/j.msea.2019.01.112",2019,"Effect of second phase particles on mechanical properties and grain growth in a CoCrFeMnNi high entropy alloy"],[164,"Co0.986 Cr0.952 Fe0.933 Mn0.914 Ni1","FCC+BCC+Sec.","WROUGHT","other",1.3,null,8,null,"T",25,1135,1260,1.8,null,null,null,null,null,null,"10.1016/j.msea.2019.01.112",2019,"Effect of second phase particles on mechanical properties and grain growth in a CoCrFeMnNi high entropy alloy"],[164,"Co0.986 Cr0.952 Fe0.933 Mn0.914 Ni1","FCC+BCC+Sec.","WROUGHT","other",2,null,8,null,"T",25,585,815,31.7,null,null,null,null,null,null,"10.1016/j.msea.2019.01.112",2019,"Effect of second phase particles on mechanical properties and grain growth in a CoCrFeMnNi high entropy alloy"],[164,"Co0.986 Cr0.952 Fe0.933 Mn0.914 Ni1","FCC+Sec.","WROUGHT","other",3.6,null,8,null,"T",25,425,735,40.2,null,null,null,null,null,null,"10.1016/j.msea.2019.01.112",2019,"Effect of second phase particles on mechanical properties and grain growth in a CoCrFeMnNi high entropy alloy"],[164,"Co0.986 Cr0.952 Fe0.933 Mn0.914 Ni1","FCC","WROUGHT","FCC",17.1,null,8,null,"T",25,270,630,46.2,null,null,220,null,null,null,"10.1016/j.msea.2019.01.112",2019,"Effect of second phase particles on mechanical properties and grain growth in a CoCrFeMnNi high entropy alloy"],[164,"Co0.986 Cr0.952 Fe0.933 Mn0.914 Ni1",null,"WROUGHT","other",0.07,null,8,null,"T",25,1120,1175,1.5,null,null,null,null,null,null,"10.1016/j.msea.2019.01.112",2019,"Effect of second phase particles on mechanical properties and grain growth in a CoCrFeMnNi high entropy alloy"],[165,"Al0.4 Hf0.6 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,9.08,9.1,null,"C",23,1841,null,null,null,null,107,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,5.98,6,null,"C",23,1100,null,null,null,null,189,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,6.46,6.5,null,"C",23,1100,null,null,null,null,151,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Ti1 V0.5","BCC","CAST","BCC",null,8.96,9,null,"C",23,1260,null,null,null,null,126,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,9.93,9.9,null,"C",23,1512,null,null,null,null,137,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,8.68,8.7,null,"C",23,1575,null,null,null,null,128,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,10.18,10.2,null,"C",23,1600,null,null,null,null,143,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,9.89,9.9,null,"C",23,929,null,null,null,null,107,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,10.91,11,null,"C",23,1515,null,null,null,null,209,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,11.72,11.8,null,"C",23,1343,null,null,null,null,222,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 V1 W1","BCC","OTHER","BCC",null,12.29,12.4,null,"C",23,1246,null,null,null,null,229,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 W1","BCC","OTHER","BCC",null,13.6,13.7,null,"C",23,996,null,null,null,null,250,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ti1","BCC","OTHER","BCC",null,7.67,7.7,null,"C",23,1100,null,null,null,null,177,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ta0.3 Ti1 Zr1","BCC","OTHER","BCC",null,7.81,7.4,null,"C",23,882,null,null,null,null,101,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ti1 Zr1","BCC","OTHER","BCC",null,6.63,6.5,null,"C",23,975,null,null,null,null,94,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Cr1 Mo1 Nb1 Ti1","BCC+Laves","ANNEAL","other",null,6.56,6.6,null,"C",23,1010,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al0.24 Nb1 Ti1 V1 Zr1","BCC+Laves","OTHER","other",null,6.34,6.3,null,"C",23,1240,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","B2+BCC","OTHER","other",null,7.14,7.1,null,"C",23,2000,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Cr1 Nb0.333 Ti0.333 Zr0.333","BCC+Laves","OTHER","other",null,6.56,6.8,null,"C",23,1576,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Cr1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+BCC+Laves","OTHER","other",null,7.98,8,null,"C",23,1595,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.3 Ti1 V0.5","BCC+Sec.","CAST","other",null,8.46,8.5,null,"C",23,1617,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.5 Ti1 V0.5","BCC+Sec.","CAST","other",null,8.16,8.2,null,"C",23,1787,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.7 Ti1 V0.5","BCC+Sec.","CAST","other",null,7.89,7.9,null,"C",23,2134,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ti1 V1 Zr1","BCC+Laves","OTHER","other",null,7.27,7.1,null,"C",23,1706,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Re0.3 Ti1 Zr1","BCC+Laves","OTHER","other",null,7.81,7.5,null,"C",23,1244,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ta1 Ti1 V1 Zr1","BCC+BCC","OTHER","other",null,8.43,8.5,null,"C",23,1395,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al0.4 Hf0.6 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,9.08,9.1,null,"C",1000,298,null,null,null,null,107,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,5.98,6,null,"C",1000,375,null,null,null,null,189,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,6.46,6.5,null,"C",1000,540,null,null,null,null,151,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Ti1 V0.5","BCC","CAST","BCC",null,8.96,9,null,"C",1000,368,null,null,null,null,126,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,9.93,9.9,null,"C",1000,814,null,null,null,null,137,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,8.68,8.7,null,"C",1000,635,null,null,null,null,128,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,10.18,10.2,null,"C",1000,855,null,null,null,null,143,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,9.89,9.9,null,"C",1000,295,null,null,null,null,107,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,10.91,11,null,"C",1000,753,null,null,null,null,209,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,11.72,11.8,null,"C",1000,620,null,null,null,null,222,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 V1 W1","BCC","OTHER","BCC",null,12.29,12.4,null,"C",1000,842,null,null,null,null,229,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 W1","BCC","OTHER","BCC",null,13.6,13.7,null,"C",1000,548,null,null,null,null,250,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ti1","BCC","OTHER","BCC",null,7.67,7.7,null,"C",1000,504,null,null,null,null,177,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ta0.3 Ti1 Zr1","BCC","OTHER","BCC",null,7.81,7.4,null,"C",1000,274,null,null,null,null,101,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ti1 Zr1","BCC","OTHER","BCC",null,6.63,6.5,null,"C",1000,141,null,null,null,null,94,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Cr1 Mo1 Nb1 Ti1","BCC+Laves","ANNEAL","other",null,6.56,6.6,null,"C",1000,550,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al0.24 Nb1 Ti1 V1 Zr1","BCC+Laves","OTHER","other",null,6.34,6.3,null,"C",1000,82,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","B2+BCC","OTHER","other",null,7.14,7.1,null,"C",1000,745,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Cr1 Nb0.333 Ti0.333 Zr0.333","BCC+Laves","OTHER","other",null,6.56,6.8,null,"C",1000,139,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Cr1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+BCC+Laves","OTHER","other",null,7.98,8,null,"C",1000,546,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.3 Ti1 V0.5","BCC+Sec.","CAST","other",null,8.46,8.5,null,"C",1000,398,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.5 Ti1 V0.5","BCC+Sec.","CAST","other",null,8.16,8.2,null,"C",1000,614,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.7 Ti1 V0.5","BCC+Sec.","CAST","other",null,7.89,7.9,null,"C",1000,673,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ti1 V1 Zr1","BCC+Laves","OTHER","other",null,7.27,7.1,null,"C",1000,467,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Re0.3 Ti1 Zr1","BCC+Laves","OTHER","other",null,7.81,7.5,null,"C",1000,323,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ta1 Ti1 V1 Zr1","BCC+BCC","OTHER","other",null,8.43,8.5,null,"C",1000,190,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al0.4 Hf0.6 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,9.08,9.1,null,"C",1200,89,null,null,null,null,107,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Cr1 Mo1 Ti1","BCC","ANNEAL","BCC",null,5.98,6,null,"C",1200,100,null,null,null,null,189,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,6.46,6.5,null,"C",1200,200,null,null,null,null,151,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Ti1 V0.5","BCC","CAST","BCC",null,8.96,9,null,"C",1200,60,null,null,null,null,126,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,9.93,9.9,null,"C",1200,556,null,null,null,null,137,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,8.68,8.7,null,"C",1200,187,null,null,null,null,128,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,10.18,10.2,null,"C",1200,404,null,null,null,null,143,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,9.89,9.9,null,"C",1200,92,null,null,null,null,107,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 Ti1 V1 W1","BCC","CAST","BCC",null,10.91,11,null,"C",1200,659,null,null,null,null,209,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,11.72,11.8,null,"C",1200,586,null,null,null,null,222,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 V1 W1","BCC","OTHER","BCC",null,12.29,12.4,null,"C",1200,735,null,null,null,null,229,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ta1 W1","BCC","OTHER","BCC",null,13.6,13.7,null,"C",1200,506,null,null,null,null,250,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ti1","BCC","OTHER","BCC",null,7.67,7.7,null,"C",1200,324,null,null,null,null,177,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ta0.3 Ti1 Zr1","BCC","OTHER","BCC",null,7.81,7.4,null,"C",1200,102,null,null,null,null,101,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ti1 Zr1","BCC","OTHER","BCC",null,6.63,6.5,null,"C",1200,61,null,null,null,null,94,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Cr1 Mo1 Nb1 Ti1","BCC+Laves","ANNEAL","other",null,6.56,6.6,null,"C",1200,105,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al0.24 Nb1 Ti1 V1 Zr1","BCC+Laves","OTHER","other",null,6.34,6.3,null,"C",1200,24,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Al1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","B2+BCC","OTHER","other",null,7.14,7.1,null,"C",1200,250,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Cr1 Nb0.333 Ti0.333 Zr0.333","BCC+Laves","OTHER","other",null,6.56,6.8,null,"C",1200,32,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Cr1 Mo0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+BCC+Laves","OTHER","other",null,7.98,8,null,"C",1200,170,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.3 Ti1 V0.5","BCC+Sec.","CAST","other",null,8.46,8.5,null,"C",1200,166,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.5 Ti1 V0.5","BCC+Sec.","CAST","other",null,8.16,8.2,null,"C",1200,188,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Hf1 Mo0.5 Nb1 Si0.7 Ti1 V0.5","BCC+Sec.","CAST","other",null,7.89,7.9,null,"C",1200,235,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Mo1 Nb1 Ti1 V1 Zr1","BCC+Laves","OTHER","other",null,7.27,7.1,null,"C",1200,116,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Re0.3 Ti1 Zr1","BCC+Laves","OTHER","other",null,7.81,7.5,null,"C",1200,89,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[165,"Nb1 Ta1 Ti1 V1 Zr1","BCC+BCC","OTHER","other",null,8.43,8.5,null,"C",1200,78,null,null,null,null,null,null,null,null,"10.1016/j.actamat.2019.06.032",2019,"High temperature strength of refractory complex concentrated alloys"],[166,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",245.48,7.753,8,null,"T",25,293.1,625.6,63,null,null,220,null,null,null,"10.1016/j.intermet.2019.106486",2019,"High-cycle fatigue and tensile deformation behaviors of coarse-grained equiatomic CoCrFeMnNi high entropy alloy and unexpected hardening behavior during cyclic loading"],[203,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",40,null,9.9,null,"T",25,985,1004,15.7,null,null,107,null,null,null,"10.1016/j.jallcom.2019.04.291",2019,"Grain growth and Hall-Petch relationship in a refractory HfNbTaZrTi high-entropy alloy"],[203,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",87,null,9.9,null,"T",25,972,999.5,12.6,null,null,107,null,null,null,"10.1016/j.jallcom.2019.04.291",2019,"Grain growth and Hall-Petch relationship in a refractory HfNbTaZrTi high-entropy alloy"],[203,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",112,null,9.9,null,"T",25,967,980.6,10.8,null,null,107,null,null,null,"10.1016/j.jallcom.2019.04.291",2019,"Grain growth and Hall-Petch relationship in a refractory HfNbTaZrTi high-entropy alloy"],[204,"Co1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,8.5,null,"T",25,null,null,15,null,null,207,null,null,null,"10.1016/j.intermet.2019.106477",2019,"Effects of heat treatment on the microstructure and properties of cold-forged CoNiFe medium entropy alloy"],[204,"Co1 Fe1 Ni1","FCC","WROUGHT","FCC",42,null,8.5,null,"T",25,276,540,51,null,null,207,null,null,null,"10.1016/j.intermet.2019.106477",2019,"Effects of heat treatment on the microstructure and properties of cold-forged CoNiFe medium entropy alloy"],[204,"Co1 Fe1 Ni1","FCC","WROUGHT","FCC",74,null,8.5,null,"T",25,217,498,47,null,null,207,null,null,null,"10.1016/j.intermet.2019.106477",2019,"Effects of heat treatment on the microstructure and properties of cold-forged CoNiFe medium entropy alloy"],[204,"Co1 Fe1 Ni1","FCC","WROUGHT","FCC",90,null,8.5,null,"T",25,201,430,44.5,null,null,207,null,null,null,"10.1016/j.intermet.2019.106477",2019,"Effects of heat treatment on the microstructure and properties of cold-forged CoNiFe medium entropy alloy"],[205,"Co1 Cr1 Ni1",null,"ANNEAL","other",16,null,8.3,null,"T",-196,600,1247,68,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.002",2019,"Superb cryogenic strength of equiatomic CrCoNi derived from gradient hierarchical microstructure"],[205,"Co1 Cr1 Ni1",null,"ANNEAL","other",16,null,8.3,null,"T",24,400,800,60,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.002",2019,"Superb cryogenic strength of equiatomic CrCoNi derived from gradient hierarchical microstructure"],[205,"Co1 Cr1 Ni1",null,"ANNEAL","other",16,null,8.3,null,"T",-196,1215,1440,48,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.002",2019,"Superb cryogenic strength of equiatomic CrCoNi derived from gradient hierarchical microstructure"],[206,"Cr0.286 Mo0.476 Nb1 Ta0.024 Ti0.31 V0.286","BCC+Sec.","POWDER","other",0.075,8.02,8,741,"C",25,2680,null,null,null,null,null,null,null,null,"10.1016/j.ijrmhm.2019.02.009",2019,"Fabrication, microstructure and mechanical property of a novel Nb-rich refractory high-entropy alloy strengthened by in-situ formation of dispersoids"],[207,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,8.68,8.7,null,"C",25,1719,null,null,null,null,128,null,null,null,"10.1016/j.intermet.2015.10.011",2016,"Effect of composing element on microstructure and mechanical properties in Mo–Nb–Hf–Zr–Ti multi-principle component alloys"],[208,"Co1 Fe1 Ni1","FCC","WROUGHT","FCC",53.7,null,8.5,null,"T",25,null,null,null,null,161,207,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr0.053 Ni1","FCC","WROUGHT","FCC",7.28,null,8.8,null,"T",25,null,null,null,null,219,207,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",0.862,null,8.1,null,"T",25,529,613,25,null,182,203,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",0.894,null,8.1,null,"T",25,522,601,25,null,182,203,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",1.25,null,8.1,null,"T",25,460,599,27,null,182,203,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",1.75,null,8.1,null,"T",25,407,536,26,null,182,203,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",2.26,null,8.1,null,"T",25,372,542,31,null,182,203,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",3.52,null,8.1,null,"T",25,322,554,49,null,182,203,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Mn1 Ni1","FCC","WROUGHT","FCC",1.43,null,8.4,null,"T",25,333,569,40,null,190,202,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Mn1 Ni1","FCC","WROUGHT","FCC",0.4,null,8.4,null,"T",25,601,608,24,null,190,202,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Mn1 Ni1","FCC","WROUGHT","FCC",1.58,null,8.4,null,"T",25,321,566,50.5,null,190,202,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Mn1 Ni1","FCC","WROUGHT","FCC",2.67,null,8.4,null,"T",25,267,506,62.5,null,190,202,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr0.25 Ni1","FCC","WROUGHT","FCC",0.212,null,8.7,null,"T",25,745,824,36,null,223,213,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr0.25 Ni1","FCC","WROUGHT","FCC",0.34,null,8.7,null,"T",25,623,746,53,null,223,213,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr0.25 Ni1","FCC","WROUGHT","FCC",1.46,null,8.7,null,"T",25,387,626,73,null,223,213,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr0.25 Ni1","FCC","WROUGHT","FCC",2.65,null,8.7,null,"T",25,330,553,101,null,223,213,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",0.295,null,8.2,null,"T",25,631,732,39,null,210,226,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",0.552999999999999,null,8.2,null,"T",25,494,732,51,null,210,226,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",0.72,null,8.2,null,"T",25,448,672,65,null,210,226,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",3.4,null,8.2,null,"T",25,273,567,85,null,210,226,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co1 Cr1 Fe1 Ni1","FCC","WROUGHT","FCC",32.2,null,8.2,null,"T",25,172,504,105,null,210,226,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co0.25 Cr1 Ni1","FCC","WROUGHT","FCC",0.497,null,8.1,null,"T",25,518,744,36,null,240,238,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co0.25 Cr1 Ni1","FCC","WROUGHT","FCC",0.848,null,8.1,null,"T",25,462,703,46,null,240,238,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co0.25 Cr1 Ni1","FCC","WROUGHT","FCC",1.78,null,8.1,null,"T",25,395,644,58,null,240,238,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[208,"Co0.25 Cr1 Ni1","FCC","WROUGHT","FCC",1.98,null,8.1,null,"T",25,356,588,64,null,240,238,null,null,null,"10.1016/j.actamat.2019.04.017",2019,"Effect of elemental combination on friction stress and Hall-Petch relationship in face-centered cubic high / medium entropy alloys"],[209,"Al1 Co1 Ni1 Ti1","BCC+B2","POWDER","other",null,null,5.7,672,"C",25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.02.083",2019,"Microstructure and mechanical properties of Ti-Al-Ni-Co-Fe based high entropy alloys prepared by powder metallurgy route"],[209,"Al1 Fe1 Ni1 Ti1","BCC+B2","POWDER","other",null,null,5.5,666,"C",25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.02.083",2019,"Microstructure and mechanical properties of Ti-Al-Ni-Co-Fe based high entropy alloys prepared by powder metallurgy route"],[209,"Al1 Co1 Fe1 Ni1 Ti1","BCC+B2","POWDER","other",null,null,6.1,667,"C",25,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.02.083",2019,"Microstructure and mechanical properties of Ti-Al-Ni-Co-Fe based high entropy alloys prepared by powder metallurgy route"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,434,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,666,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,761,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,747,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,null,"C",20,1511,null,5,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,null,"C",400,1366,null,17,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,null,"C",600,410,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[210,"Al1 Cr1 Cu1 Fe1 Ni1","FCC+BCC+B2","POWDER","other",null,null,6.8,null,"C",800,48,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.01.341",2019,"Structure formation and mechanical properties of the high-entropy AlCuNiFeCr alloy prepared by mechanical alloying and spark plasma sintering"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8,null,"T",25,210,null,57,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,1150,null,6,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,1300,null,2,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",4,null,8,null,"T",25,1050,null,8,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",8,null,8,null,"T",25,560,null,49,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",8,null,8,null,"T",25,520,null,47,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",10,null,8,null,"T",25,320,null,59,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[211,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",20,null,8,null,"T",25,280,null,62,null,null,220,null,null,null,"10.1016/j.msea.2019.04.025",2019,"Regulating the strength and ductility of a cold rolled FeCrCoMnNi high-entropy alloy via annealing treatment"],[212,"Co0.5 Fe0.5 Mo0.1 Ni1 V0.25","FCC","CAST","FCC",null,null,8.4,null,"T",-196,419,null,null,34.6,null,202,null,null,null,"10.1016/j.scriptamat.2019.02.038",2019,"A promising CoFeNi2V0.5Mo0.2 high entropy alloy with exceptional ductility"],[212,"Co0.5 Fe0.5 Mo0.1 Ni1 V0.25","FCC","CAST","FCC",null,null,8.4,null,"T",-70,350,null,null,28.8,null,202,null,null,null,"10.1016/j.scriptamat.2019.02.038",2019,"A promising CoFeNi2V0.5Mo0.2 high entropy alloy with exceptional ductility"],[212,"Co0.5 Fe0.5 Mo0.1 Ni1 V0.25","FCC","CAST","FCC",null,null,8.4,null,"T",20,252,null,null,82.1,null,202,null,null,null,"10.1016/j.scriptamat.2019.02.038",2019,"A promising CoFeNi2V0.5Mo0.2 high entropy alloy with exceptional ductility"],[212,"Co0.5 Fe0.5 Mo0.1 Ni1 V0.25","FCC","CAST","FCC",null,null,8.4,null,"T",600,214,null,null,189.2,null,202,null,null,null,"10.1016/j.scriptamat.2019.02.038",2019,"A promising CoFeNi2V0.5Mo0.2 high entropy alloy with exceptional ductility"],[212,"Co0.5 Fe0.5 Mo0.1 Ni1 V0.25","FCC","CAST","FCC",null,null,8.4,null,"T",700,212,null,null,154.3,null,202,null,null,null,"10.1016/j.scriptamat.2019.02.038",2019,"A promising CoFeNi2V0.5Mo0.2 high entropy alloy with exceptional ductility"],[212,"Co0.5 Fe0.5 Mo0.1 Ni1 V0.25","FCC","CAST","FCC",null,null,8.4,null,"T",800,187,null,null,119.5,null,202,null,null,null,"10.1016/j.scriptamat.2019.02.038",2019,"A promising CoFeNi2V0.5Mo0.2 high entropy alloy with exceptional ductility"],[213,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,132,"C",300,null,null,null,null,null,210,null,null,null,"10.1088/2053-1591/ab0597",2019,"Effect of cooling rate on microstructure and mechanical properties of Al0.3CoCrFeNi high-entropy alloy"],[213,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,147,"C",340,null,null,null,null,null,210,null,null,null,"10.1088/2053-1591/ab0597",2019,"Effect of cooling rate on microstructure and mechanical properties of Al0.3CoCrFeNi high-entropy alloy"],[213,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.6,165,"C",430,null,null,null,null,null,210,null,null,null,"10.1088/2053-1591/ab0597",2019,"Effect of cooling rate on microstructure and mechanical properties of Al0.3CoCrFeNi high-entropy alloy"],[215,"Al1 Co1 Cr1 Fe1 Ni1","B2+BCC","CAST","other",204,null,6.7,519,"C",25,1631,null,16,null,null,null,null,null,null,"10.1016/j.matchar.2019.03.006",2019,"The strengthening effects of relatively lightweight AlCoCrFeNi high entropy alloy"],[215,"Al1 Co1 Cr1 Fe1 Ni1","B2+BCC","CAST","other",185,null,6.7,493,"C",25,1209,null,23,null,null,null,null,null,null,"10.1016/j.matchar.2019.03.006",2019,"The strengthening effects of relatively lightweight AlCoCrFeNi high entropy alloy"],[216,"Co0.318 Cr0.227 Fe1 Mo0.091 Ni0.182","FCC+Sec.","WROUGHT","other",4.4,null,8.2,null,"T",25,443,840,61.6,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[216,"Co0.327 Cr0.227 Fe1 Mo0.136 Ni0.127","FCC+Sec.","WROUGHT","other",3.2,null,8.2,null,"T",25,711,1096,38.3,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[216,"Co0.318 Cr0.227 Fe1 Mo0.091 Ni0.182","FCC+Sec.","WROUGHT","other",26.3,null,8.2,null,"T",25,301,718,77.5,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[216,"Co0.327 Cr0.227 Fe1 Mo0.136 Ni0.127","FCC+Sec.","WROUGHT","other",4.8,null,8.2,null,"T",25,684,1078,48.1,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[216,"Co0.318 Cr0.227 Fe1 Mo0.091 Ni0.182","FCC+Sec.","WROUGHT","other",70.1,null,8.2,null,"T",25,261,673,86.6,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[216,"Co0.327 Cr0.227 Fe1 Mo0.136 Ni0.127","FCC+Sec.","WROUGHT","other",5.4,null,8.2,null,"T",25,506,933,59.3,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[216,"Co0.318 Cr0.227 Fe1 Mo0.091 Ni0.182","FCC+Sec.","WROUGHT","other",136.5,null,8.2,null,"T",25,231,638,87.2,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[216,"Co0.327 Cr0.227 Fe1 Mo0.136 Ni0.127","FCC+Sec.","WROUGHT","other",120.6,null,8.2,null,"T",25,296,693,63.2,null,null,null,null,null,null,"10.1016/j.jallcom.2018.12.040",2019,"Effect of μ-precipitates on the microstructure and mechanical properties of non-equiatomic CoCrFeNiMo medium-entropy alloys"],[217,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","ANNEAL","FCC",300,null,8,null,null,null,null,null,null,null,null,220,null,null,null,"10.1016/j.jallcom.2018.11.181",2019,"Microstructure and mechanical properties of a CoCrFeMnNi high entropy alloy processed by milling and spark plasma sintering"],[217,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","POWDER","FCC",4.2,null,8,null,null,null,null,null,null,null,null,220,null,null,null,"10.1016/j.jallcom.2018.11.181",2019,"Microstructure and mechanical properties of a CoCrFeMnNi high entropy alloy processed by milling and spark plasma sintering"],[217,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","POWDER","FCC",1.3,null,8,null,null,null,null,null,null,null,null,220,null,null,null,"10.1016/j.jallcom.2018.11.181",2019,"Microstructure and mechanical properties of a CoCrFeMnNi high entropy alloy processed by milling and spark plasma sintering"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC","CAST","other",null,null,8.1,null,"T",25,615,950,14,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC+Sec.","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC+Sec.","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC+Sec.","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC+Sec.","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[218,"Cr1 Cu1 Fe1 Mn1 Ni1","FCC+BCC","ANNEAL","other",null,null,8.1,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11665-019-04003-4",2019,"Microstructure and Mechanical Properties of a Multiphase FeCrCuMnNi High-Entropy Alloy"],[219,"Mo1 Nb1 Re0.5 W1","BCC","CAST","BCC",null,null,13.7,473,"C",25,896,null,10.4,7.08,null,298,null,null,null,"10.1016/j.jallcom.2018.11.111",2019,"Microstructure evolution, mechanical properties and strengthening mechanism of refractory high-entropy alloy matrix composites with addition of TaC"],[219,"C0.2 Mo1 Nb1 Re0.5 Ta0.2 W1","BCC+Sec.","CAST","other",null,null,13.5,510,"C",25,1074,null,12.1,8.33,null,null,null,null,null,"10.1016/j.jallcom.2018.11.111",2019,"Microstructure evolution, mechanical properties and strengthening mechanism of refractory high-entropy alloy matrix composites with addition of TaC"],[219,"C0.4 Mo1 Nb1 Re0.5 Ta0.4 W1","BCC+Sec.","CAST","other",null,null,13.4,558,"C",25,1144,null,12.7,8.81,null,null,null,null,null,"10.1016/j.jallcom.2018.11.111",2019,"Microstructure evolution, mechanical properties and strengthening mechanism of refractory high-entropy alloy matrix composites with addition of TaC"],[219,"C0.5 Mo1 Nb1 Re0.5 Ta0.5 W1","BCC+Sec.","CAST","other",null,null,13.3,583,"C",25,1202,null,14.2,10.25,null,null,null,null,null,"10.1016/j.jallcom.2018.11.111",2019,"Microstructure evolution, mechanical properties and strengthening mechanism of refractory high-entropy alloy matrix composites with addition of TaC"],[219,"C0.6 Mo1 Nb1 Re0.5 Ta0.6 W1","BCC+Sec.","CAST","other",null,null,13.3,615,"C",25,1241,null,13.8,9.64,null,null,null,null,null,"10.1016/j.jallcom.2018.11.111",2019,"Microstructure evolution, mechanical properties and strengthening mechanism of refractory high-entropy alloy matrix composites with addition of TaC"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",23,520,696,49,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",600,470,561,11,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",700,461,560,9,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",800,445,536,6,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",900,219,312,8,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",23,627,1197,27,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",600,594,1006,12,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",700,547,840,17,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",800,399,575,20,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[220,"Al0.278 Co0.694 Cr0.222 Fe0.417 Ni1 Ti0.167","FCC","ANNEAL","FCC",null,null,7.4,null,"T",900,243,319,34,null,null,185,null,null,null,"10.3390/e21020169",2019,"Microstructure and Mechanical Properties of Precipitate Strengthened High Entropy Alloy Al10Co25Cr8Fe15Ni36Ti6 with Additions of Hafnium and Molybdenum"],[221,"Co0.286 Cr0.429 Fe1 Mn0.143 Ni0.714 V0.286","FCC","WROUGHT","FCC",8.7,null,7.9,null,"T",25,401,717,49,null,null,208,null,null,null,"10.1016/j.msea.2018.12.077",2019,"Effect of grain size on the tensile behavior of V10Cr15Mn5Fe35Co10Ni25 high entropy alloy"],[221,"Co0.286 Cr0.429 Fe1 Mn0.143 Ni0.714 V0.286","FCC","WROUGHT","FCC",10.4,null,7.9,null,"T",25,350,690,50,null,null,208,null,null,null,"10.1016/j.msea.2018.12.077",2019,"Effect of grain size on the tensile behavior of V10Cr15Mn5Fe35Co10Ni25 high entropy alloy"],[221,"Co0.286 Cr0.429 Fe1 Mn0.143 Ni0.714 V0.286","FCC","WROUGHT","FCC",11.2,null,7.9,null,"T",25,312,660,51,null,null,208,null,null,null,"10.1016/j.msea.2018.12.077",2019,"Effect of grain size on the tensile behavior of V10Cr15Mn5Fe35Co10Ni25 high entropy alloy"],[221,"Co0.286 Cr0.429 Fe1 Mn0.143 Ni0.714 V0.286","FCC","WROUGHT","FCC",48.2,null,7.9,null,"T",25,240,610,53,null,null,208,null,null,null,"10.1016/j.msea.2018.12.077",2019,"Effect of grain size on the tensile behavior of V10Cr15Mn5Fe35Co10Ni25 high entropy alloy"],[221,"Co0.286 Cr0.429 Fe1 Mn0.143 Ni0.714 V0.286","FCC","WROUGHT","FCC",82.7,null,7.9,null,"T",25,210,593,52,null,null,208,null,null,null,"10.1016/j.msea.2018.12.077",2019,"Effect of grain size on the tensile behavior of V10Cr15Mn5Fe35Co10Ni25 high entropy alloy"],[223,"Co1 Cr1 Fe1 Mn1 Ni1","Other","POWDER","other",null,7.5,8,410.9,"T",25,null,null,null,null,184,null,null,null,null,"10.1016/j.jallcom.2019.07.195",2019,"Structure and properties of equiatomic CoCrFeNiMn alloy fabricated by high-energy ball milling and spark plasma sintering"],[224,"Al1 Co0.5 Cr0.5 Fe0.5 Ni0.5","BCC+B2","CAST","other",80.67,null,5.9,581.7,"C",25,1241,null,9.98,null,null,null,null,null,null,"10.1016/j.jallcom.2019.07.100",2019,"Effects of Ti-to-Al ratios on the phases, microstructures, mechanical properties, and corrosion resistance of Al2-xCoCrFeNiTix high-entropy alloys"],[224,"Al1 Co0.556 Cr0.556 Fe0.556 Ni0.556 Ti0.111","BCC+B2","CAST","other",66.93,null,5.9,605.5,"C",25,1324,null,10.02,null,null,null,null,null,null,"10.1016/j.jallcom.2019.07.100",2019,"Effects of Ti-to-Al ratios on the phases, microstructures, mechanical properties, and corrosion resistance of Al2-xCoCrFeNiTix high-entropy alloys"],[224,"Al1 Co0.667 Cr0.667 Fe0.667 Ni0.667 Ti0.333","BCC+B2","CAST","other",56.27,null,6.1,641.7,"C",25,1566,null,11.85,null,null,null,null,null,null,"10.1016/j.jallcom.2019.07.100",2019,"Effects of Ti-to-Al ratios on the phases, microstructures, mechanical properties, and corrosion resistance of Al2-xCoCrFeNiTix high-entropy alloys"],[224,"Al1 Co0.833 Cr0.833 Fe0.833 Ni0.833 Ti0.667","BCC+B2","CAST","other",49.95,null,6.2,639.6,"C",25,1503,null,7.73,null,null,null,null,null,null,"10.1016/j.jallcom.2019.07.100",2019,"Effects of Ti-to-Al ratios on the phases, microstructures, mechanical properties, and corrosion resistance of Al2-xCoCrFeNiTix high-entropy alloys"],[224,"Al1 Co1 Cr1 Fe1 Ni1 Ti1","BCC+B2+Laves","CAST","other",30.49,null,6.2,703,"C",25,1448,null,6.18,null,null,null,null,null,null,"10.1016/j.jallcom.2019.07.100",2019,"Effects of Ti-to-Al ratios on the phases, microstructures, mechanical properties, and corrosion resistance of Al2-xCoCrFeNiTix high-entropy alloys"],[224,"Al0.667 Co0.833 Cr0.833 Fe0.833 Ni0.833 Ti1","BCC+B2+Laves","CAST","other",22.03,null,6.3,856.9,"C",25,1272,null,3.15,null,null,null,null,null,null,"10.1016/j.jallcom.2019.07.100",2019,"Effects of Ti-to-Al ratios on the phases, microstructures, mechanical properties, and corrosion resistance of Al2-xCoCrFeNiTix high-entropy alloys"],[226,"Al1 Co1 Cr1 Fe1 Ni1",null,null,"other",null,null,6.7,null,"C",25,1300,null,24.2,null,null,null,null,null,null,"10.1016/j.jallcom.2018.01.247",2018,"Studies on the microstructure and properties of AlxCoCrFeNiTi1-x high entropy alloys"],[226,"Al0.8 Co1 Cr1 Fe1 Ni1 Ti0.2",null,null,"other",null,null,6.8,null,"C",25,968,null,30.1,null,null,null,null,null,null,"10.1016/j.jallcom.2018.01.247",2018,"Studies on the microstructure and properties of AlxCoCrFeNiTi1-x high entropy alloys"],[226,"Al0.5 Co1 Cr1 Fe1 Ni1 Ti0.5",null,null,"other",null,null,6.9,null,"C",25,1100,null,27.5,null,null,null,null,null,null,"10.1016/j.jallcom.2018.01.247",2018,"Studies on the microstructure and properties of AlxCoCrFeNiTi1-x high entropy alloys"],[225,"Al1 Co1 Cu1 Fe1 Ni1",null,null,"other",null,null,7,null,"C",25,1060,null,19.1,null,null,null,null,null,null,"10.1016/j.matdes.2016.12.036",2017,"Microstructure, mechanical and corrosion behaviors of AlCoCuFeNi-(Cr,Ti) high entropy alloys"],[225,"Al1 Co1 Cr1 Cu1 Fe1 Ni1",null,null,"other",null,null,7.1,null,"C",25,1285,null,24.6,null,null,null,null,null,null,"10.1016/j.matdes.2016.12.036",2017,"Microstructure, mechanical and corrosion behaviors of AlCoCuFeNi-(Cr,Ti) high entropy alloys"],[225,"Al1 Co1 Cu1 Fe1 Ni1 Ti1",null,null,"other",null,null,6.5,null,"C",25,1612,null,12.8,null,null,null,null,null,null,"10.1016/j.matdes.2016.12.036",2017,"Microstructure, mechanical and corrosion behaviors of AlCoCuFeNi-(Cr,Ti) high entropy alloys"],[225,"Al1 Co1 Cr1 Cu1 Fe1 Ni1 Ti1",null,null,"other",null,null,6.6,null,"C",25,1523,null,8.1,null,null,null,null,null,null,"10.1016/j.matdes.2016.12.036",2017,"Microstructure, mechanical and corrosion behaviors of AlCoCuFeNi-(Cr,Ti) high entropy alloys"],[227,"Al1 Cr1 Fe1 Ni1",null,null,"other",null,null,6.3,null,"C",25,1271,null,14,null,null,null,null,null,null,"10.1016/j.jallcom.2018.07.346",2018,"Effect of Co content on the mechanical properties of A2 and B2 phases in AlCoxCrFeNi high-entropy alloys"],[227,"Al1 Co0.25 Cr1 Fe1 Ni1",null,null,"other",null,null,6.4,null,"C",25,1363,null,19,null,null,null,null,null,null,"10.1016/j.jallcom.2018.07.346",2018,"Effect of Co content on the mechanical properties of A2 and B2 phases in AlCoxCrFeNi high-entropy alloys"],[227,"Al1 Co0.5 Cr1 Fe1 Ni1",null,null,"other",null,null,6.5,null,"C",25,1349,null,20,null,null,null,null,null,null,"10.1016/j.jallcom.2018.07.346",2018,"Effect of Co content on the mechanical properties of A2 and B2 phases in AlCoxCrFeNi high-entropy alloys"],[227,"Al1 Co0.75 Cr1 Fe1 Ni1",null,null,"other",null,null,6.6,null,"C",25,1331,null,20,null,null,null,null,null,null,"10.1016/j.jallcom.2018.07.346",2018,"Effect of Co content on the mechanical properties of A2 and B2 phases in AlCoxCrFeNi high-entropy alloys"],[227,"Al1 Co1 Cr1 Fe1 Ni1",null,null,"other",null,null,6.7,null,"C",25,1313,null,19,null,null,null,null,null,null,"10.1016/j.jallcom.2018.07.346",2018,"Effect of Co content on the mechanical properties of A2 and B2 phases in AlCoxCrFeNi high-entropy alloys"],[228,"Co1 Cr1 Fe1 Mn1",null,null,"other",null,null,7.8,null,"C",25,338,null,4.3,null,null,null,null,null,null,"10.1016/j.intermet.2016.09.003",2016,"Compositional dependence of phase formation and mechanical properties in three CoCrFeNi-(Mn/Al/Cu) high entropy alloys"],[229,"Al1 Co1 Cr1 Fe1 Ni1","FCC+BCC","POWDER","other",null,null,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.301",2019,"Effect of heat treatment on the phase evolution and mechanical properties of atomized AlCoCrFeNi high-entropy alloy powders"],[229,"Al1 Co1 Cr1 Fe1 Ni1","FCC+BCC","POWDER","other",null,null,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.301",2019,"Effect of heat treatment on the phase evolution and mechanical properties of atomized AlCoCrFeNi high-entropy alloy powders"],[229,"Al1 Co1 Cr1 Fe1 Ni1","FCC+BCC","POWDER","other",null,null,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.301",2019,"Effect of heat treatment on the phase evolution and mechanical properties of atomized AlCoCrFeNi high-entropy alloy powders"],[229,"Al1 Co1 Cr1 Fe1 Ni1","FCC+BCC","POWDER","other",null,null,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.301",2019,"Effect of heat treatment on the phase evolution and mechanical properties of atomized AlCoCrFeNi high-entropy alloy powders"],[229,"Al1 Co1 Cr1 Fe1 Ni1","FCC+BCC","POWDER","other",null,null,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.301",2019,"Effect of heat treatment on the phase evolution and mechanical properties of atomized AlCoCrFeNi high-entropy alloy powders"],[230,"Hf1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",29.4,null,11.2,null,"C",25,1550,null,26.3,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[230,"Hf1 Mo1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",31.7,null,11.1,null,"C",25,1637,null,15.5,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[230,"Hf1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",null,null,11.2,null,"C",800,577,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[230,"Hf1 Mo1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",null,null,11.1,null,"C",800,1065,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[230,"Hf1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",null,null,11.2,null,"C",1000,409,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[230,"Hf1 Mo1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",null,null,11.1,null,"C",1000,736,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[230,"Hf1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",null,null,11.2,null,"C",1200,345,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[230,"Hf1 Mo1 Nb1 Ta1 Ti1 W1 Zr1","BCC+Laves","CAST","other",null,null,11.1,null,"C",1200,703,null,35,null,null,null,null,null,null,"10.1016/j.jallcom.2019.06.138",2019,"Microstructures and mechanical properties of HfNbTaTiZrW and HfNbTaTiZrMoW refractory high-entropy alloys"],[231,"Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,6.5,372,"C",25,1223,null,48,null,null,94,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[231,"Hf1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.4,351,"C",25,1000,null,52,null,null,89,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[231,"Nb1 Ti1 Zr1","BCC","ANNEAL","BCC",300,null,6.5,null,null,null,null,null,null,null,null,94,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[231,"Hf1 Nb1 Ti1 Zr1","BCC","ANNEAL","BCC",50,null,8.4,null,null,null,null,null,null,null,null,89,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[231,"Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,6.5,null,"C",800,462,null,51,null,null,94,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[231,"Hf1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.4,null,"C",800,303,null,51,null,null,89,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[231,"Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,6.5,null,"C",1000,218,null,51,null,null,94,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[231,"Hf1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.4,null,"C",1000,154,null,51,null,null,89,null,null,null,"10.1007/s12598-019-01310-6",2019,"Microstructure and mechanical properties of NbZrTi and NbHfZrTi alloys"],[232,"Co1 Cr1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.2,168.9,"C",25,163,null,80,null,null,226,null,null,null,"10.1016/j.intermet.2019.106550",2019,"Microstructure, mechanical properties and corrosion resistance of CoCrFeNiW(x = 0, 0.2, 0.5) high entropy alloys"],[232,"Co1 Cr1 Fe1 Ni1 W0.2","FCC","CAST","FCC",null,null,8.9,244,"C",25,335,null,52,null,null,238,null,null,null,"10.1016/j.intermet.2019.106550",2019,"Microstructure, mechanical properties and corrosion resistance of CoCrFeNiW(x = 0, 0.2, 0.5) high entropy alloys"],[232,"Co1 Cr1 Fe1 Ni1 W0.5","FCC+Sec.","CAST","other",null,null,9.8,357.9,"C",25,556,null,35.6,null,null,null,null,null,null,"10.1016/j.intermet.2019.106550",2019,"Microstructure, mechanical properties and corrosion resistance of CoCrFeNiW(x = 0, 0.2, 0.5) high entropy alloys"],[233,"Cr1 Fe1 Mo1 Nb1 V1","BCC+Laves","CAST","other",null,null,8.1,826,"C",25,2663,null,2.7,null,null,null,null,null,null,"10.1007/s40195-019-00935-x",2019,"Microstructure, Mechanical Properties, and Corrosion Behavior of MoNbFeCrV, MoNbFeCrTi, and MoNbFeVTi High-Entropy Alloys"],[233,"Cr1 Fe1 Mo1 Nb1 Ti1","BCC+Laves","CAST","other",null,null,7.6,792,"C",25,1647,null,0.9,null,null,null,null,null,null,"10.1007/s40195-019-00935-x",2019,"Microstructure, Mechanical Properties, and Corrosion Behavior of MoNbFeCrV, MoNbFeCrTi, and MoNbFeVTi High-Entropy Alloys"],[233,"Fe1 Mo1 Nb1 Ti1 V1","BCC+B2","CAST","other",null,null,7.4,699,"C",25,1707,null,1.7,null,null,null,null,null,null,"10.1007/s40195-019-00935-x",2019,"Microstructure, Mechanical Properties, and Corrosion Behavior of MoNbFeCrV, MoNbFeCrTi, and MoNbFeVTi High-Entropy Alloys"],[234,"Al0.15 Cr0.5 Cu0.25 Fe0.5 Ni1","FCC","WROUGHT","FCC",null,null,7.8,107,"T",25,62,169,98,null,16.6,200,null,null,null,"10.1016/j.matchar.2019.109792",2019,"Precipitation behavior and its impact on mechanical properties in an aged carbon-containing Al0.3Cu0.5CrFeNi2 high-entropy alloy"],[234,"Al0.15 Cr0.5 Cu0.25 Fe0.5 Ni1","FCC+L12","WROUGHT","other",null,null,7.8,209,"T",25,350,642,73,null,47,null,null,null,null,"10.1016/j.matchar.2019.109792",2019,"Precipitation behavior and its impact on mechanical properties in an aged carbon-containing Al0.3Cu0.5CrFeNi2 high-entropy alloy"],[234,"Al0.15 Cr0.5 Cu0.25 Fe0.5 Ni1","FCC+L12+Sec.","WROUGHT","other",null,null,7.8,321,"T",25,535,904,39,null,35.2,null,null,null,null,"10.1016/j.matchar.2019.109792",2019,"Precipitation behavior and its impact on mechanical properties in an aged carbon-containing Al0.3Cu0.5CrFeNi2 high-entropy alloy"],[234,"Al0.15 Cr0.5 Cu0.25 Fe0.5 Ni1","FCC+L12+Sec.","WROUGHT","other",null,null,7.8,159,"T",25,255,584,88,null,51.4,null,null,null,null,"10.1016/j.matchar.2019.109792",2019,"Precipitation behavior and its impact on mechanical properties in an aged carbon-containing Al0.3Cu0.5CrFeNi2 high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",60,null,9.9,null,null,20,null,null,null,null,88.9,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,40,null,null,null,null,88.6,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,60,null,null,null,null,88.4,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,80,null,null,null,null,88.1,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,100,null,null,null,null,87.9,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,120,null,null,null,null,87.7,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,140,null,null,null,null,87.5,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,160,null,null,null,null,87.3,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,180,null,null,null,null,87.1,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,200,null,null,null,null,86.9,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,220,null,null,null,null,86.7,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,240,null,null,null,null,86.5,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,260,null,null,null,null,86.3,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,280,null,null,null,null,86.1,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,300,null,null,null,null,85.8,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,320,null,null,null,null,85.5,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,340,null,null,null,null,85.2,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,360,null,null,null,null,84.9,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,380,null,null,null,null,84.5,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,400,null,null,null,null,84.2,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,420,null,null,null,null,83.8,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,440,null,null,null,null,83.5,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,460,null,null,null,null,83.1,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,480,null,null,null,null,82.7,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,500,null,null,null,null,82.3,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,520,null,null,null,null,81.9,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,540,null,null,null,null,81.6,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,560,null,null,null,null,81.3,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,580,null,null,null,null,81,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,600,null,null,null,null,80.7,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,620,null,null,null,null,80.4,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,640,null,null,null,null,80.2,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,660,null,null,null,null,80,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,680,null,null,null,null,79.8,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,700,null,null,null,null,79.6,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,720,null,null,null,null,79.3,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,740,null,null,null,null,79,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,760,null,null,null,null,78.7,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,780,null,null,null,null,78.4,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,800,null,null,null,null,78.1,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,820,null,null,null,null,77.7,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[235,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","WROUGHT","BCC",null,null,9.9,null,null,840,null,null,null,null,77.4,107,null,null,null,"10.1016/j.jallcom.2019.05.322",2019,"Temperature dependence of elastic moduli in a refractory HfNbTaTiZr high-entropy alloy"],[237,"Co1 Cr1 Fe1 Ni1 Ti0.3","FCC","WROUGHT","FCC",90,null,7.8,263,"T",25,540,772,60,null,null,214,null,null,null,"10.1016/j.msea.2019.138181",2019,"The effects of substitution of Co with Ni on microstructure, mechanical properties, and age hardening of Co1-xCrFeNi1+xTi0.3 high-entropy alloys"],[237,"Co1 Cr1 Fe1 Ni1 Ti0.3","FCC","WROUGHT","FCC",216,null,7.8,185,null,null,null,null,null,null,null,214,null,null,null,"10.1016/j.msea.2019.138181",2019,"The effects of substitution of Co with Ni on microstructure, mechanical properties, and age hardening of Co1-xCrFeNi1+xTi0.3 high-entropy alloys"],[237,"Co0.333 Cr0.667 Fe0.667 Ni1 Ti0.2","FCC","WROUGHT","FCC",263,null,7.8,197,null,null,null,null,null,null,null,213,null,null,null,"10.1016/j.msea.2019.138181",2019,"The effects of substitution of Co with Ni on microstructure, mechanical properties, and age hardening of Co1-xCrFeNi1+xTi0.3 high-entropy alloys"],[237,"Co0.333 Cr0.667 Fe0.667 Ni1 Ti0.2","FCC","WROUGHT","FCC",90,null,7.8,308,"T",25,586,875,46,null,null,213,null,null,null,"10.1016/j.msea.2019.138181",2019,"The effects of substitution of Co with Ni on microstructure, mechanical properties, and age hardening of Co1-xCrFeNi1+xTi0.3 high-entropy alloys"],[237,"Cr0.5 Fe0.5 Ni1 Ti0.15","FCC","WROUGHT","FCC",90,null,7.8,335,"T",25,672,982,37,null,null,212,null,null,null,"10.1016/j.msea.2019.138181",2019,"The effects of substitution of Co with Ni on microstructure, mechanical properties, and age hardening of Co1-xCrFeNi1+xTi0.3 high-entropy alloys"],[237,"Cr0.5 Fe0.5 Ni1 Ti0.15","FCC","WROUGHT","FCC",324,null,7.8,204,null,null,null,null,null,null,null,212,null,null,null,"10.1016/j.msea.2019.138181",2019,"The effects of substitution of Co with Ni on microstructure, mechanical properties, and age hardening of Co1-xCrFeNi1+xTi0.3 high-entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",20,573,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",100,486,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",200,378,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",300,314,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",400,232,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",600,222,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",800,210,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1","BCC","WROUGHT","BCC",null,null,10,null,"C",1000,160,null,null,null,null,136,100,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Al1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,8.2,null,"C",20,1152,null,null,null,null,120,180,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Al1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,8.2,null,"C",200,740,null,null,null,null,120,180,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Al1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,8.2,null,"C",100,740,null,null,null,null,120,180,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",20,1210,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",60,1071,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",100,1029,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",200,868,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",300,732,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",400,685,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",600,593,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",800,564,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,null,"C",1000,539,null,null,null,null,179,80,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",20,1054,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",100,968,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",200,869,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",300,754,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",400,627,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",600,596,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",800,564,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,12.1,null,"C",1000,459,null,null,null,null,198,90,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Hf1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10.9,null,"C",20,860,null,null,null,null,119,160,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Hf1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10.9,null,"C",20,834,null,null,null,null,119,160,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Hf1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10.9,null,"C",200,610,null,null,null,null,119,160,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Hf1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10.9,null,"C",600,473,null,null,null,null,119,160,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,7.6,null,"C",20,1630,null,null,null,null,196,260,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,7.6,null,"C",200,1268,null,null,null,null,196,260,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,7.6,null,"C",400,1115,null,null,null,null,196,260,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,7.6,null,"C",600,1062,null,null,null,null,196,260,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Nb1 Ti1","BCC","ANNEAL","BCC",null,null,7.6,null,"C",800,1058,null,null,null,null,196,260,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",20,1795,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",100,1506,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",200,1266,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",300,1183,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",400,1175,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",600,1162,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",800,1162,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[238,"Cr1 Mo1 Ta1 Ti1","BCC+Laves","ANNEAL","other",null,null,9.9,null,"C",1000,1159,null,null,null,null,null,200,5,null,"10.1016/j.actamat.2019.06.006",2019,"Solid-solution strengthening in refractory high entropy alloys"],[239,"Hf0.25 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.5,338.3,"C",25,1160,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.5,338.3,"C",600,405,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.5,338.3,"C",800,85,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.125 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.6,332.4,"C",25,1115,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.25 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.7,322.5,"C",25,1065,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.25 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.7,322.5,"C",600,718,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.25 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.7,322.5,"C",800,135,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.375 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.8,318.9,"C",25,1025,null,50,null,null,100,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.5 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.9,310.4,"C",25,980,null,50,null,null,101,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.5 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.9,310.4,"C",600,859,null,50,null,null,101,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[239,"Hf0.25 Nb0.5 Ti1 V0.5 Zr0.5","BCC","CAST","BCC",null,null,6.9,310.4,"C",800,195,null,50,null,null,101,null,null,null,"10.1007/s40195-019-00921-3",2019,"A Novel Series of Refractory High-Entropy Alloys Ti2ZrHf0.5VNbx with High Specific Yield Strength and Good Ductility"],[240,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","WROUGHT","other",null,null,8.1,398,"T",25,1190,1300,4.3,null,null,null,null,null,null,"10.1016/j.matchar.2019.05.043",2019,"Investigation of microstructure, texture, and mechanical properties of FeCrCuMnNi multiphase high entropy alloy during recrystallization"],[240,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","WROUGHT","other",1.4,null,8.1,367,"T",25,1170,1270,10.7,null,null,null,null,null,null,"10.1016/j.matchar.2019.05.043",2019,"Investigation of microstructure, texture, and mechanical properties of FeCrCuMnNi multiphase high entropy alloy during recrystallization"],[240,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","WROUGHT","other",1.5,null,8.1,312,"T",25,1030,1190,11.4,null,null,null,null,null,null,"10.1016/j.matchar.2019.05.043",2019,"Investigation of microstructure, texture, and mechanical properties of FeCrCuMnNi multiphase high entropy alloy during recrystallization"],[240,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","WROUGHT","other",1.9,null,8.1,271,"T",25,970,1110,12.1,null,null,null,null,null,null,"10.1016/j.matchar.2019.05.043",2019,"Investigation of microstructure, texture, and mechanical properties of FeCrCuMnNi multiphase high entropy alloy during recrystallization"],[240,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","WROUGHT","other",2.2,null,8.1,183,"T",25,850,960,14.3,null,null,null,null,null,null,"10.1016/j.matchar.2019.05.043",2019,"Investigation of microstructure, texture, and mechanical properties of FeCrCuMnNi multiphase high entropy alloy during recrystallization"],[240,"Cr1 Cu1 Fe1 Mn1 Ni1","Other","WROUGHT","other",6.5,null,8.1,162,"T",25,470,640,34.8,null,null,null,null,null,null,"10.1016/j.matchar.2019.05.043",2019,"Investigation of microstructure, texture, and mechanical properties of FeCrCuMnNi multiphase high entropy alloy during recrystallization"],[241,"Co0.476 Cr0.476 Fe0.476 Nb0.095 Ni1","FCC+Laves","CAST","other",null,null,8.3,null,"T",25,170,537,32,null,null,null,null,null,null,"10.1016/j.matlet.2019.04.012",2019,"Effect of prolonged aging on phase evolution and mechanical properties of intermetallic strengthened CoCrFeNi2.1Nbx high entropy alloys"],[241,"Co0.476 Cr0.476 Fe0.476 Nb0.19 Ni1","FCC+Laves","CAST","other",null,null,8.4,null,"T",25,640,840,7,null,null,null,null,null,null,"10.1016/j.matlet.2019.04.012",2019,"Effect of prolonged aging on phase evolution and mechanical properties of intermetallic strengthened CoCrFeNi2.1Nbx high entropy alloys"],[241,"Co0.476 Cr0.476 Fe0.476 Nb0.095 Ni1","FCC+Laves+Sec.","ANNEAL","other",null,null,8.3,null,"T",25,560,860,19,null,null,null,null,null,null,"10.1016/j.matlet.2019.04.012",2019,"Effect of prolonged aging on phase evolution and mechanical properties of intermetallic strengthened CoCrFeNi2.1Nbx high entropy alloys"],[241,"Co0.476 Cr0.476 Fe0.476 Nb0.19 Ni1","FCC+Laves","ANNEAL","other",null,null,8.4,null,"T",25,null,null,3,null,null,null,null,null,null,"10.1016/j.matlet.2019.04.012",2019,"Effect of prolonged aging on phase evolution and mechanical properties of intermetallic strengthened CoCrFeNi2.1Nbx high entropy alloys"],[242,"C0.053 Co1 Cr1 Fe1 Mn1 Ni1","FCC+Sec.","ANNEAL","other",null,null,8,345,"C",800,null,null,null,null,null,null,null,null,null,"10.1007/s40195-019-00916-0",2019,"Hot Deformation Behavior and Hardness of a CoCrFeMnNi High-Entropy Alloy with High Content of Carbon"],[242,"C0.053 Co1 Cr1 Fe1 Mn1 Ni1","FCC+Sec.","ANNEAL","other",null,null,8,285,"C",1000,null,null,null,null,null,null,null,null,null,"10.1007/s40195-019-00916-0",2019,"Hot Deformation Behavior and Hardness of a CoCrFeMnNi High-Entropy Alloy with High Content of Carbon"],[243,"Al1 Co1 Cr1 Fe1 Ni1","BCC+FCC","POWDER","other",10.21,null,6.7,550,"C",25,1221.1,null,30,23.2,null,null,null,null,null,"10.1016/j.matchar.2019.05.022",2019,"Effect of heating rate on microstructure and mechanical properties of AlCoCrFeNi high entropy alloy produced by spark plasma sintering"],[243,"Al1 Co1 Cr1 Fe1 Ni1","BCC+FCC","POWDER","other",15.68,null,6.7,510,"C",25,1186.7,null,33,26.5,null,null,null,null,null,"10.1016/j.matchar.2019.05.022",2019,"Effect of heating rate on microstructure and mechanical properties of AlCoCrFeNi high entropy alloy produced by spark plasma sintering"],[243,"Al1 Co1 Cr1 Fe1 Ni1","BCC+FCC","POWDER","other",16.7,null,6.7,503,"C",25,1112.9,null,30,21.4,null,null,null,null,null,"10.1016/j.matchar.2019.05.022",2019,"Effect of heating rate on microstructure and mechanical properties of AlCoCrFeNi high entropy alloy produced by spark plasma sintering"],[244,"Cr1 Mo1 Nb1 Ti1 W1","BCC+Laves","POWDER","other",null,null,9.9,null,"C",25,2500,null,10,null,null,null,null,null,null,"10.1007/s11837-019-03432-9",2019,"Microstructure and Mechanical Properties of WMoNbCrTi HEAs Sintered from the Powders Milled for Different Durations"],[244,"Cr1 Mo1 Nb1 Ti1 W1","BCC+Laves","POWDER","other",null,null,9.9,null,"C",25,1260,null,4.8,null,null,null,null,null,null,"10.1007/s11837-019-03432-9",2019,"Microstructure and Mechanical Properties of WMoNbCrTi HEAs Sintered from the Powders Milled for Different Durations"],[244,"Cr1 Mo1 Nb1 Ti1 W1","BCC+Laves","POWDER","other",null,null,9.9,null,"C",25,1630,null,4.85,null,null,null,null,null,null,"10.1007/s11837-019-03432-9",2019,"Microstructure and Mechanical Properties of WMoNbCrTi HEAs Sintered from the Powders Milled for Different Durations"],[244,"Cr1 Mo1 Nb1 Ti1 W1","BCC+Laves","POWDER","other",null,null,9.9,null,"C",25,2030,null,4.3,null,null,null,null,null,null,"10.1007/s11837-019-03432-9",2019,"Microstructure and Mechanical Properties of WMoNbCrTi HEAs Sintered from the Powders Milled for Different Durations"],[244,"Cr1 Mo1 Nb1 Ti1 W1","BCC+Laves","POWDER","other",null,null,9.9,null,"C",25,1920,null,5,null,null,null,null,null,null,"10.1007/s11837-019-03432-9",2019,"Microstructure and Mechanical Properties of WMoNbCrTi HEAs Sintered from the Powders Milled for Different Durations"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2","POWDER","other",null,null,6.7,null,"C",25,null,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2","POWDER","other",null,null,6.7,null,"C",25,null,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,null,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,null,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,520,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,1150,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,1287,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,1286,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,1390,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[246,"Al1 Co1 Cr1 Fe1 Ni1","BCC+B2+FCC","POWDER","other",null,null,6.7,null,"C",25,1626,null,null,null,null,null,null,null,null,"10.1088/2053-1591/ab2517",2019,"Microstructure and mechanical properties of AlCoCrFeNi high entropy alloys produced by spark plasma sintering"],[247,"Cr1 Fe1 Mn1 Ni1","FCC",null,"FCC",null,null,7.8,null,null,null,null,null,null,null,null,222,null,null,null,"10.1016/j.jallcom.2019.04.296",2019,"Microstructure and mechanical properties of cold drawing CoCrFeMnNi high entropy alloy"],[248,"Co0.25 Cr0.25 Fe1 Mn1","FCC","CAST","FCC",null,null,7.7,null,"T",25,213,471,55.7,null,null,212,null,null,null,"10.1016/j.msea.2019.138039",2019,"Development of a large size FCC high-entropy alloy with excellent mechanical properties"],[248,"Co0.25 Cr0.25 Fe1 Mn1","FCC","CAST","FCC",null,null,7.7,null,"T",25,393,634,26.5,null,null,212,null,null,null,"10.1016/j.msea.2019.138039",2019,"Development of a large size FCC high-entropy alloy with excellent mechanical properties"],[248,"Co0.25 Cr0.25 Fe1 Mn1","FCC","WROUGHT","FCC",null,null,7.7,null,"T",25,792,1055,22.6,null,null,212,null,null,null,"10.1016/j.msea.2019.138039",2019,"Development of a large size FCC high-entropy alloy with excellent mechanical properties"],[248,"Co0.25 Cr0.25 Fe1 Mn1","FCC","WROUGHT","FCC",18,null,7.7,null,"T",25,559,1004,33.9,null,null,212,null,null,null,"10.1016/j.msea.2019.138039",2019,"Development of a large size FCC high-entropy alloy with excellent mechanical properties"],[248,"Co0.25 Cr0.25 Fe1 Mn1","FCC","WROUGHT","FCC",32,null,7.7,null,"T",25,512,978,55.3,null,null,212,null,null,null,"10.1016/j.msea.2019.138039",2019,"Development of a large size FCC high-entropy alloy with excellent mechanical properties"],[248,"Co0.25 Cr0.25 Fe1 Mn1","FCC","WROUGHT","FCC",63,null,7.7,null,"T",25,476,947,69.4,null,null,212,null,null,null,"10.1016/j.msea.2019.138039",2019,"Development of a large size FCC high-entropy alloy with excellent mechanical properties"],[249,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,353,635,null,43,null,220,null,null,null,"10.1016/j.jallcom.2019.04.121",2019,"Strengthening of a CrMnFeCoNi high-entropy alloy by carbide precipitation"],[249,"C0.041 Co1 Cr1 Fe1 Mn1 Nb0.041 Ni1","FCC+Sec.","WROUGHT","other",null,null,8,null,"T",25,732,911,null,32,null,null,null,null,null,"10.1016/j.jallcom.2019.04.121",2019,"Strengthening of a CrMnFeCoNi high-entropy alloy by carbide precipitation"],[250,"Al0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+HCP","WROUGHT","other",null,null,7.3,null,"C",25,1500,null,6,null,null,null,null,null,null,"10.1016/j.matlet.2019.03.075",2019,"Precipitation strengthening in a hot-worked TiNbTa0.5ZrAl0.5 refractory high entropy alloy"],[250,"Al0.5 Nb1 Ta0.5 Ti1 Zr1","BCC","POWDER","BCC",null,null,7.3,null,null,null,null,null,null,null,null,102,null,null,null,"10.1016/j.matlet.2019.03.075",2019,"Precipitation strengthening in a hot-worked TiNbTa0.5ZrAl0.5 refractory high entropy alloy"],[250,"Al0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+HCP","WROUGHT","other",null,null,7.3,null,"C",25,1200,null,8,null,null,null,null,null,null,"10.1016/j.matlet.2019.03.075",2019,"Precipitation strengthening in a hot-worked TiNbTa0.5ZrAl0.5 refractory high entropy alloy"],[250,"Al0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+HCP","WROUGHT","other",null,null,7.3,null,"C",25,2100,null,12,null,null,null,null,null,null,"10.1016/j.matlet.2019.03.075",2019,"Precipitation strengthening in a hot-worked TiNbTa0.5ZrAl0.5 refractory high entropy alloy"],[250,"Al0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+HCP","WROUGHT","other",null,null,7.3,null,"C",25,1740,null,20,null,null,null,null,null,null,"10.1016/j.matlet.2019.03.075",2019,"Precipitation strengthening in a hot-worked TiNbTa0.5ZrAl0.5 refractory high entropy alloy"],[250,"Al0.5 Nb1 Ta0.5 Ti1 Zr1","BCC+HCP","WROUGHT","other",null,null,7.3,null,"C",25,1630,null,12,null,null,null,null,null,null,"10.1016/j.matlet.2019.03.075",2019,"Precipitation strengthening in a hot-worked TiNbTa0.5ZrAl0.5 refractory high entropy alloy"],[251,"Mo1 Nb1 Ta1 W1","BCC","POWDER","BCC",null,null,13.7,504.5,"C",25,1217,null,7.5,3.8,null,250,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.25 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.9,567,"C",25,1826,null,16.5,10.5,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.5 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.2,697,"C",25,1883,null,11,5.8,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.75 Ta1 W1","BCC+Sec.","POWDER","other",null,null,11.6,682.6,"C",25,2483,null,6.2,1.6,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Ta1 W1","BCC","POWDER","BCC",null,null,13.7,420,null,200,null,null,null,null,null,250,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.25 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.9,425,null,200,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.5 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.2,535,null,200,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.75 Ta1 W1","BCC+Sec.","POWDER","other",null,null,11.6,640,null,200,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Ta1 W1","BCC","POWDER","BCC",null,null,13.7,425,null,400,null,null,null,null,null,250,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.25 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.9,420,null,400,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.5 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.2,560,null,400,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.75 Ta1 W1","BCC+Sec.","POWDER","other",null,null,11.6,600,null,400,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Ta1 W1","BCC","POWDER","BCC",null,null,13.7,340,null,600,null,null,null,null,null,250,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.25 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.9,425,null,600,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.5 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.2,540,null,600,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.75 Ta1 W1","BCC+Sec.","POWDER","other",null,null,11.6,650,null,600,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Ta1 W1","BCC","POWDER","BCC",null,null,13.7,335.5,null,800,null,null,null,null,null,250,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.25 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.9,399.2,null,800,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.5 Ta1 W1","BCC+Sec.","POWDER","other",null,null,12.2,550.5,null,800,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.75 Ta1 W1","BCC+Sec.","POWDER","other",null,null,11.6,597.3,null,800,null,null,null,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[251,"Mo1 Nb1 Si0.75 Ta1 W1","BCC+Sec.","POWDER","other",null,null,11.6,null,"C",1200,960,null,22.7,null,null,null,null,null,null,"10.1007/s10853-019-03567-9",2019,"Correction to: Effect of Si additions on microstructure and mechanical properties of refractory NbTaWMo high-entropy alloys"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,1199.26,1335.35,4.37,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,1269.23,1366.3,1.16,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",null,null,8,null,"T",25,1371.21,1402.83,1.58,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",0.166,null,8,null,"T",25,1362.74,1471.05,3.15,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",0.248,null,8,null,"T",25,1241.81,1325.25,4.31,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",0.467999999999999,null,8,null,"T",25,930.3,1021.32,19.01,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",1.79,null,8,null,"T",25,729.21,901.74,26.3,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",4.57,null,8,null,"T",25,494.82,735.6,38.7,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",13.45,null,8,null,"T",25,366.88,673.25,42.55,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",22.37,null,8,null,"T",25,290.65,618.51,43.15,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[252,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","WROUGHT","FCC",30.6,null,8,null,"T",25,258.76,598.75,45.05,null,null,220,null,null,null,"10.1007/s11661-019-05231-y",2019,"Effect of Annealing Temperature on Microstructure and Mechanical Properties of a Severe Cold-Rolled FeCoCrNiMn High-Entropy Alloy"],[253,"C0.1 Co1 Cr1 Fe1 Mn1 Ni1 Ti0.1","FCC+Sec.","POWDER","other",null,7.5,7.8,457,"C",25,1616,null,null,4.5,null,null,null,null,null,"10.1016/j.vacuum.2019.04.043",2019,"Microstructure and mechanical properties of FeCoCrNiMnTi0.1C0.1 high-entropy alloy produced by mechanical alloying and vacuum hot pressing sintering"],[253,"C0.1 Co1 Cr1 Fe1 Mn1 Ni1 Ti0.1","FCC+Sec.","POWDER","other",null,7.64,7.8,461,"C",25,1652,null,null,10,null,null,null,null,null,"10.1016/j.vacuum.2019.04.043",2019,"Microstructure and mechanical properties of FeCoCrNiMnTi0.1C0.1 high-entropy alloy produced by mechanical alloying and vacuum hot pressing sintering"],[253,"C0.1 Co1 Cr1 Fe1 Mn1 Ni1 Ti0.1","FCC+Sec.","POWDER","other",null,7.66,7.8,428,"C",25,1490,null,null,11.4,null,null,null,null,null,"10.1016/j.vacuum.2019.04.043",2019,"Microstructure and mechanical properties of FeCoCrNiMnTi0.1C0.1 high-entropy alloy produced by mechanical alloying and vacuum hot pressing sintering"],[253,"C0.1 Co1 Cr1 Fe1 Mn1 Ni1 Ti0.1","FCC+Sec.","POWDER","other",null,7.69,7.8,378,"C",25,1194,null,null,20.1,null,null,null,null,null,"10.1016/j.vacuum.2019.04.043",2019,"Microstructure and mechanical properties of FeCoCrNiMnTi0.1C0.1 high-entropy alloy produced by mechanical alloying and vacuum hot pressing sintering"],[255,"Co1 Cr1 Fe1 Ni1","FCC",null,"FCC",null,null,8.2,null,null,null,null,null,null,null,null,226,null,null,null,"10.1016/j.intermet.2012.03.005",2012,"Effects of Al addition on the microstructure and mechanical property of AlxCoCrFeNi high-entropy alloys"],[255,"Al0.5 Co1 Cr1 Fe1 Ni1","FCC+BCC",null,"other",null,null,7.3,250,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.intermet.2012.03.005",2012,"Effects of Al addition on the microstructure and mechanical property of AlxCoCrFeNi high-entropy alloys"],[255,"Al1 Co1 Cr1 Fe1 Ni1","BCC",null,"BCC",null,null,6.7,500,null,null,null,null,null,null,null,184,null,null,null,"10.1016/j.intermet.2012.03.005",2012,"Effects of Al addition on the microstructure and mechanical property of AlxCoCrFeNi high-entropy alloys"],[255,"Al1 Co0.5 Cr0.5 Fe0.5 Ni0.5","BCC",null,"BCC",null,null,5.9,733,null,null,null,null,null,null,null,160,null,null,null,"10.1016/j.intermet.2012.03.005",2012,"Effects of Al addition on the microstructure and mechanical property of AlxCoCrFeNi high-entropy alloys"],[256,"Al0.5 Co1 Fe1 Ni1 Ti0.5","FCC+BCC",null,"other",null,null,6.9,733,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2013.12.096",2014,"Effect of Cr addition on the alloying behavior, microstructure and mechanical properties of twinned CoFeNiAl0.5Ti0.5 alloy"],[256,"Al0.5 Co1 Cr1 Fe1 Ni1 Ti0.5","FCC+FCC",null,"other",null,null,6.9,640,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2013.12.096",2014,"Effect of Cr addition on the alloying behavior, microstructure and mechanical properties of twinned CoFeNiAl0.5Ti0.5 alloy"],[257,"Al0.5 Co1 Cr1 Fe1 Mn1 Ni1",null,null,"other",null,null,7.4,null,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[257,"Al0.5 Co1 Cr1 Fe1 Mn1 Ni1 Ta0.2","FCC+BCC+Sec.",null,"other",null,null,7.8,385,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[257,"Al0.5 Co1 Cr1 Fe1 Mn1 Ni1 Ta0.4","FCC+BCC+Sec.",null,"other",null,null,8.3,486,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[257,"Al0.55 Co1 Cr1 Fe1 Mn1 Ni1 Ta0.6","FCC+BCC+Sec.",null,"other",null,null,8.6,570,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[257,"Co1 Cr1 Fe1 Mn1 Ni1","FCC+BCC",null,"other",null,null,8,null,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[257,"Co1 Cr1 Fe1 Mn1 Ni1 Ta0.2","FCC+BCC",null,"other",null,null,8.5,200,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[257,"Co1 Cr1 Fe1 Mn1 Ni1 Ta0.4","FCC+BCC",null,"other",null,null,9,290,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[257,"Co1 Cr1 Fe1 Mn1 Ni1 Ta0.6","FCC+BCC",null,"other",null,null,9.4,345,null,null,null,null,null,null,null,null,null,null,null,"10.4028/www.scientific.net/MSF.846.13",2016,"Effect of Ta Addition on Microstructure and Hardness of FeCrNiMnCoTasubx/sub and Alsub0.5/subFeCrNiMnCoTasubx/sub High-Entropy Alloys"],[258,"Al0.3 Co1 Cr1 Fe1 Ni1","FCC",null,"FCC",null,null,7.6,160,null,null,null,null,null,null,null,210,null,null,null,"10.1016/j.jallcom.2010.02.032",2010,"The effects of secondary elemental Mo or Ti addition in Al0.3CoCrFeNi high-entropy alloy on age hardening at 700°C"],[258,"Al0.3 Co1 Cr1 Fe1 Mo0.1 Ni1","FCC",null,"FCC",null,null,7.7,200,null,null,null,null,null,null,null,214,null,null,null,"10.1016/j.jallcom.2010.02.032",2010,"The effects of secondary elemental Mo or Ti addition in Al0.3CoCrFeNi high-entropy alloy on age hardening at 700°C"],[254,"Al0.143 Cr0.343 Fe1 Mn0.8 Ni0.571","FCC","CAST","FCC",null,null,7.5,null,"T",25,280,556,48,48.2,null,204,null,null,null,"10.1016/j.msea.2019.05.056",2019,"Effect of Al content and cold rolling on the microstructure and mechanical properties of Al5Cr12Fe35Mn28Ni20 high-entropy alloy"],[254,"Al0.286 Cr0.343 Fe1 Mn0.657 Ni0.571","FCC","WROUGHT","FCC",null,null,7.2,null,"T",25,320,625,42.2,42.45,null,195,null,null,null,"10.1016/j.msea.2019.05.056",2019,"Effect of Al content and cold rolling on the microstructure and mechanical properties of Al5Cr12Fe35Mn28Ni20 high-entropy alloy"],[254,"Al0.143 Cr0.343 Fe1 Mn0.8 Ni0.571","FCC","CAST","FCC",null,null,7.5,null,"T",25,1253,1392,6.78,8.2,null,204,null,null,null,"10.1016/j.msea.2019.05.056",2019,"Effect of Al content and cold rolling on the microstructure and mechanical properties of Al5Cr12Fe35Mn28Ni20 high-entropy alloy"],[254,"Al0.286 Cr0.343 Fe1 Mn0.657 Ni0.571","FCC","WROUGHT","FCC",null,null,7.2,null,"T",25,1400,1500,5.83,7.85,null,195,null,null,null,"10.1016/j.msea.2019.05.056",2019,"Effect of Al content and cold rolling on the microstructure and mechanical properties of Al5Cr12Fe35Mn28Ni20 high-entropy alloy"],[259,"Co1 Cu1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8.4,null,null,800,null,null,null,null,null,189,null,null,null,"10.1016/j.msea.2019.04.096",2019,"Establishing processing-microstructure-property paradigm in complex concentrated equiatomic CoCuFeMnNi alloy"],[259,"Co1 Cu1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8.4,null,null,900,null,null,null,null,null,189,null,null,null,"10.1016/j.msea.2019.04.096",2019,"Establishing processing-microstructure-property paradigm in complex concentrated equiatomic CoCuFeMnNi alloy"],[259,"Co1 Cu1 Fe1 Mn1 Ni1","FCC","CAST","FCC",null,null,8.4,null,null,100,null,null,null,null,null,189,null,null,null,"10.1016/j.msea.2019.04.096",2019,"Establishing processing-microstructure-property paradigm in complex concentrated equiatomic CoCuFeMnNi alloy"],[260,"Co1 Cr1 Fe1 Mo0.15 Ni1","FCC","CAST","FCC",null,null,8.3,null,"T",25,489,895,null,53.6,null,231,null,null,null,"10.1016/j.msea.2019.05.078",2019,"Altered microstructural evolution and mechanical properties of CoCrFeNiMo0.15 high-entropy alloy by cryogenic rolling"],[260,"Co1 Cr1 Fe1 Mo0.15 Ni1",null,"WROUGHT","other",null,null,8.3,null,"T",25,873,null,null,23.4,null,null,null,null,null,"10.1016/j.msea.2019.05.078",2019,"Altered microstructural evolution and mechanical properties of CoCrFeNiMo0.15 high-entropy alloy by cryogenic rolling"],[260,"Co1 Cr1 Fe1 Mo0.15 Ni1",null,"WROUGHT","other",null,null,8.3,null,"T",25,952,1122,null,14.2,null,null,null,null,null,"10.1016/j.msea.2019.05.078",2019,"Altered microstructural evolution and mechanical properties of CoCrFeNiMo0.15 high-entropy alloy by cryogenic rolling"],[260,"Co1 Cr1 Fe1 Mo0.15 Ni1","FCC","WROUGHT","FCC",null,null,8.3,null,"T",25,1136,1245,null,8.7,null,231,null,null,null,"10.1016/j.msea.2019.05.078",2019,"Altered microstructural evolution and mechanical properties of CoCrFeNiMo0.15 high-entropy alloy by cryogenic rolling"],[260,"Co1 Cr1 Fe1 Mo0.15 Ni1","FCC","WROUGHT","FCC",null,null,8.3,null,"T",25,1301,1493,null,7.2,null,231,null,null,null,"10.1016/j.msea.2019.05.078",2019,"Altered microstructural evolution and mechanical properties of CoCrFeNiMo0.15 high-entropy alloy by cryogenic rolling"],[260,"Co1 Cr1 Fe1 Mo0.15 Ni1",null,"WROUGHT","other",null,null,8.3,null,"T",25,1311,1421,null,5.9,null,null,null,null,null,"10.1016/j.msea.2019.05.078",2019,"Altered microstructural evolution and mechanical properties of CoCrFeNiMo0.15 high-entropy alloy by cryogenic rolling"],[260,"Co1 Cr1 Fe1 Mo0.15 Ni1",null,"WROUGHT","other",null,null,8.3,null,"T",25,1273,1458,null,5.5,null,null,null,null,null,"10.1016/j.msea.2019.05.078",2019,"Altered microstructural evolution and mechanical properties of CoCrFeNiMo0.15 high-entropy alloy by cryogenic rolling"],[261,"Co0.139 Cr0.446 Cu0.99 Ni1 Si0.277",null,"WROUGHT","other",null,null,7.6,142.5,"T",25,777,810,3.4,null,null,null,null,null,null,"10.1016/j.msea.2019.05.003",2019,"Microstructure and properties of a Cu–Ni–Si–Co–Cr alloy with high strength and high conductivity"],[261,"Co0.139 Cr0.446 Cu0.99 Ni1 Si0.277",null,"WROUGHT","other",null,null,7.6,147.4,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2019.05.003",2019,"Microstructure and properties of a Cu–Ni–Si–Co–Cr alloy with high strength and high conductivity"],[261,"Co0.139 Cr0.446 Cu0.99 Ni1 Si0.277",null,"WROUGHT","other",null,null,7.6,155.2,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2019.05.003",2019,"Microstructure and properties of a Cu–Ni–Si–Co–Cr alloy with high strength and high conductivity"],[262,"Al1 Co1 Cr0.091 Nb0.727 W0.212",null,"CAST","other",null,null,7.2,1084,null,25,null,null,null,null,null,null,null,null,null,"10.1038/s41467-019-10533-1",2019,"Materials informatics for the screening of multi-principal elements and high-entropy alloys"],[262,"Co1 Cr0.917 Ni1 Ta0.5 Ti0.75",null,"CAST","other",null,null,8.8,1011,null,25,null,null,null,null,null,null,null,null,null,"10.1038/s41467-019-10533-1",2019,"Materials informatics for the screening of multi-principal elements and high-entropy alloys"],[262,"Al0.947 Co0.158 Mo1 Ni0.289 W0.237",null,"CAST","other",null,null,8,725,null,25,null,null,null,null,null,null,null,null,null,"10.1038/s41467-019-10533-1",2019,"Materials informatics for the screening of multi-principal elements and high-entropy alloys"],[262,"Co0.043 Nb0.638 Ni1 Ta0.255 Ti0.191",null,"CAST","other",null,null,9.5,815,null,25,null,null,null,null,null,null,null,null,null,"10.1038/s41467-019-10533-1",2019,"Materials informatics for the screening of multi-principal elements and high-entropy alloys"],[262,"Co0.273 Cr0.477 Nb0.477 Ni0.045 Ti1",null,"CAST","other",null,null,6.4,422,null,25,null,null,null,null,null,null,null,null,null,"10.1038/s41467-019-10533-1",2019,"Materials informatics for the screening of multi-principal elements and high-entropy alloys"],[262,"Co1 Cr0.487 Nb0.231 Ta0.026 Ti0.821",null,"CAST","other",null,null,6.9,856,null,25,null,null,null,null,null,null,null,null,null,"10.1038/s41467-019-10533-1",2019,"Materials informatics for the screening of multi-principal elements and high-entropy alloys"],[262,"Co0.564 Nb0.795 Ta0.103 Ti1 W0.103",null,"CAST","other",null,null,7.7,277,null,25,null,null,null,null,null,null,null,null,null,"10.1038/s41467-019-10533-1",2019,"Materials informatics for the screening of multi-principal elements and high-entropy alloys"],[263,"Al1 Co1 Cu1 Fe1 Ni0.5","FCC+BCC+B2","CAST","other",null,null,6.9,430.9,"C",25,929,null,13.7,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.014",2019,"Composition and phase structure dependence of mechanical and magnetic properties for AlCoCuFeNihigh entropy alloys"],[263,"Al1 Co1 Cu1 Fe1 Ni0.8","FCC+BCC+B2","CAST","other",null,null,7,420,"C",25,1021,null,12.3,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.014",2019,"Composition and phase structure dependence of mechanical and magnetic properties for AlCoCuFeNihigh entropy alloys"],[263,"Al1 Co1 Cu1 Fe1 Ni1","FCC+BCC+B2","CAST","other",null,null,7,391.4,"C",25,994,null,24.5,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.014",2019,"Composition and phase structure dependence of mechanical and magnetic properties for AlCoCuFeNihigh entropy alloys"],[263,"Al0.667 Co0.667 Cu0.667 Fe0.667 Ni1","FCC+BCC+B2","CAST","other",null,null,7.2,323.8,"C",25,680,null,35.9,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.014",2019,"Composition and phase structure dependence of mechanical and magnetic properties for AlCoCuFeNihigh entropy alloys"],[263,"Al0.5 Co0.5 Cu0.5 Fe0.5 Ni1","FCC+BCC+B2","CAST","other",null,null,7.3,299.2,"C",25,544,null,null,null,null,null,null,null,null,"10.1016/j.jmst.2018.12.014",2019,"Composition and phase structure dependence of mechanical and magnetic properties for AlCoCuFeNihigh entropy alloys"],[263,"Al0.333 Co0.333 Cu0.333 Fe0.333 Ni1","FCC","CAST","FCC",null,null,7.5,279.6,"C",25,268,null,null,null,null,167,null,null,null,"10.1016/j.jmst.2018.12.014",2019,"Composition and phase structure dependence of mechanical and magnetic properties for AlCoCuFeNihigh entropy alloys"],[99,"Al0.5 Cr1 Fe1 Mn1 Ni1","BCC+B2","CAST","other",null,null,7.1,353,"C",25,1091,null,50,null,null,null,null,null,null,"10.1016/j.msea.2019.04.104",2019,"The microstructure and mechanical properties of novel Al-Cr-Fe-Mn-Ni high-entropy alloys with trimodal distributions of coherent B2 precipitates"],[99,"Al0.6 Cr1 Fe1 Mn1 Ni1","BCC+B2","CAST","other",null,null,6.9,361,"C",25,1130,null,53,null,null,null,null,null,null,"10.1016/j.msea.2019.04.104",2019,"The microstructure and mechanical properties of novel Al-Cr-Fe-Mn-Ni high-entropy alloys with trimodal distributions of coherent B2 precipitates"],[99,"Al0.7 Cr1 Fe1 Mn1 Ni1","BCC+B2","CAST","other",null,null,6.8,375,"C",25,1170,null,55,null,null,null,null,null,null,"10.1016/j.msea.2019.04.104",2019,"The microstructure and mechanical properties of novel Al-Cr-Fe-Mn-Ni high-entropy alloys with trimodal distributions of coherent B2 precipitates"],[99,"Al0.8 Cr1 Fe1 Mn1 Ni1","BCC+B2","CAST","other",null,null,6.7,386,"C",25,1200,null,55,null,null,null,null,null,null,"10.1016/j.msea.2019.04.104",2019,"The microstructure and mechanical properties of novel Al-Cr-Fe-Mn-Ni high-entropy alloys with trimodal distributions of coherent B2 precipitates"],[100,"Co1 Cr0.2 Fe1 Ni1 Si0.2","FCC","CAST","FCC",null,null,7.8,130.1,"T",25,225,732,61,null,null,195,null,null,null,"10.1016/j.jmmm.2019.01.096",2019,"A novel FeCoNiCr0.2Si0.2 high entropy alloy with an excellent balance of mechanical and soft magnetic properties"],[100,"Co1 Cr0.2 Fe1 Ni1 Si0.2","FCC","WROUGHT","FCC",null,null,7.8,318.2,"T",25,705,1070,10,null,null,195,null,null,null,"10.1016/j.jmmm.2019.01.096",2019,"A novel FeCoNiCr0.2Si0.2 high entropy alloy with an excellent balance of mechanical and soft magnetic properties"],[100,"Co1 Cr0.2 Fe1 Ni1 Si0.2","FCC","WROUGHT","FCC",null,null,7.8,153.8,"T",25,320,920,60,null,null,195,null,null,null,"10.1016/j.jmmm.2019.01.096",2019,"A novel FeCoNiCr0.2Si0.2 high entropy alloy with an excellent balance of mechanical and soft magnetic properties"],[101,"Al1 Cu0.077 Fe0.077 Mg0.077 Si0.231 Zn0.077","FCC","CAST","FCC",null,3.08,3.2,null,"C",25,422,null,1,null,88.7,73,null,null,null,"10.1038/s41598-019-43329-w",2019,"Design, Microstructure and Mechanical Properties of Cast Medium Entropy Aluminium Alloys"],[101,"Al1 Cu0.077 Mg0.077 Ni0.077 Si0.231 Zn0.077","FCC","CAST","FCC",null,3.15,3.2,null,"C",25,534,null,1,null,107.8,72,null,null,null,"10.1038/s41598-019-43329-w",2019,"Design, Microstructure and Mechanical Properties of Cast Medium Entropy Aluminium Alloys"],[101,"Al1 Cr0.077 Cu0.077 Mg0.077 Si0.231 Zn0.077","FCC","CAST","FCC",null,3.06,3.1,null,"C",25,490,null,6,null,78.4,75,null,null,null,"10.1038/s41598-019-43329-w",2019,"Design, Microstructure and Mechanical Properties of Cast Medium Entropy Aluminium Alloys"],[101,"Al1 Cu0.077 Mg0.077 Mn0.077 Si0.231 Zn0.077","FCC","CAST","FCC",null,2.98,3.2,null,"C",25,622,null,2,null,114.1,73,null,null,null,"10.1038/s41598-019-43329-w",2019,"Design, Microstructure and Mechanical Properties of Cast Medium Entropy Aluminium Alloys"],[101,"Al1 Cu0.077 Mg0.077 Si0.231 Zn0.077 Zr0.077","FCC+Laves","CAST","other",null,3.06,3.2,null,"C",25,565,null,4,null,105.1,null,null,null,null,"10.1038/s41598-019-43329-w",2019,"Design, Microstructure and Mechanical Properties of Cast Medium Entropy Aluminium Alloys"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","CAST","FCC",null,null,8.3,null,"T",25,400,780,53,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,838,43,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,899,31,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,902,29,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,811,52,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,790,49,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,799,47,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,802,37,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,823,36,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,849,34,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[102,"Co1 Cr1 Fe1 Mo0.2 Ni1","FCC","ANNEAL","FCC",null,null,8.3,null,"T",25,null,895,32,null,null,232,null,null,null,"10.3390/e21050448",2019,"Effects of Annealing on Microstructure and Mechanical Properties of Metastable Powder Metallurgy CoCrFeNiMo0.2 High Entropy Alloy"],[103,"Al0.569 Ca0.143 Li0.857 Mg1 Si0.286 Y0.003",null,"CAST","other",null,1.7,1.6,null,"C",25,556,null,3,null,null,null,null,null,null,"10.3390/ma12071136",2019,"Novel Ultralight-Weight Complex Concentrated Alloys with High Strength"],[103,"Al0.429 Ca0.286 Li1 Mg1 Si0.143",null,"CAST","other",null,1.57,1.4,null,"C",25,418,null,13,null,null,null,null,null,null,"10.3390/ma12071136",2019,"Novel Ultralight-Weight Complex Concentrated Alloys with High Strength"],[103,"Al0.312 Ca0.021 Li0.792 Mg1 Si0.021",null,"CAST","other",null,1.52,1.4,null,"C",25,360,null,9,null,null,null,null,null,null,"10.3390/ma12071136",2019,"Novel Ultralight-Weight Complex Concentrated Alloys with High Strength"],[103,"Al0.333 Ca0.011 Li0.844 Mg1 Si0.033",null,"CAST","other",null,1.5,1.4,null,"C",25,342,null,44,null,null,null,null,null,null,"10.3390/ma12071136",2019,"Novel Ultralight-Weight Complex Concentrated Alloys with High Strength"],[103,"Al0.333 Ca0.011 Li0.867 Mg1 Si0.011",null,"CAST","other",null,1.46,1.4,null,"C",25,300,null,63,null,null,null,null,null,null,"10.3390/ma12071136",2019,"Novel Ultralight-Weight Complex Concentrated Alloys with High Strength"],[104,"Co1 Cr1 Fe1 Ni1 Pd1","FCC","CAST","FCC",null,null,9.2,295,null,null,null,null,null,null,null,201,null,null,null,"10.3390/e21030288",2019,"Effect of Mn Addition on the Microstructures and Mechanical Properties of CoCrFeNiPd High Entropy Alloy"],[104,"Co1 Cr1 Fe1 Mn0.2 Ni1 Pd1","FCC+Sec.","CAST","other",null,null,9.1,470,"C",25,650,null,50,null,null,null,null,null,null,"10.3390/e21030288",2019,"Effect of Mn Addition on the Microstructures and Mechanical Properties of CoCrFeNiPd High Entropy Alloy"],[104,"Co1 Cr1 Fe1 Mn0.4 Ni1 Pd1","FCC+Sec.","CAST","other",null,null,9.1,380,"C",25,650,null,46,null,null,null,null,null,null,"10.3390/e21030288",2019,"Effect of Mn Addition on the Microstructures and Mechanical Properties of CoCrFeNiPd High Entropy Alloy"],[104,"Co1 Cr1 Fe1 Mn0.6 Ni1 Pd1","FCC+Sec.","CAST","other",null,null,9,180,"C",25,650,null,42,null,null,null,null,null,null,"10.3390/e21030288",2019,"Effect of Mn Addition on the Microstructures and Mechanical Properties of CoCrFeNiPd High Entropy Alloy"],[104,"Co1 Cr1 Fe1 Mn0.8 Ni1 Pd1","FCC+Sec.","CAST","other",null,null,8.9,160,"C",25,650,null,35,null,null,null,null,null,null,"10.3390/e21030288",2019,"Effect of Mn Addition on the Microstructures and Mechanical Properties of CoCrFeNiPd High Entropy Alloy"],[106,"Al0.3 Co1 Cr1 Fe1 Mn1 Ni1",null,"POWDER","other",null,null,7.6,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.scriptamat.2018.12.021",2019,"Strengthening of Al0.3CoCrFeMnNi-based ODS high entropy alloys with incremental changes in the concentration of Y2O3"],[107,"Cr1 Fe1 Ni1 Ti0.2","BCC+FCC","CAST","other",null,null,7.6,362.2,"C",25,2297.94,null,5.8,null,null,null,null,null,null,"10.1557/jmr.2019.40",2019,"Effects of titanium addition on microstructure and mechanical properties of CrFeNiTix (x = 0.2–0.6) compositionally complex alloys"],[107,"Cr1 Fe1 Ni1 Ti0.3","BCC+FCC","CAST","other",null,null,7.5,510.3,"C",25,2364.63,null,null,null,null,null,null,null,null,"10.1557/jmr.2019.40",2019,"Effects of titanium addition on microstructure and mechanical properties of CrFeNiTix (x = 0.2–0.6) compositionally complex alloys"],[107,"Cr1 Fe1 Ni1 Ti0.4","BCC","CAST","BCC",null,null,7.4,600.3,"C",25,2475.41,null,6.5,null,null,212,null,null,null,"10.1557/jmr.2019.40",2019,"Effects of titanium addition on microstructure and mechanical properties of CrFeNiTix (x = 0.2–0.6) compositionally complex alloys"],[107,"Cr1 Fe1 Ni1 Ti0.5","BCC","CAST","BCC",null,null,7.3,709.6,"C",25,2380.79,null,null,null,null,208,null,null,null,"10.1557/jmr.2019.40",2019,"Effects of titanium addition on microstructure and mechanical properties of CrFeNiTix (x = 0.2–0.6) compositionally complex alloys"],[107,"Cr1 Fe1 Ni1 Ti0.6","BCC","CAST","BCC",null,null,7.2,718.2,"C",25,1923.15,null,null,null,null,204,null,null,null,"10.1557/jmr.2019.40",2019,"Effects of titanium addition on microstructure and mechanical properties of CrFeNiTix (x = 0.2–0.6) compositionally complex alloys"],[108,"Al1 Cr1 Cu1 Fe1 Mn1","Other","POWDER","other",null,null,6.5,633,"C",25,1010,null,null,null,null,null,null,null,null,"10.1557/jmr.2019.18",2019,"Materials-structure-property correlation study of spark plasma sintered AlCuCrFeMnWx (x = 0, 0.05, 0.1, 0.5) high-entropy alloys"],[108,"Al1 Cr1 Cu1 Fe1 Mn1 W0.05","BCC+Sec.","POWDER","other",null,null,6.7,680,"C",25,1120,null,null,null,null,null,null,null,null,"10.1557/jmr.2019.18",2019,"Materials-structure-property correlation study of spark plasma sintered AlCuCrFeMnWx (x = 0, 0.05, 0.1, 0.5) high-entropy alloys"],[108,"Al1 Cr1 Cu1 Fe1 Mn1 W0.1","BCC+Sec.","POWDER","other",null,null,6.8,705,"C",25,1250,null,null,null,null,null,null,null,null,"10.1557/jmr.2019.18",2019,"Materials-structure-property correlation study of spark plasma sintered AlCuCrFeMnWx (x = 0, 0.05, 0.1, 0.5) high-entropy alloys"],[108,"Al1 Cr1 Cu1 Fe1 Mn1 W0.5","BCC+Sec.","POWDER","other",null,null,7.9,780,"C",25,1510,null,null,null,null,null,null,null,null,"10.1557/jmr.2019.18",2019,"Materials-structure-property correlation study of spark plasma sintered AlCuCrFeMnWx (x = 0, 0.05, 0.1, 0.5) high-entropy alloys"],[109,"Co1 Cu1 Fe1 Nb0.5 Ni1","FCC+Laves","WROUGHT","other",null,null,8.6,null,null,null,null,null,null,null,null,null,null,null,null,"10.1557/jmr.2019.36",2019,"Effect of niobium addition in FeCoNiCuNbx high-entropy alloys"],[109,"Co0.2 Cu0.2 Fe0.2 Nb1 Ni0.2","FCC+Laves","WROUGHT","other",null,null,8.6,null,"C",25,1498,null,37.6,null,null,null,null,null,null,"10.1557/jmr.2019.36",2019,"Effect of niobium addition in FeCoNiCuNbx high-entropy alloys"],[109,"Co0.133 Cu0.133 Fe0.133 Nb1 Ni0.133","FCC+Laves","WROUGHT","other",null,null,8.6,null,"C",25,1635,null,31.7,null,null,null,null,null,null,"10.1557/jmr.2019.36",2019,"Effect of niobium addition in FeCoNiCuNbx high-entropy alloys"],[109,"Co0.086 Cu0.086 Fe0.086 Nb1 Ni0.086","FCC+Laves","WROUGHT","other",null,null,8.6,null,"C",25,1703,null,18.3,null,null,null,null,null,null,"10.1557/jmr.2019.36",2019,"Effect of niobium addition in FeCoNiCuNbx high-entropy alloys"],[109,"Co0.067 Cu0.067 Fe0.067 Nb1 Ni0.067","FCC+Laves","WROUGHT","other",null,null,8.6,null,"C",25,1415,null,12.6,null,null,null,null,null,null,"10.1557/jmr.2019.36",2019,"Effect of niobium addition in FeCoNiCuNbx high-entropy alloys"],[110,"Co0.2 Cr0.2 Fe1 Mn0.6","FCC+Sec.","CAST","other",null,null,7.8,296,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2019.01.088",2019,"The effect of boron content on the microstructure and mechanical properties of Fe50-XMn30Co10Cr10BX (x = 0, 0.3, 0.6 and 1.7 wt%) multi-component alloys prepared by arc-melting"],[110,"B0.006 Co0.201 Cr0.201 Fe1 Mn0.604","FCC+Sec.","CAST","other",null,null,7.8,282,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2019.01.088",2019,"The effect of boron content on the microstructure and mechanical properties of Fe50-XMn30Co10Cr10BX (x = 0, 0.3, 0.6 and 1.7 wt%) multi-component alloys prepared by arc-melting"],[110,"B0.012 Co0.202 Cr0.202 Fe1 Mn0.607","FCC+Sec.","CAST","other",null,null,7.8,306,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2019.01.088",2019,"The effect of boron content on the microstructure and mechanical properties of Fe50-XMn30Co10Cr10BX (x = 0, 0.3, 0.6 and 1.7 wt%) multi-component alloys prepared by arc-melting"],[110,"B0.035 Co0.207 Cr0.207 Fe1 Mn0.621","FCC+Sec.","CAST","other",null,null,7.7,452,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2019.01.088",2019,"The effect of boron content on the microstructure and mechanical properties of Fe50-XMn30Co10Cr10BX (x = 0, 0.3, 0.6 and 1.7 wt%) multi-component alloys prepared by arc-melting"],[111,"Al0.186 Cr0.149 Fe1 Mn0.861 Ni0.28",null,"CAST","other",null,null,7.3,null,"T",25,159,535,null,40.8,null,null,null,null,null,"10.1016/j.msea.2019.01.083",2019,"The effects of carbon on the phase stability and mechanical properties of heat-treated FeNiMnCrAl high entropy alloys"],[111,"Al0.186 Cr0.149 Fe1 Mn0.861 Ni0.28",null,"ANNEAL","other",null,null,7.3,null,"T",25,409,584,null,10.5,null,null,null,null,null,"10.1016/j.msea.2019.01.083",2019,"The effects of carbon on the phase stability and mechanical properties of heat-treated FeNiMnCrAl high entropy alloys"],[111,"Al0.186 Cr0.149 Fe1 Mn0.861 Ni0.28",null,"CAST","other",null,null,7.3,null,"T",25,355,1174,null,49.5,null,null,null,null,null,"10.1016/j.msea.2019.01.083",2019,"The effects of carbon on the phase stability and mechanical properties of heat-treated FeNiMnCrAl high entropy alloys"],[111,"Al0.186 Cr0.149 Fe1 Mn0.861 Ni0.28",null,"ANNEAL","other",null,null,7.3,null,"T",25,954,978,null,2,null,null,null,null,null,"10.1016/j.msea.2019.01.083",2019,"The effects of carbon on the phase stability and mechanical properties of heat-treated FeNiMnCrAl high entropy alloys"],[112,"Hf1 Ti1 Zr1","Other","CAST","other",null,null,8.3,211,"C",25,773,null,17,null,null,null,null,null,null,"10.1007/s00339-019-2484-1",2019,"Effect of Sc and Y addition on the microstructure and properties of HCP-structured high-entropy alloys"],[112,"Hf1 Sc1 Ti1 Zr1","Other","CAST","other",null,null,6.8,233,"C",25,1001,null,21.9,null,null,null,null,null,null,"10.1007/s00339-019-2484-1",2019,"Effect of Sc and Y addition on the microstructure and properties of HCP-structured high-entropy alloys"],[112,"Hf1 Ti1 Y1 Zr1","Other","CAST","other",null,null,7,241,"C",25,554,null,17.7,null,null,null,null,null,null,"10.1007/s00339-019-2484-1",2019,"Effect of Sc and Y addition on the microstructure and properties of HCP-structured high-entropy alloys"],[112,"Hf1 Sc1 Ti1 Y1 Zr1","Other","CAST","other",null,null,6.2,256,"C",25,793,null,15.7,null,null,null,null,null,null,"10.1007/s00339-019-2484-1",2019,"Effect of Sc and Y addition on the microstructure and properties of HCP-structured high-entropy alloys"],[113,"Co1 Cr1 Fe1 Ga1 Ni1","FCC+BCC","CAST","other",null,null,7.5,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s10853-018-03196-8",2018,"The effect of cooling rate on the microstructure and mechanical properties of NiCoFeCrGa high-entropy alloy"],[114,"Al0.562 Co0.938 Cr0.312 Fe0.312 Ni1","FCC+BCC","CAST","other",null,null,7.1,305.4,"C",25,1129.4,null,16.7,null,null,null,null,null,null,"10.1007/s00339-019-2506-z",2019,"Enhancing the mechanical properties of casting eutectic high entropy alloys with Mo addition"],[114,"Al0.581 Co0.968 Cr0.323 Fe0.323 Mo0.032 Ni1","FCC+BCC","CAST","other",null,null,7.1,357.3,"C",25,1159.7,null,18.2,null,null,null,null,null,null,"10.1007/s00339-019-2506-z",2019,"Enhancing the mechanical properties of casting eutectic high entropy alloys with Mo addition"],[114,"Al0.6 Co1 Cr0.333 Fe0.333 Mo0.067 Ni1","FCC+BCC","CAST","other",null,null,7.1,362.6,"C",25,1250.8,null,14.7,null,null,null,null,null,null,"10.1007/s00339-019-2506-z",2019,"Enhancing the mechanical properties of casting eutectic high entropy alloys with Mo addition"],[114,"Al0.6 Co1 Cr0.333 Fe0.333 Mo0.1 Ni0.967","FCC+BCC","CAST","other",null,null,7.2,359.7,"C",25,1198.3,null,14.5,null,null,null,null,null,null,"10.1007/s00339-019-2506-z",2019,"Enhancing the mechanical properties of casting eutectic high entropy alloys with Mo addition"],[116,"Co1 Cr1 Fe1 Mo0.149 Ni1","FCC+Sec.","POWDER","other",null,null,8.3,210,null,null,null,null,null,null,null,null,null,null,null,"10.3390/met9030351",2019,"Gradient Distribution of Microstructures and Mechanical Properties in a FeCoCrNiMo High-Entropy Alloy during Spark Plasma Sintering"],[117,"Co1 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,8.3,null,"C",25,300,null,50.2,null,null,206,null,null,null,"10.1007/s11665-018-3837-1",2019,"Improvement of Microstructure and Mechanical Properties of CoCrCuFeNi High-Entropy Alloys By V Addition"],[117,"Co1 Cr1 Cu1 Fe1 Ni1 V0.042","FCC+Sec.","CAST","other",null,null,8.3,null,"C",25,303,null,50.2,null,null,null,null,null,null,"10.1007/s11665-018-3837-1",2019,"Improvement of Microstructure and Mechanical Properties of CoCrCuFeNi High-Entropy Alloys By V Addition"],[117,"Co1 Cr1 Cu1 Fe1 Ni1 V0.087","FCC+Sec.","CAST","other",null,null,8.3,null,"C",25,307,null,50.2,null,null,null,null,null,null,"10.1007/s11665-018-3837-1",2019,"Improvement of Microstructure and Mechanical Properties of CoCrCuFeNi High-Entropy Alloys By V Addition"],[117,"Co1 Cr1 Cu1 Fe1 Ni1 V0.136","FCC+Sec.","CAST","other",null,null,8.3,null,"C",25,330,null,50.2,null,null,null,null,null,null,"10.1007/s11665-018-3837-1",2019,"Improvement of Microstructure and Mechanical Properties of CoCrCuFeNi High-Entropy Alloys By V Addition"],[117,"Co1 Cr1 Cu1 Fe1 Ni1 V0.19","FCC+Sec.","CAST","other",null,null,8.2,null,"C",25,616,null,28,null,null,null,null,null,null,"10.1007/s11665-018-3837-1",2019,"Improvement of Microstructure and Mechanical Properties of CoCrCuFeNi High-Entropy Alloys By V Addition"],[118,"Co1 Cr1 Cu1 Ni1 Zn1","BCC+FCC","POWDER","other",500,5.26,8.1,null,"C",25,null,null,null,null,null,null,null,null,null,"10.3390/e21020122",2019,"Phases, Microstructures and Mechanical Properties of CoCrNiCuZn High-Entropy Alloy Prepared by Mechanical Alloying and Spark Plasma Sintering"],[118,"Co1 Cr1 Cu1 Ni1 Zn1","BCC+FCC","POWDER","other",500,6.26,8.1,null,"C",25,null,null,null,null,null,null,null,null,null,"10.3390/e21020122",2019,"Phases, Microstructures and Mechanical Properties of CoCrNiCuZn High-Entropy Alloy Prepared by Mechanical Alloying and Spark Plasma Sintering"],[118,"Co1 Cr1 Cu1 Ni1 Zn1","BCC+FCC","POWDER","other",500,7.84,8.1,null,"C",25,null,null,null,null,null,null,null,null,null,"10.3390/e21020122",2019,"Phases, Microstructures and Mechanical Properties of CoCrNiCuZn High-Entropy Alloy Prepared by Mechanical Alloying and Spark Plasma Sintering"],[118,"Co1 Cr1 Cu1 Ni1 Zn1","FCC+FCC","POWDER","other",500,7.89,8.1,615,"C",25,2121,null,null,null,null,null,null,null,null,"10.3390/e21020122",2019,"Phases, Microstructures and Mechanical Properties of CoCrNiCuZn High-Entropy Alloy Prepared by Mechanical Alloying and Spark Plasma Sintering"],[119,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",500,null,9.9,359,"T",25,1155,1212,12.3,null,59,107,null,null,null,"10.3390/e21020114",2019,"Microstructure and Room Temperature Mechanical Properties of Different 3 and 4 Element Medium Entropy Alloys from HfNbTaTiZr System"],[119,"Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",500,null,8.9,358,"T",25,1144,1205,6.4,null,98,115,null,null,null,"10.3390/e21020114",2019,"Microstructure and Room Temperature Mechanical Properties of Different 3 and 4 Element Medium Entropy Alloys from HfNbTaTiZr System"],[119,"Nb1 Ta1 Ti1","BCC","CAST","BCC",500,null,10,246,"T",25,620,683,18.5,null,143,136,null,null,null,"10.3390/e21020114",2019,"Microstructure and Room Temperature Mechanical Properties of Different 3 and 4 Element Medium Entropy Alloys from HfNbTaTiZr System"],[119,"Ta1 Ti1 Zr1","BCC+Sec.","CAST","other",500,null,9,485,"T",25,null,284,0,null,157,null,null,null,null,"10.3390/e21020114",2019,"Microstructure and Room Temperature Mechanical Properties of Different 3 and 4 Element Medium Entropy Alloys from HfNbTaTiZr System"],[119,"Nb1 Ti1 Zr1","BCC","CAST","BCC",500,null,6.5,295,"T",25,956,991,14.2,null,88,94,null,null,null,"10.3390/e21020114",2019,"Microstructure and Room Temperature Mechanical Properties of Different 3 and 4 Element Medium Entropy Alloys from HfNbTaTiZr System"],[119,"Nb1 Ta0.667 Ti0.667 Zr0.333","BCC+Sec.","CAST","other",500,null,9.2,294,"T",25,822,852,0.33,null,127,null,null,null,null,"10.3390/e21020114",2019,"Microstructure and Room Temperature Mechanical Properties of Different 3 and 4 Element Medium Entropy Alloys from HfNbTaTiZr System"],[119,"Nb0.333 Ta0.667 Ti0.667 Zr1","BCC+Sec.","CAST","other",500,null,8.6,489,"T",25,null,843,0,null,93,null,null,null,null,"10.3390/e21020114",2019,"Microstructure and Room Temperature Mechanical Properties of Different 3 and 4 Element Medium Entropy Alloys from HfNbTaTiZr System"],[120,"Ag1 Al1 Nb1 Ti1 Zn1",null,"ANNEAL","other",null,4.75,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2018.12.020",2019,"Microstructural characterization and mechanical behavior of an AgAlNbTiZn complex composition alloy produced using powder metallurgy (P/M)"],[120,"Ag1 Al1 Nb1 Ti1 Zn1",null,"ANNEAL","other",null,5.78,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2018.12.020",2019,"Microstructural characterization and mechanical behavior of an AgAlNbTiZn complex composition alloy produced using powder metallurgy (P/M)"],[120,"Ag1 Al1 Nb1 Ti1 Zn1",null,"ANNEAL","other",null,5.79,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2018.12.020",2019,"Microstructural characterization and mechanical behavior of an AgAlNbTiZn complex composition alloy produced using powder metallurgy (P/M)"],[120,"Ag1 Al1 Nb1 Ti1 Zn1",null,"ANNEAL","other",null,5.51,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2018.12.020",2019,"Microstructural characterization and mechanical behavior of an AgAlNbTiZn complex composition alloy produced using powder metallurgy (P/M)"],[120,"Ag1 Al1 Nb1 Ti1 Zn1",null,"ANNEAL","other",null,null,6.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.msea.2018.12.020",2019,"Microstructural characterization and mechanical behavior of an AgAlNbTiZn complex composition alloy produced using powder metallurgy (P/M)"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484",null,"CAST","other",null,null,6.3,310,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484 Ti0.026",null,"CAST","other",null,null,6.3,330,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484 Ti0.097",null,"CAST","other",null,null,6.2,500,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484",null,"ANNEAL","other",null,null,6.3,340,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484 Ti0.026",null,"ANNEAL","other",null,null,6.3,410,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484 Ti0.097",null,"ANNEAL","other",null,null,6.2,495,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484",null,"ANNEAL","other",null,null,6.3,330,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484 Ti0.026",null,"ANNEAL","other",null,null,6.3,330,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[121,"Al0.774 Fe1 Mn0.903 Ni0.484 Ti0.097",null,"ANNEAL","other",null,null,6.2,490,null,null,null,null,null,null,null,null,null,null,null,"10.1155/2019/2157592",2019,"Effect of Heat Treatment and Titanium Addition on the Microstructure and Mechanical Properties of Cast Fe31Mn28 Ni15Al24.5Tix High-Entropy Alloys"],[122,"Al1 C1 Co1 Cr1 Fe1 Ni1","FCC+BCC","POWDER","other",null,null,6.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1080/00325899.2019.1576389",2019,"Microstructural evolution and mechanical properties of AlCrFeNiCoC high entropy alloy produced via spark plasma sintering"],[123,"Cr1 Cu1 Fe1 Ni1 Ti1","FCC+BCC","CAST","other",null,null,7.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11106-019-00012-z",2019,"The Influence of Synthesis Conditions on the Phase Composition, Structure, and Properties of the High-Entropy Ti–Cr–Fe–Ni–Cu Alloy"],[123,"Cr1 Cu1 Fe1 Ni1 Ti1","FCC+BCC","WROUGHT","other",null,null,7.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11106-019-00012-z",2019,"The Influence of Synthesis Conditions on the Phase Composition, Structure, and Properties of the High-Entropy Ti–Cr–Fe–Ni–Cu Alloy"],[123,"Cr1 Cu1 Fe1 Ni1 Ti1","FCC+BCC","WROUGHT","other",null,null,7.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11106-019-00012-z",2019,"The Influence of Synthesis Conditions on the Phase Composition, Structure, and Properties of the High-Entropy Ti–Cr–Fe–Ni–Cu Alloy"],[123,"Cr1 Cu1 Fe1 Ni1 Ti1","FCC+BCC","WROUGHT","other",null,null,7.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1007/s11106-019-00012-z",2019,"The Influence of Synthesis Conditions on the Phase Composition, Structure, and Properties of the High-Entropy Ti–Cr–Fe–Ni–Cu Alloy"],[124,"Co1 Cr1 Fe1 Mn1 Ni1","FCC","POWDER","other",0.03,7.753,8,435,"C",25,1100,null,47,null,null,null,null,null,null,"10.1080/00325899.2018.1554834",2018,"Microstructure and properties of CrMnFeCoNi high-entropy alloy prepared by mechanical alloying and spark plasma sintering"],[126,"Al0.5 Cr1 Cu1 Fe1 Ni1","FCC","CAST","FCC",null,null,7.4,226,null,null,null,null,null,null,null,185,null,null,null,"10.1007/s11665-018-3822-8",2018,"Effect of Cold Deformation and Heat Treatment on the Microstructure and Mechanical Behavior of High Entropy Alloy CuCrFeNi2Al0.5"],[126,"Al0.5 Cr1 Cu1 Fe1 Ni1","FCC","WROUGHT","FCC",null,null,7.4,287,null,null,null,null,null,null,null,185,null,null,null,"10.1007/s11665-018-3822-8",2018,"Effect of Cold Deformation and Heat Treatment on the Microstructure and Mechanical Behavior of High Entropy Alloy CuCrFeNi2Al0.5"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",25,1512,null,12,null,null,137,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"C",25,1600,null,4,null,null,143,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.7,null,"C",25,1351,null,20,null,null,128,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Zr1","BCC","CAST","BCC",null,null,10.9,null,"C",25,1524,null,16,null,null,141,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,10.8,null,"C",25,1369,null,27,null,null,155,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",800,1007,null,null,null,null,137,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"C",800,1045,null,null,null,null,143,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.7,null,"C",800,829,null,null,null,null,128,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Zr1","BCC","CAST","BCC",null,null,10.9,null,"C",800,1005,null,null,null,null,141,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,10.8,null,"C",800,822,null,null,null,null,155,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",1000,814,null,null,null,null,137,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"C",1000,855,null,null,null,null,143,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.7,null,"C",1000,721,null,null,null,null,128,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Zr1","BCC","CAST","BCC",null,null,10.9,null,"C",1000,927,null,null,null,null,141,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,10.8,null,"C",1000,778,null,null,null,null,155,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,9.9,null,"C",1200,556,null,null,null,null,137,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Ta1 Ti1 Zr1","BCC","CAST","BCC",null,null,10.2,null,"C",1200,404,null,null,null,null,143,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ti1 Zr1","BCC","CAST","BCC",null,null,8.7,null,"C",1200,301,null,null,null,null,128,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Zr1","BCC","CAST","BCC",null,null,10.9,null,"C",1200,694,null,null,null,null,141,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,10.8,null,"C",1200,699,null,null,null,null,155,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Zr1","BCC","CAST","BCC",null,null,10.9,null,"C",1400,278,null,null,null,null,141,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[127,"Hf1 Mo1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,10.8,null,"C",1400,367,null,null,null,null,155,null,null,null,"10.3390/e21010015",2018,"Effects of Mo, Nb, Ta, Ti, and Zr on Mechanical Properties of Equiatomic Hf-Mo-Nb-Ta-Ti-Zr Alloys"],[128,"Co1 Cr1 Fe1 Ni1","FCC","POWDER","FCC",0.0276,null,8.2,null,null,null,null,null,null,null,null,226,null,null,null,"10.3390/coatings9010016",2018,"Microstructure and Properties of CoCrFeNi(WC) High-Entropy Alloy Coatings Prepared Using Mechanical Alloying and Hot Pressing Sintering"],[128,"C0.111 Co1 Cr1 Fe1 Ni1 W0.111","FCC","POWDER","FCC",0.0189,7.24,8.5,475,null,null,null,null,null,null,null,null,null,null,null,"10.3390/coatings9010016",2018,"Microstructure and Properties of CoCrFeNi(WC) High-Entropy Alloy Coatings Prepared Using Mechanical Alloying and Hot Pressing Sintering"],[128,"C0.429 Co1 Cr1 Fe1 Ni1 W0.429","FCC","POWDER","FCC",0.0182,8.39,9.1,531,null,null,null,null,null,null,null,null,null,null,null,"10.3390/coatings9010016",2018,"Microstructure and Properties of CoCrFeNi(WC) High-Entropy Alloy Coatings Prepared Using Mechanical Alloying and Hot Pressing Sintering"],[129,"Al0.267 Co1 Cr0.667 Fe0.667 Ni0.667 Ti0.2","FCC+BCC","POWDER","other",null,null,7.3,515,"C",25,2012,null,null,17.52,null,null,null,null,null,"10.5755/j01.ms.25.3.19452",2019,"Enhancement of Mechanical Properties and Corrosion Resistance of Ultra-Fine Grain Al0.4FeCrCo1.5NiTi0.3 High-Entropy Alloy by MA and SPS Technologies"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"T",25,1438,null,null,null,91.5,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"T",25,1159,null,null,null,99.2,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"T",25,1134,null,null,null,109.1,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"T",25,1094,null,null,null,105.3,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",null,null,9.9,null,"T",25,1071,null,null,null,91.6,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",24.4,null,9.9,null,"T",25,1107,null,null,null,92.6,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",31.3,null,9.9,null,"T",25,1115,null,null,null,92.6,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",38.4,null,9.9,null,"T",25,1114,null,null,null,101.3,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",50.1,null,9.9,null,null,null,null,null,null,null,null,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",21.9,null,9.9,null,null,null,null,null,null,null,null,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[130,"Hf1 Nb1 Ta1 Ti1 Zr1","BCC","OTHER","BCC",75.5,null,9.9,null,null,null,null,null,null,null,null,107,null,null,null,"10.1007/s11661-018-4646-8",2018,"Effect of Cold Deformation and Annealing on the Microstructure and Tensile Properties of a HfNbTaTiZr Refractory High Entropy Alloy"],[131,"Al1 Nb1 Ti1 Zr1","B2","CAST","other",250,5.85,5.7,422,"C",25,1509,null,8.6,null,null,null,null,null,null,"10.1080/02670836.2018.1446267",2018,"Microstructure and mechanical properties of a novel refractory AlNbTiZr high-entropy alloy"],[131,"Al1 Nb1 Ti1 Zr1","B2+Sec.","ANNEAL","other",null,null,5.7,418,"C",25,1579,null,17.8,null,null,null,null,null,null,"10.1080/02670836.2018.1446267",2018,"Microstructure and mechanical properties of a novel refractory AlNbTiZr high-entropy alloy"],[132,"Al1 Nb1 Ti1 V1 Zr0.5",null,null,"other",null,5.64,5.6,null,"C",25,1480,null,50,null,null,null,null,null,null,"10.1016/j.matlet.2015.08.099",2015,"An AlNbTiVZr0.5 high-entropy alloy combining high specific strength and good ductility"],[133,"Al1 Nb1 Ti1 Zr1",null,null,"other",null,5.79,5.7,539.6,"C",25,1080,null,5.9,null,null,null,null,null,null,"10.1179/1743284715Y.0000000032",2015,"Effect of Al on structure and mechanical properties of AlxNbTiVZr (x = 0, 0.5, 1, 1.5) high entropy alloys"],[134,"Nb1 Ti1 V1 Zr1",null,null,"other",null,6.52,6.5,335,"C",25,1105,null,50,null,null,null,null,null,null,"10.1016/j.actamat.2012.11.032",2013,"Low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system: Microstructure and phase analysis"],[134,"Cr1 Nb1 Ti1 Zr1",null,null,"other",null,6.67,6.6,418,"C",25,1260,null,6,null,null,null,null,null,null,"10.1016/j.actamat.2012.11.032",2013,"Low-density, refractory multi-principal element alloys of the Cr–Nb–Ti–V–Zr system: Microstructure and phase analysis"],[264,"Al1 Cr1 Fe1 Mo0.5 Ni1","BCC+B2","CAST","other",null,null,6.8,623.7,"C",25,1914.1,4023.6,null,27.9,null,null,null,null,null,"10.1088/0256-307X/35/3/036102",2018,"Microstructures and Mechanical Properties of AlCrFeNiMo 0.5 Ti  x  High Entropy Alloys"],[264,"Al1 Cr1 Fe1 Mo0.5 Ni1 Ti0.25","BCC+B2","CAST","other",null,null,6.6,712,"C",25,2161.7,3641.5,null,14.3,null,null,null,null,null,"10.1088/0256-307X/35/3/036102",2018,"Microstructures and Mechanical Properties of AlCrFeNiMo 0.5 Ti  x  High Entropy Alloys"],[264,"Al1 Cr1 Fe1 Mo0.5 Ni1 Ti0.4","BCC+B2","CAST","other",null,null,6.5,731.9,"C",25,2185.1,3673.4,null,14.8,null,null,null,null,null,"10.1088/0256-307X/35/3/036102",2018,"Microstructures and Mechanical Properties of AlCrFeNiMo 0.5 Ti  x  High Entropy Alloys"],[264,"Al1 Cr1 Fe1 Mo0.5 Ni1 Ti0.5","BCC+B2","CAST","other",null,null,6.5,751.7,"C",25,2228.7,3166.2,null,10.4,null,null,null,null,null,"10.1088/0256-307X/35/3/036102",2018,"Microstructures and Mechanical Properties of AlCrFeNiMo 0.5 Ti  x  High Entropy Alloys"],[264,"Al1 Cr1 Fe1 Mo0.5 Ni1 Ti0.6","BCC+B2","CAST","other",null,null,6.4,756.1,"C",25,1314.5,1314.5,null,null,null,null,null,null,null,"10.1088/0256-307X/35/3/036102",2018,"Microstructures and Mechanical Properties of AlCrFeNiMo 0.5 Ti  x  High Entropy Alloys"],[264,"Al1 Cr1 Fe1 Mo0.5 Ni1 Ti0.75","BCC+B2","CAST","other",null,null,6.4,766.2,"C",25,618.8,618.8,null,null,null,null,null,null,null,"10.1088/0256-307X/35/3/036102",2018,"Microstructures and Mechanical Properties of AlCrFeNiMo 0.5 Ti  x  High Entropy Alloys"],[265,"Al1 Cr1 Mo1 Nb1","Other","ANNEAL","other",null,null,7.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Cr1 Mo1 Nb1","Other","CAST","other",null,null,7.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Hf1 Nb1 Ti1","B2","ANNEAL","other",null,null,7.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Hf1 Nb1 Ti1","B2","CAST","other",null,null,7.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Hf1 Ta1 Ti1","B2","ANNEAL","other",null,null,9.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Hf1 Ta1 Ti1","B2","CAST","other",null,null,9.7,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Mo1 Nb1 Ti1","BCC+Sec.","ANNEAL","other",null,null,6.5,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Mo1 Nb1 Ti1","BCC","CAST","BCC",null,null,6.5,509,null,null,null,null,null,null,null,151,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Nb1 Ta1 Ti1","BCC+Sec.","ANNEAL","other",null,null,8.2,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Al1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,8.2,458,null,null,null,null,null,null,null,120,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Cr1 Mo1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,9.9,555,null,null,null,null,null,null,null,219,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Cr1 Mo1 Ta1 Ti1","BCC","CAST","BCC",null,null,9.9,539,null,null,null,null,null,null,null,219,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Cr1 Nb1 Ta1 Ti1","BCC+Sec.","ANNEAL","other",null,null,9.4,null,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Cr1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,9.4,495,null,null,null,null,null,null,null,162,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Cr1 Nb1 Ti1 W1","BCC+Sec.","ANNEAL","other",null,null,9.9,630,null,null,null,null,null,null,null,null,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Cr1 Nb1 Ti1 W1","BCC","CAST","BCC",null,null,9.9,630,null,null,null,null,null,null,null,217,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Hf1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10.9,280,null,null,null,null,null,null,null,119,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Hf1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,10.9,270,null,null,null,null,null,null,null,119,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Mo1 Nb1 Ta1 Ti1 W1","BCC","ANNEAL","BCC",null,null,11.8,482,null,null,null,null,null,null,null,222,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Mo1 Nb1 Ta1 Ti1 W1","BCC","CAST","BCC",null,null,11.8,446,null,null,null,null,null,null,null,222,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Mo1 Nb1 Ta1 Ti1","BCC","ANNEAL","BCC",null,null,10,407,null,null,null,null,null,null,null,179,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"],[265,"Mo1 Nb1 Ta1 Ti1","BCC","CAST","BCC",null,null,10,431,null,null,null,null,null,null,null,179,null,null,null,"10.1016/j.matdes.2018.06.003",2018,"Phase equilibria, mechanical properties and design of quaternary refractory high entropy alloys"]]}''')
df = pd.DataFrame(data["rows"], columns=data["headers"])
original = df.copy(deep=True)
H = list(df.columns)
phase, process, test, temp, ys = H[4], H[3], H[9], H[10], H[11]
pretty = ["ID de referencia","Composición química","Microestructura","Procesamiento","Clasificación BCC/FCC/other","Tamaño de grano (µm)","Densidad experimental (g/cm³)","Densidad calculada (g/cm³)","Dureza Vickers (HV)","Tipo de ensayo","Temperatura de ensayo (°C)","Límite elástico (MPa)","Resistencia máxima / UTS (MPa)","Elongación (%)","Elongación plástica (%)","Módulo de Young experimental (GPa)","Módulo de Young calculado (GPa)","Contenido de O (wppm)","Contenido de N (wppm)","Contenido de C (wppm)","DOI","Año","Título del artículo"]
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
assert df.shape == (1545, 23)
display(df.head())
print("Filas y columnas:", df.shape)
display(df.dtypes.rename("Tipo de dato").to_frame())

## 3. Exploración inicial y calidad
Se distinguen **ausencia de información** y **valor cero**: un dato faltante no se rellena con cero. Se conservará una elongación de 0 % como observación posible.

| Variable | Faltantes | Porcentaje (%) |
| --- | --- | --- |
| ID de referencia | 0 | 0 |
| Composición química | 0 | 0 |
| Microestructura | 143 | 9.26 |
| Procesamiento | 119 | 7.7 |
| Clasificación BCC/FCC/other | 0 | 0 |
| Tamaño de grano (µm) | 1,308 | 84.66 |
| Densidad experimental (g/cm³) | 1,433 | 92.75 |
| Densidad calculada (g/cm³) | 0 | 0 |
| Dureza Vickers (HV) | 1,015 | 65.7 |
| Tipo de ensayo | 395 | 25.57 |
| Temperatura de ensayo (°C) | 181 | 11.72 |
| Límite elástico (MPa) | 478 | 30.94 |
| Resistencia máxima / UTS (MPa) | 1,006 | 65.11 |
| Elongación (%) | 926 | 59.94 |
| Elongación plástica (%) | 1,396 | 90.36 |
| Módulo de Young experimental (GPa) | 1,400 | 90.61 |
| Módulo de Young calculado (GPa) | 816 | 52.82 |
| Contenido de O (wppm) | 1,488 | 96.31 |
| Contenido de N (wppm) | 1,500 | 97.09 |
| Contenido de C (wppm) | 1,541 | 99.74 |
| DOI | 0 | 0 |
| Año | 0 | 0 |
| Título del artículo | 10 | 0.65 |

Hay **20 duplicados exactos**, definidos como filas iguales en las 23 columnas. La falta de tamaño de grano y de propiedades experimentales limita su uso como variables de control. La disponibilidad de una propiedad no garantiza que se midiera junto con las demás.

La base reúne estudios con distintos materiales y condiciones. Estas frecuencias describen la base compilada; no representan la frecuencia de las aleaciones en la industria.

In [ ]:
audit = pd.DataFrame({"Variable": pretty, "Faltantes": df.isna().sum().to_numpy(), "% faltantes": df.isna().mean().to_numpy()*100})
display(audit)
print("Duplicados exactos:", df.duplicated().sum())
display(df.describe(include="all").T)
for col in [phase, test, process]:
    display(df[col].value_counts(dropna=False).rename("Registros").to_frame())
fig, ax = plt.subplots(figsize=(9, 6))
audit.sort_values("% faltantes").plot.barh(x="Variable", y="% faltantes", ax=ax, legend=False, color="#637084")
ax.set_xlabel("Datos faltantes (%)")
ax.set_ylabel("Variable (categoría)")
ax.set_title("Disponibilidad de información en la base original")
plt.tight_layout()
plt.show()

## 4. Preparación y justificación de decisiones
| Decisión | Por qué | Cómo afecta |
| --- | --- | --- |
| Eliminar solo duplicados exactos | Evitar contar dos veces la misma fila. | Quedan 1,525 registros; se verifica la sensibilidad con la base original. |
| No imputar faltantes | Evitar inventar propiedades y reducir artificialmente la variación. | Cada análisis utiliza los registros con sus variables necesarias. |
| No eliminar extremos automáticamente | Una aleación o condición real puede tener valores altos o bajos. | Se usan mediana y cuartiles; no se confunden extremos con errores. |
| BCC y FCC para la pregunta principal | Comparar dos categorías identificables sin agrupar todas las fases. | Se excluye other; el resultado no representa toda la base. |
| 20–30 °C | Definir un intervalo operativo cercano a temperatura ambiente. | No se fija una temperatura exacta ni se estudia su efecto causal. |
| YS finito y positivo | Requerir un esfuerzo de fluencia utilizable para esta pregunta. | Se excluyen faltantes o valores no positivos de YS. |
| Separar T y C | Evitar mezclar modos de carga. | Se obtienen dos comparaciones, no una resistencia universal. |
| No filtrar proceso en la comparación principal | Conservar cobertura de grupos pequeños. | Persisten diferencias de composición y fabricación. |
| Tres propiedades completas en la tabla complementaria | Comparar promedios obtenidos de las mismas filas. | La muestra disminuye y puede favorecer registros mejor documentados. |

Eliminar filas idénticas es una decisión operativa: debe comprobarse en las fuentes si fueron repeticiones de compilación o ensayos independientes con resultados iguales. No se elimina una fila solo por repetir composición o DOI.

In [ ]:
# Conversión controlada a numérico: reportar cualquier valor no interpretable.
work = df.copy(deep=True)
numeric_idx = [0,5,6,7,8,10,11,12,13,14,15,16,17,18,19,21]
conversion_report = []
for i in numeric_idx:
    converted = pd.to_numeric(work[H[i]], errors="coerce")
    bad = work[H[i]].notna() & converted.isna()
    conversion_report.append([pretty[i], int(bad.sum())])
    work[H[i]] = converted
conversion_report = pd.DataFrame(conversion_report, columns=["Variable", "Valores no numéricos"])
display(conversion_report)
assert conversion_report["Valores no numéricos"].sum() == 0, "Revisar conversiones antes de continuar"
assert int(work.duplicated().sum()) == 20
clean = work.drop_duplicates().copy()
assert len(clean) == 1525
pd.testing.assert_frame_equal(df, original)
print("Se conserva la base original; filas analizadas sin duplicados:", len(clean))

## 5. Exploración para formular la pregunta
La base ofrece categorías BCC/FCC y mediciones de límite elástico, pero la cobertura depende del ensayo y la temperatura. Antes de comparar, se examinan distribuciones generales y la disponibilidad conjunta.

El límite elástico es relevante para una pieza que debe evitar deformación permanente. Esto motiva una comparación entre estructuras, sin suponer que la estructura es el único factor que controla la resistencia.

In [ ]:
display(pd.crosstab(clean[phase], clean[test], dropna=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
clean[temp].dropna().plot.hist(bins=30, ax=axes[0], color="#637084")
axes[0].set(xlabel="Temperatura de ensayo (°C)", ylabel="Registros (conteo)", title="Temperaturas disponibles")
clean[ys].dropna().plot.hist(bins=30, ax=axes[1], color="#267760")
axes[1].set(xlabel="Límite elástico (MPa)", ylabel="Registros (conteo)", title="YS antes de seleccionar condiciones")
plt.tight_layout()
plt.show()

## 6. Pregunta de análisis
**¿Cómo difiere el límite elástico entre los registros de aleaciones BCC y FCC ensayados entre 20 y 30 °C, comparando por separado compresión y tracción?**

Se comparan medianas, medias, cuartiles y tamaños de muestra. La mediana resume el centro con menor sensibilidad a valores extremos que la media. El intervalo Q1–Q3 describe la mitad central; no es un intervalo de confianza. No se plantea una prueba causal ni se afirma significación estadística.

## 7. Selección de registros y trazabilidad
| Paso | Filas restantes | Excluidas en este paso |
| --- | --- | --- |
| Base original | 1,545 | 0 |
| Sin duplicados exactos | 1,525 | 20 |
| Solo BCC/FCC | 731 | 794 |
| Temperatura 20–30 °C | 415 | 316 |
| YS positivo disponible | 336 | 79 |
| Tipo de ensayo T o C | 336 | 0 |

Las exclusiones son **secuenciales**: una fila descartada antes no se vuelve a contar. Quedan **336 registros**, divididos en **177 de compresión** y **159 de tracción**.

In [ ]:
steps = [("Original", work)]
steps.append(("Sin duplicados", clean))
steps.append(("BCC/FCC", steps[-1][1].loc[lambda x: x[phase].isin(["BCC","FCC"])]))
steps.append(("20–30 °C", steps[-1][1].loc[lambda x: np.isfinite(x[temp]) & x[temp].between(20,30)]))
steps.append(("YS positivo", steps[-1][1].loc[lambda x: np.isfinite(x[ys]) & (x[ys]>0)]))
steps.append(("T o C", steps[-1][1].loc[lambda x: x[test].isin(["T","C"])]))
subset = steps[-1][1].copy()
funnel = pd.DataFrame([(name,len(frame)) for name,frame in steps], columns=["Paso","Filas"])
funnel["Excluidas"] = (-funnel["Filas"].diff()).fillna(0).astype(int)
display(funnel)
assert funnel["Filas"].tolist() == [1545,1525,731,415,336,336]
display(pd.crosstab(subset[test], subset[phase]))

## 8. Resultados principales
**Tabla de referencia calculada con los datos integrados; esfuerzos en MPa.** C = compresión; T = tracción. Los DOI distintos cuentan fuentes, no garantizan independencia.

| Ensayo | Estructura | n | Media | Mediana | Q1 | Q3 | DOI distintos |
| --- | --- | --- | --- | --- | --- | --- | --- |
| C | BCC | 145 | 1,334.73 | 1,300 | 1,100 | 1,530 | 48 |
| C | FCC | 32 | 473.78 | 315 | 215.75 | 641.5 | 15 |
| T | BCC | 32 | 1,040 | 978.5 | 897 | 1,144.25 | 13 |
| T | FCC | 127 | 513.02 | 386.2 | 271.5 | 629 | 41 |

- **Compresión:** mediana BCC de **1,300 MPa** frente a **315 MPa** en FCC: diferencia BCC − FCC de **985 MPa** (145 y 32 registros).
- **Tracción:** mediana BCC de **978.5 MPa** frente a **386.2 MPa** en FCC: diferencia de **592.3 MPa** (32 y 127 registros).

Las medias son mayores que las medianas en los cuatro grupos, compatible con la influencia de valores altos. Los tamaños están desbalanceados y la dispersión impide tratar cada categoría como un único material.

### Figura de referencia


Figura calculada directamente con la copia integrada, independiente de la ejecución de Python. Los rangos se superponen: no todos los registros BCC superan a todos los FCC. Las barras finas representan mínimos y máximos, no intervalos de confianza.

In [ ]:
summary = subset.groupby([test,phase])[ys].agg(n="count", media="mean", mediana="median", desviacion="std", minimo="min", maximo="max")
summary["Q1"] = subset.groupby([test,phase])[ys].quantile(.25)
summary["Q3"] = subset.groupby([test,phase])[ys].quantile(.75)
summary["DOI distintos"] = subset.groupby([test,phase])[H[20]].nunique()
display(summary.round(2))
fig, axes = plt.subplots(1,2, figsize=(11,5), sharey=True)
for ax, mode, title in zip(axes, ["C","T"], ["Compresión","Tracción"]):
    part = subset[subset[test]==mode]
    groups = [part.loc[part[phase]==p,ys].to_numpy() for p in ["BCC","FCC"]]
    ax.boxplot(groups, positions=[1,2], widths=.45)
    ax.set_xticks([1,2])
    ax.set_xticklabels([f"{p} (n={len(v)})" for p,v in zip(["BCC","FCC"],groups)])
    ax.set_xlabel("Estructura cristalina (categoría)")
    ax.set_ylabel("Límite elástico (MPa)")
    ax.set_title(title+" · 20–30 °C")
    ax.grid(axis="y", alpha=.2)
    med = part.groupby(phase)[ys].median()
    print(title, "Diferencia de medianas BCC − FCC:", round(med["BCC"]-med["FCC"],2), "MPa")
plt.tight_layout()
plt.show()
assert np.isclose(summary.loc[("C","BCC"),"mediana"], 1300)
assert np.isclose(summary.loc[("C","FCC"),"mediana"], 315)
assert np.isclose(summary.loc[("T","BCC"),"mediana"], 978.5)
assert np.isclose(summary.loc[("T","FCC"),"mediana"], 386.2)

**Cómo leer los diagramas de caja:** el eje X identifica BCC/FCC y no representa deformación; el eje Y representa YS en MPa. La línea central es la mediana y la caja abarca Q1–Q3. Los bigotes usan la regla de 1.5 veces el intervalo intercuartílico; los puntos exteriores no se eliminan del análisis.

## 9. Sensibilidad a los duplicados
| Base | Ensayo | Estructura | n | Mediana YS (MPa) |
| --- | --- | --- | --- | --- |
| Original | C | BCC | 145 | 1,300 |
| Original | C | FCC | 33 | 330 |
| Original | T | BCC | 32 | 978.5 |
| Original | T | FCC | 127 | 386.2 |
| Sin duplicados | C | BCC | 145 | 1,300 |
| Sin duplicados | C | FCC | 32 | 315 |
| Sin duplicados | T | BCC | 32 | 978.5 |
| Sin duplicados | T | FCC | 127 | 386.2 |

Al quitar duplicados, solo cambia esta mediana: FCC en compresión pasa de **330 a 315 MPa** y de 33 a 32 registros. La diferencia BCC − FCC pasa de **970 a 985 MPa**. La dirección de la comparación se conserva. Esto examina una decisión de limpieza, pero no corrige el sesgo por selección de estudios.

In [ ]:
sensitivity = []
for name, frame in [("Original",work),("Sin duplicados",clean)]:
    part = frame[frame[phase].isin(["BCC","FCC"]) & frame[temp].between(20,30) & np.isfinite(frame[ys]) & (frame[ys]>0) & frame[test].isin(["C","T"])]
    stats = part.groupby([test,phase])[ys].agg(n="count", mediana="median").reset_index()
    stats.insert(0,"Base",name)
    sensitivity.append(stats)
display(pd.concat(sensitivity, ignore_index=True))
# Examinar el procesamiento sin confundirlo con un ajuste causal.
display(subset.groupby([test,phase,process], dropna=False)[ys].agg(["count","median"]))

## 10. Análisis complementario: ensayos y relación resistencia–elongación
Se conserva la exploración por microestructura del sitio. Para dureza se exige HV disponible y positivo; no se requiere T/C porque la dureza y el tipo de ensayo son campos distintos. Para la tabla de tracción/compresión se requieren YS, UTS y elongación en la misma fila, de modo que sus medias correspondan a un conjunto común. Esta selección puede ser más restrictiva que analizar cada propiedad por separado.

La dispersión usa solamente tracción a 20–30 °C con YS y elongación disponibles, sin exigir UTS. **X: Elongación (%); Y: Límite elástico (MPa).** Cada punto es un registro. No se interpreta la columna de elongación en compresión como ductilidad en tracción, ni UTS de compresión como resistencia a tracción.

In [ ]:
hardness = clean[clean[H[2]].notna() & np.isfinite(clean[H[8]]) & (clean[H[8]]>0)]
display(hardness.groupby(H[2])[H[8]].agg(["count","mean","median"]).sort_values("count",ascending=False))
for mode in ["T","C"]:
    available = clean[clean[test]==mode]
    common = available.dropna(subset=[H[2],H[11],H[12],H[13]])
    common = common[(common[H[11]]>0)&(common[H[12]]>0)&(common[H[13]]>=0)]
    print(mode, "Registros del ensayo:",len(available), "Conjunto común:",len(common), "Excluidos:",len(available)-len(common))
    report = common.groupby(H[2])[[H[11],H[12],H[13]]].agg(["count","mean"])
    display(report)
paired = subset[(subset[test]=="T") & np.isfinite(subset[H[13]]) & (subset[H[13]]>=0)].copy()
fig, ax = plt.subplots(figsize=(8,5))
for p,color in [("BCC","#267760"),("FCC","#c18c41")]:
    part = paired[paired[phase]==p]
    ax.scatter(part[H[13]],part[ys],label=f"{p} (n={len(part)})",alpha=.6,color=color)
    # Spearman sin exigir scipy: correlación de rangos con empates promediados.
    rho = part[H[13]].rank().corr(part[ys].rank()) if len(part)>2 else np.nan
    print(p, "n =",len(part), "Spearman descriptivo =",rho)
ax.set(xlabel="Elongación (%)",ylabel="Límite elástico (MPa)",title="Tracción a 20–30 °C: resistencia y alargamiento")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()
print("Las correlaciones son descriptivas; no equivalen a causalidad ni significación estadística.")

## 11. Interpretación desde Ingeniería de Materiales
En la muestra seleccionada, los registros BCC presentan mayor límite elástico central en ambos modos de carga. Esto describe un mayor esfuerzo nominal asociado con la fluencia en esos registros; **no permite elegir una aleación concreta solo por ser BCC**.

La deformación plástica de metales está relacionada con el movimiento de dislocaciones. La estructura cristalina influye en sus mecanismos de deslizamiento, pero también intervienen composición, fases secundarias, tamaño de grano, textura y procesamiento. La comparación entre grupos mezcla estos factores. No demuestra que cambiar únicamente BCC por FCC produzca las diferencias de 985 o 592.3 MPa.

Para seleccionar un material real también se necesitan ductilidad, rigidez, densidad, condiciones de servicio y variabilidad. El límite elástico no es dureza ni módulo de Young. Tampoco debe suponerse equivalencia de tracción y compresión; aquí se comparan por separado.

## 12. Limitaciones
- **Dependencia entre registros:** un artículo o una composición puede contribuir varias filas; n no equivale a n ensayos independientes.
- **Cobertura desigual:** 145 BCC frente a 32 FCC en compresión; 32 BCC frente a 127 FCC en tracción.
- **Faltantes y selección:** el subconjunto final puede diferir de los registros excluidos; no se corrige ese sesgo.
- **Variables de confusión:** no se controla composición ni procesamiento, y falta tamaño de grano en muchos registros.
- **Temperatura:** 20–30 °C es un intervalo operativo; no garantiza condiciones idénticas.
- **Definiciones:** los criterios experimentales de YS, elongación y cargas de dureza pueden variar entre artículos.
- **Calidad de origen:** se analiza la copia proporcionada; no se validó cada medición con su artículo ni se verificaron todas las unidades individualmente.
- **Alcance estadístico:** no se reportan pruebas de significación ni intervalos de confianza que supongan independencia injustificada.
- **Curvas:** la base contiene propiedades resumidas; no permite reconstruir curvas esfuerzo–deformación experimentales completas.

## 13. Conclusión
**La pregunta se responde afirmativamente en términos descriptivos: en estos registros, BCC presenta un mayor límite elástico mediano que FCC, tanto en compresión como en tracción, entre 20 y 30 °C.**

En compresión, las medianas son **1,300 y 315 MPa**, respectivamente, con una diferencia de **985 MPa**. En tracción son **978.5 y 386.2 MPa**, con una diferencia de **592.3 MPa**. Eliminar duplicados no cambia la dirección de los resultados.

La conclusión se limita al subconjunto analizado y no establece superioridad universal ni un efecto causal de la estructura. Un siguiente estudio debería comparar composiciones y tratamientos semejantes, revisar los artículos originales y considerar la agrupación por estudio.

## 14. Fuentes y trazabilidad
- Base de datos suministrada: MPEA_dataset(1).xlsx; copia original integrada en este notebook y en el sitio.
- Metodología: [NIST/SEMATECH, Exploratory Data Analysis](https://itl.nist.gov/div898/handbook/eda/eda.htm).
- Definiciones de ensayo: [Instron, ensayo de tracción](https://instron.com/en/resources/test-types/tensile-test/) y [Struers, Vickers](https://www.struers.com/en/Knowledge/Hardness-testing/Vickers).
- Los DOI, años y títulos se conservan en las columnas originales. No se atribuyen citas específicas a Askeland & Wright sin contar con la edición o fragmentos correspondientes.

## 15. Lista de comprobación de la entrega
El reporte incluye descripción, auditoría, preparación, pregunta, selección justificada, resultados, visualizaciones, interpretación, limitaciones y conclusión. **Antes de entregar, ejecuta todas las celdas en Colab y guarda las salidas.** Si una comprobación falla, revisa el origen antes de modificar cifras para forzarla.

In [ ]:
# Exportaciones opcionales; no modifican los datos originales.
df.to_csv("MPEA_original_completo.csv",index=False,encoding="utf-8-sig")
subset.to_csv("MPEA_comparacion_20_30C.csv",index=False,encoding="utf-8-sig")
summary.to_csv("MPEA_resultados_principales.csv",encoding="utf-8-sig")
pd.testing.assert_frame_equal(df, original)
print("Análisis completado. Archivos CSV creados en el panel de archivos de Colab.")